# Task 2 — Season Classification

**Status:** the Task 2 component is complete through the locked handoff. All model choices, development evidence, the scratch refit, and image-only inference are hash-linked; Notebook 06 remains locked until the whole group freezes.

**How to finish this notebook:** run the existing cells in order later, keep their outputs, and check that every measured interpretation still matches its verified artifact. Final group evaluation stays outside this notebook.

**Structural rule:** every leaf `###` subsection owns exactly one code cell. A broader `###` owns no direct code cell; divide it into `####` subsubsections, with one code cell per `####` leaf.

**Frozen boundaries:** teacher images only; `data/processed/splits.csv` is the only split; all submitted models are trained from scratch; all runs go through `fashion.train.registry` to `results/runs.csv`; holdout remains sealed until Notebook 06.

**Core risks:** weak-visual-signal labels, article-type shortcut learning, class imbalance, acquisition artifacts, and calibration failure.

**Primary development metric:** pooled out-of-fold macro-F1 over `Fall`, `Spring`, `Summer`, and `Winter`.

## 1. Task contract and reproducibility

Freeze the decision, labels, paths, seed, and execution environment before any result is viewed.

### 1.1 Frozen task configuration

This cell creates the single configuration object used by every later cell.

In [ ]:
import hashlib
import json

import pandas as pd
from IPython.display import display

from fashion.config import (
    CV_FOLD_SUMMARY_JSON,
    RANDOM_SEED,
    ROOT,
    TASK2_CHECKPOINT_DIR,
    TASK2_EVIDENCE_DIR,
    TASK2_FIGURE_DIR,
    TASK2_MODEL_PATH,
    TASK2_SELECTION_FREEZE_JSON,
)
from fashion.data.dataset import load_label_maps

TASK2_TARGET = "season"
SEASON_MASK = "has_season_label"
SEASON_LABELS = tuple(load_label_maps()[TASK2_TARGET]["classes"])
assert SEASON_LABELS == ("Fall", "Spring", "Summer", "Winter")

with CV_FOLD_SUMMARY_JSON.open(encoding="utf-8") as handle:
    cv_summary = json.load(handle)
for directory in (TASK2_EVIDENCE_DIR, TASK2_FIGURE_DIR, TASK2_CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

task_contract = pd.DataFrame(
    [{
        "target": TASK2_TARGET,
        "labels": " | ".join(SEASON_LABELS),
        "seed": RANDOM_SEED,
        "cv_folds": cv_summary["fold_count"],
        "cv_assignment_sha256": cv_summary["cv_assignment_sha256"],
        "evidence_directory": TASK2_EVIDENCE_DIR.relative_to(ROOT).as_posix(),
        "figure_directory": TASK2_FIGURE_DIR.relative_to(ROOT).as_posix(),
        "checkpoint_directory": TASK2_CHECKPOINT_DIR.relative_to(ROOT).as_posix(),
        "freeze_manifest": TASK2_SELECTION_FREEZE_JSON.relative_to(ROOT).as_posix(),
        "final_model": TASK2_MODEL_PATH.relative_to(ROOT).as_posix(),
    }]
)
assert task_contract.loc[0, "cv_folds"] == 5
display(task_contract)

> **Interpretation to write after running:** Confirm that the output matches the assignment contract. Explain that inference is image-only and that the official CSV must always contain one Season label.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 1.2 Environment and deterministic execution

Package versions, hardware, and random state must be recorded so runs can be reproduced.

In [ ]:
from pathlib import Path

from fashion.train.artifacts import atomic_write_json
from fashion.train.reproducibility import (
    capture_git_state,
    capture_runtime,
    seed_everything,
)

seed_everything(RANDOM_SEED, deterministic=True)
runtime = capture_runtime()
runtime["executable_name"] = Path(runtime.pop("executable")).name
environment_record = {
    "schema_version": "1.0.0",
    "seed": RANDOM_SEED,
    "deterministic_algorithms": True,
    "mixed_precision_policy": "torch.amp on CUDA; disabled on CPU",
    "git": capture_git_state(),
    "runtime": runtime,
}
environment_path = TASK2_EVIDENCE_DIR / "environment.json"
atomic_write_json(environment_path, environment_record)
environment_table = (
    pd.json_normalize(environment_record, sep=".")
    .transpose()
    .rename_axis("field")
    .reset_index()
    .rename(columns={0: "value"})
)
display(environment_table)

> **Interpretation to write after running:** State the actual machine and software used. Note any deterministic-operation limitation rather than hiding it.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 1.3 How Task 2 files affect one another

This map separates inputs that create a model from evidence that only explains it. The direction of every arrow is part of the leakage and deployment contract.

#### 1.3.1 Declare and audit the impact edges

In [ ]:
from IPython.display import display

from fashion.task2.evidence import (
    build_file_impact_edges,
    validate_file_impact_edges,
)

file_impact_edges = build_file_impact_edges()
validate_file_impact_edges(file_impact_edges)
display(file_impact_edges)

> **Interpretation:** **Result —** the audited table declares every producer, artifact, consumer, and effect used by Task 2. **Meaning —** canonical data and source code can change training; a config change creates a new experiment identity; evidence can describe a trained model but cannot modify it. **Decision —** trace every report claim back to a registered run ID, and let holdout, prediction, and app code consume only the frozen bundle. **Limitation —** this table verifies the declared dependency contract, not model quality. **Trace —** `results/evidence/task2/file_impact_edges.csv` and its SHA-256 manifest.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 1.3.2 Render the file-impact flow

In [ ]:
from IPython.display import Image, display

from fashion.config import ROOT
from fashion.task2.evidence import build_task2_evidence

file_impact_manifest = build_task2_evidence()
display(Image(filename=str(ROOT / file_impact_manifest["figure_path"])))

> **Interpretation:** **Result —** the PNG makes the one-way Task 2 lifecycle visible in Jupyter and exported HTML. **Meaning —** the selection freeze comes before development refit, while Notebook 06, the prediction CLI, and the app all read the same frozen bytes and manifest. **Decision —** use this flow when checking whether a future edit invalidates training, evidence only, or deployment. **Limitation —** arrows show allowed file impact, not runtime order or measured performance. **Trace —** `results/figures/task2/file_impact_flow.png` with its SHA-256 recorded in `file_impact_manifest.json`.

> **Chart guide.** Read arrows from producer to artifact/interface to consumer. Training data, config, and source affect runs; frozen evidence affects the narrative; only the final bundle may feed later evaluation and deployment.

## 2. Data and EDA handoff

Reproduce only the Task 2 facts needed from Notebook 01. These are data facts, not model results.

### 2.1 Load the development task frame

Task 2 uses teacher images and excludes only rows with a truly blank Season label.

In [ ]:
from fashion.data.dataset import get_samples, load_splits

splits = load_splits()
development = get_samples(splits, partition="development")
season_development = get_samples(development, target=TASK2_TARGET).reset_index(drop=True)
protected = splits["partition"].isin(["holdout", "quarantine"])
visible_protected_labels = splits.loc[protected, TASK2_TARGET].astype(str).str.strip().ne("").sum()
visible_protected_masks = splits.loc[protected, SEASON_MASK].astype(bool).sum()

assert len(development) == 32_773
assert len(season_development) == 32_753
assert len(development) - len(season_development) == 20
assert not season_development["partition"].isin(["holdout", "quarantine"]).any()
assert visible_protected_labels == visible_protected_masks == 0

task2_data_audit = pd.DataFrame(
    [
        {"check": "development rows", "value": len(development)},
        {"check": "valid Season rows", "value": len(season_development)},
        {"check": "masked Season rows", "value": len(development) - len(season_development)},
        {"check": "visible protected Season labels", "value": int(visible_protected_labels)},
        {"check": "protected rows in modelling frame", "value": 0},
    ]
)
display(task2_data_audit)

> **Interpretation:** The Task 2 handoff exactly reproduces Notebook 01: 32,773 development rows, 32,753 valid Season labels, and 20 masked labels. No holdout or quarantine label is visible, and no protected row enters the modelling frame. This confirms that the earlier EDA counts were accurate and freezes one comparable population for every later baseline, model, and ablation.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 2.2 Audit class balance, folds, and EDA evidence

This topic needs separate numeric and visual evidence, so it is divided into two subsubsections.

#### 2.2.1 Reproduce class and fold statistics

The numeric audit checks the exact EDA facts that drive imbalance and validation choices.

In [ ]:
from fashion.train.artifacts import atomic_write_csv

season_counts = season_development[TASK2_TARGET].value_counts().reindex(SEASON_LABELS)
expected_counts = pd.Series(
    {"Fall": 8_928, "Spring": 1_329, "Summer": 16_235, "Winter": 6_261}
)
pd.testing.assert_series_equal(season_counts, expected_counts, check_names=False)
fold_support = (
    pd.crosstab(season_development[TASK2_TARGET], season_development["cv_fold"])
    .reindex(index=SEASON_LABELS, columns=range(5), fill_value=0)
)
assert fold_support.gt(0).all().all()
imbalance_ratio = float(season_counts.max() / season_counts.min())
eda_handoff = pd.DataFrame(
    {
        "class": SEASON_LABELS,
        "development_products": season_counts.to_numpy(),
        "share_percent": (season_counts / season_counts.sum() * 100).to_numpy(),
        **{f"fold_{fold}_validation_products": fold_support[fold].to_numpy() for fold in range(5)},
    }
)
eda_handoff["largest_to_smallest_ratio"] = imbalance_ratio
atomic_write_csv(TASK2_EVIDENCE_DIR / "eda_handoff.csv", eda_handoff)
display(
    eda_handoff.style.format(
        {"share_percent": "{:.2f}", "largest_to_smallest_ratio": "{:.2f}"}
    )
)

> **Interpretation:** Summer contains 16,235 products (49.57%) while Spring contains 1,329 (4.06%), a 12.22:1 ratio; every fold still contains all four labels. The EDA hypothesis entering modelling is therefore explicit: accuracy can look reasonable when a model mostly predicts Summer, while pooled macro-F1 must expose failure on Spring. Section 5.1 tests this claim with B0 before any image model is accepted.

> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.

#### 2.2.2 Display the saved shortcut and transform evidence

The visual handoff keeps Notebook 03 tied to the acquisition, compression, shortcut, and transform risks found in Notebook 01.

In [ ]:
from IPython.display import Image, display

from fashion.config import DATA_PREPARATION_FIGURE_DIR
from fashion.data.hashing import compute_sha256

eda_figure_paths = {
    "acquisition shortcut": DATA_PREPARATION_FIGURE_DIR / "acquisition_shortcut_risk.png",
    "file-size shortcut": DATA_PREPARATION_FIGURE_DIR / "season_file_size_shortcut.png",
    "ArticleType shortcut": DATA_PREPARATION_FIGURE_DIR / "shortcut_risk_heatmaps.png",
    "transform risk": DATA_PREPARATION_FIGURE_DIR / "transform_risk.png",
}
eda_figure_registry = pd.DataFrame(
    [
        {
            "evidence": name,
            "path": path.relative_to(ROOT).as_posix(),
            "sha256": compute_sha256(path),
            "claim_boundary": "development-only description; not model accuracy or causality",
        }
        for name, path in eda_figure_paths.items()
        if path.is_file()
    ]
)
assert len(eda_figure_registry) == len(eda_figure_paths)
atomic_write_csv(TASK2_EVIDENCE_DIR / "eda_figure_registry.csv", eda_figure_registry)
display(eda_figure_registry)

> **Interpretation:** The earlier EDA provides hypotheses, not model scores. Shape and colour patterns justify B1; transform risk justifies aspect-preserving padding plus controlled P0/P1 and A0/A1 tests; ArticleType, file size, and acquisition year are possible shortcuts that must later be tested on OOF predictions. The figures remain development-only descriptions and do not prove accuracy or causality. Section 8.2.3 records which early insights were supported, contradicted, or still untested after the first measured gates.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 2.2.3 Acquisition-year shortcut evidence

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(eda_figure_paths["acquisition shortcut"])))

> **Interpretation — Acquisition-year shortcut figure**
>
> **Why this output exists.** The EDA raised whether collection year could act as an easier proxy than visual Season signal.
>
> **How to read it.** The labelled axes identify the year grouping and Season pattern; colours identify the Season categories.
>
> **Measured finding.** This is a development-only descriptive pattern, not an accuracy score or causal effect.
>
> **Decision impact and limitation.** It motivates an acquisition-year error slice while forbidding year as an inference feature.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 2.2.4 File-size shortcut evidence

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(eda_figure_paths["file-size shortcut"])))

> **Interpretation — File-size shortcut figure**
>
> **Why this output exists.** The EDA suggested that file size might encode source or product quality and change difficulty.
>
> **How to read it.** The axes and colour legend describe file-size groups and Season distribution; no prediction metric is shown.
>
> **Measured finding.** The figure supplies a hypothesis later tested with training-fitted file-size quartiles.
>
> **Decision impact and limitation.** It supports robustness analysis only; it does not prove causality and file size is never a model input.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 2.2.5 ArticleType shortcut evidence

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(eda_figure_paths["ArticleType shortcut"])))

> **Interpretation — ArticleType shortcut figure**
>
> **Why this output exists.** ArticleType was a plausible metadata shortcut because product categories can correlate with Season.
>
> **How to read it.** The heatmap axes name ArticleType/Season groups and the colour scale encodes association strength.
>
> **Measured finding.** The pattern motivates aligned-versus-conflict slices and the image-only auxiliary-head experiment.
>
> **Decision impact and limitation.** It is an association in development metadata, not proof that the model uses ArticleType or that it generalises.

> **Chart guide.** The chart separates fold-fitted ArticleType-aligned and conflict slices. Higher macro-F1 is better; the aligned-to-conflict gap measures shortcut sensitivity.

#### 2.2.6 Transform-risk evidence

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(eda_figure_paths["transform risk"])))

> **Interpretation — Transform-risk figure**
>
> **Why this output exists.** The images are small and non-square, so resizing or colour changes could erase useful signal.
>
> **How to read it.** The axes and legend identify the transform condition and the measured image/content-pixel diagnostic.
>
> **Measured finding.** The figure justifies aspect-preserving P0/P1 and controlled A0/A1 tests.
>
> **Decision impact and limitation.** The final transform is selected by OOF evidence; this EDA plot alone cannot select a model.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

## 3. Development-validation protocol

Use all five canonical folds and create one out-of-fold prediction per valid product.

### 3.1 Build the five canonical fold views

Every candidate must see exactly the same training and validation rows.

In [ ]:
from fashion.data.dataset import iter_cv_folds

fold_frames = {}
fold_rows = []
for fold, training_all, validation_all in iter_cv_folds(splits):
    training = get_samples(training_all, target=TASK2_TARGET).reset_index(drop=True)
    validation = get_samples(validation_all, target=TASK2_TARGET).reset_index(drop=True)
    family_overlap = set(training["product_family_group"]) & set(validation["product_family_group"])
    assert not family_overlap
    assert set(training[TASK2_TARGET]) == set(validation[TASK2_TARGET]) == set(SEASON_LABELS)
    fold_frames[fold] = {"training": training, "validation": validation}
    fold_rows.append(
        {
            "validation_fold": fold,
            "training_products": len(training),
            "validation_products": len(validation),
            "training_families": training["product_family_group"].nunique(),
            "validation_families": validation["product_family_group"].nunique(),
            "training_classes": training[TASK2_TARGET].nunique(),
            "validation_classes": validation[TASK2_TARGET].nunique(),
            "family_crossings": len(family_overlap),
        }
    )
fold_handoff = pd.DataFrame(fold_rows)
assert len(fold_handoff) == 5
assert fold_handoff["validation_products"].sum() == len(season_development)
atomic_write_csv(TASK2_EVIDENCE_DIR / "fold_handoff.csv", fold_handoff)
display(fold_handoff)

> **Interpretation to write after running:** State that all-five-fold CV was chosen for stable comparison and that no best-looking fold will be selected.

> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.

### 3.2 Freeze the OOF coverage contract

Pooled evidence is valid only when each product is predicted outside its training data exactly once.

In [ ]:
from fashion.train.artifacts import canonical_sha256

expected_oof_index = season_development[
    ["id", "cv_fold", TASK2_TARGET, "product_family_group"]
].copy()
protected_ids = set(splits.loc[protected, "id"].astype(int))
assert expected_oof_index["id"].is_unique
assert not (set(expected_oof_index["id"].astype(int)) & protected_ids)
assert set(expected_oof_index[TASK2_TARGET]) == set(SEASON_LABELS)

oof_probability_columns = [f"prob_{label}" for label in SEASON_LABELS]
oof_contract = {
    "schema_version": "1.0.0",
    "expected_row_count": len(expected_oof_index),
    "expected_unique_ids": len(expected_oof_index),
    "expected_id_sha256": canonical_sha256(sorted(expected_oof_index["id"].astype(int))),
    "required_columns": ["id", "fold", "y_true", "y_pred", *oof_probability_columns],
    "labels": list(SEASON_LABELS),
    "probability_rule": "finite values in [0,1], four columns present, each row sums to one",
    "coverage_rule": (
        "each valid development ID appears exactly once; "
        "protected IDs appear zero times"
    ),
    "validator": "fashion.train.metrics.validate_oof",
}
atomic_write_json(TASK2_EVIDENCE_DIR / "oof_contract.json", oof_contract)
display(pd.Series(oof_contract, name="OOF contract").to_frame())

> **Interpretation to write after running:** Explain why pooled OOF predictions are the fair unit for model comparison.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 3.3 Freeze metrics and selection rules

Metrics must be declared before candidate results are visible.

In [ ]:
metric_contract = {
    "schema_version": "1.0.0",
    "primary_metric": "pooled five-fold OOF macro-F1",
    "labels": list(SEASON_LABELS),
    "zero_division": 0,
    "secondary_metrics": [
        "fold macro-F1 mean and standard deviation",
        "per-class precision, recall, F1, and support",
        "balanced accuracy and accuracy",
        "count and row-normalised confusion matrices",
        "NLL, multiclass Brier score, ECE, and risk-coverage",
        "model size, parameter count, latency, RAM, and VRAM",
    ],
    "uncertainty": "10,000 paired bootstrap samples grouped by product_family_group",
    "near_tie_macro_f1_points": 0.5,
    "near_tie_requires_ci_to_contain_zero": True,
    "smaller_model_robustness_tolerance_points": 1.0,
    "winner_rule": (
        "quality first; then robustness, calibration, and deployment cost; "
        "never accuracy alone"
    ),
}
atomic_write_json(TASK2_EVIDENCE_DIR / "metric_contract.json", metric_contract)
display(pd.Series(metric_contract, name="Metric contract").to_frame())

> **Interpretation to write after running:** Explain how macro-F1 gives Spring the same voting weight as Summer and how secondary metrics expose different failure types.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

## 4. Preprocessing and leakage controls

Fit every learned image value inside the current training folds and keep diagnostic metadata out of the model input.

### 4.1 Fit and inspect the fold-specific image pipeline

Aspect ratio and training-only statistics protect the small 60 x 80 images from distortion and leakage.

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image as PILImage

from fashion.data.images import transform_image_with_mask
from fashion.data.torch import build_task_loaders
from fashion.train.artifacts import atomic_write_bytes

fold0_loaders = build_task_loaders(
    validation_fold=0,
    image_size=(80, 60),
    batch_size=8,
    validation_batch_size=8,
    target=TASK2_TARGET,
    augmentation="a0",
    seed=RANDOM_SEED,
    num_workers=0,
)
validation_batch = next(iter(fold0_loaders.validation))
mean = torch.tensor(fold0_loaders.stats.mean).view(1, 3, 1, 1)
std = torch.tensor(fold0_loaders.stats.std).view(1, 3, 1, 1)
display_images = (validation_batch["image"] * std + mean).clamp(0, 1)

validation_frame = fold_frames[0]["validation"]
nonstandard_aspect = validation_frame.loc[
    ~np.isclose(validation_frame["width"] / validation_frame["height"], 60 / 80)
]
assert not nonstandard_aspect.empty
first_row = nonstandard_aspect.iloc[0]
with PILImage.open(ROOT / first_row["path"]) as source:
    _, first_content_mask = transform_image_with_mask(source, image_size=(80, 60))
assert first_content_mask.any() and (~first_content_mask).any()

figure, axes = plt.subplots(2, 4, figsize=(12, 7))
for axis, image, target, item_id in zip(
    axes.flat[:6],
    display_images[:6],
    validation_batch["target"][:6],
    validation_batch["id"][:6],
    strict=True,
):
    axis.imshow(np.moveaxis(image.numpy(), 0, -1))
    axis.set_title(f"ID {int(item_id)} | {fold0_loaders.labels[int(target)]}")
    axis.axis("off")
axes.flat[6].imshow(first_content_mask, cmap="gray", vmin=0, vmax=1)
axes.flat[6].set_title("Content-pixel mask")
axes.flat[6].axis("off")
axes.flat[7].axis("off")
figure.suptitle("Fold 0 validation transform: aspect preserved, training-fitted normalisation")
figure.tight_layout()
preprocessing_figure_path = TASK2_FIGURE_DIR / "fold0_preprocessing_audit.png"
buffer = io.BytesIO()
figure.savefig(buffer, format="png", dpi=160, bbox_inches="tight")
atomic_write_bytes(preprocessing_figure_path, buffer.getvalue())
plt.close(figure)
preprocessing_summary = {
    **fold0_loaders.audit(),
    "batch_shape": tuple(validation_batch["image"].shape),
}
display(pd.Series(preprocessing_summary).to_frame("value"))

> **Interpretation to write after running:** Confirm that product shape is preserved. Explain any unusual colour or padding behaviour visible in the batch.

> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.

#### 4.1.1 Preprocessing audit figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(preprocessing_figure_path)))

> **Interpretation — 4.1.1 Preprocessing audit figure**
>
> **Why this output exists.** This output isolates 4.1.1 preprocessing audit figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Panels show the source image, aspect-ratio-preserving padded tensor, and content mask. Mask value 1 is a real image pixel; 0 is padding excluded from fitted statistics.
>
> **What it shows.** The content mask separates real pixels from padding, allowing training-fold mean and standard deviation to exclude artificial borders.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 4.2 Declare the controlled transform comparison

P0/P1 and A0/A1 answer whether upscaling and mild colour change help without changing the model budget.

In [ ]:
from fashion.data.torch import ImageTransformSpec

transform_rows = []
for config_id, image_size, augmentation, gate, eligibility in (
    ("P0_A0", (80, 60), "a0", "input-size", "always"),
    ("P1_A0", (128, 96), "a0", "input-size", "always"),
    ("P0_A1", (80, 60), "a1", "augmentation", "only if P0_A0 wins size gate"),
    ("P1_A1", (128, 96), "a1", "augmentation", "only if P1_A0 wins size gate"),
):
    spec = ImageTransformSpec(image_size=image_size, augmentation=augmentation)
    transform_rows.append(
        {
            "config_id": config_id,
            "gate": gate,
            "image_size": str(image_size),
            "augmentation": augmentation,
            "transform_id": spec.transform_id,
            "model_family": "resnet18_small_stem",
            "seed": RANDOM_SEED,
            "folds": "0,1,2,3,4",
            "screen_epochs": 8,
            "eligibility": eligibility,
        }
    )
transform_matrix = pd.DataFrame(transform_rows)
assert transform_matrix[["model_family", "seed", "folds", "screen_epochs"]].nunique().eq(1).all()
atomic_write_csv(TASK2_EVIDENCE_DIR / "transform_matrix.csv", transform_matrix)
display(transform_matrix)

> **Interpretation to write after running:** Explain which visual question each comparison answers. Do not select a transform until the controlled results exist.

> **Column guide.** Candidate/variant names the row; `pooled_macro_f1` is computed across all OOF rows; `fold_sd_macro_f1` is five-fold variation; Spring fields expose minority performance; delta is candidate minus reference; selected/eligible fields apply the frozen rule.

### 4.3 Run the leakage and transform audit

Year, file size, and true ArticleType can define later slices but must never enter the inference tensor.

In [ ]:
import inspect

from fashion.models.season import SeasonModelSpec, assert_final_model, build_season_model

audit_model = build_season_model(SeasonModelSpec(family="smallcnn"))
model_boundary = assert_final_model(audit_model)
dataset_keys = set(fold0_loaders.validation.dataset[0])
first_validation = fold0_loaders.validation.dataset[0]["image"]
second_validation = fold0_loaders.validation.dataset[0]["image"]
forward_inputs = list(inspect.signature(audit_model.forward).parameters)
forbidden_inputs = {"year", "file_size_bytes", "articleType", "season", "cv_fold", "partition"}

leakage_checks = {
    "dataset exposes only id, image, and encoded target": dataset_keys == {"id", "image", "target"},
    "model forward accepts images only": forward_inputs == ["images"],
    "forbidden metadata absent from model inputs": not (dataset_keys & forbidden_inputs),
    "validation transform is deterministic": torch.equal(first_validation, second_validation),
    "normalisation fitted on fold-0 training IDs": fold0_loaders.stats.validation_fold == 0,
    "training and validation IDs do not overlap": not (
        set(fold0_loaders.training_ids) & set(fold0_loaders.validation_ids)
    ),
    "protected labels remain sealed": visible_protected_labels == visible_protected_masks == 0,
    "audit model is scratch and final-eligible": (
        model_boundary["training_origin"] == "scratch"
        and model_boundary["final_eligible"]
    ),
}
assert all(leakage_checks.values())
leakage_audit = {
    "schema_version": "1.0.0",
    "validation_fold": 0,
    "checks": leakage_checks,
    "forbidden_inference_fields": sorted(forbidden_inputs),
    "model_boundary": model_boundary,
}
atomic_write_json(TASK2_EVIDENCE_DIR / "leakage_audit.json", leakage_audit)
display(pd.DataFrame(leakage_checks.items(), columns=["check", "passed"]))

> **Interpretation to write after running:** State exactly which metadata is retained only for post-prediction analysis and why that does not leak into training.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

## 5. Baselines

Establish simple comparison anchors before interpreting deep models.

### 5.1 B0 training-fold majority baseline

B0 is selected first because it is the simplest leakage-safe predictor: each fold uses only the class distribution of its four training folds. Its strength is that it cheaply checks split handling and turns the EDA imbalance warning into a measurable lower bound. Its known limitation is complete image blindness; that limitation directly motivates B1.

In [ ]:
from fashion.config import RUNS_CSV, TASK2_CONFIG_DIR
from fashion.task2.evidence import build_experiment_evidence
from fashion.task2.experiments import run_or_load_experiment

b0_outputs = run_or_load_experiment(
    TASK2_CONFIG_DIR / "b0_majority.json",
    mode="run_or_load",
)
b0_manifest = build_experiment_evidence(
    b0_outputs,
    registry_path=RUNS_CSV,
    expected_ids=season_development["id"],
    protected_ids=protected_ids,
    probability_note=(
        "Each validation product receives empirical Season priors fitted only on "
        "the other four training folds; these are valid probabilistic baseline "
        "outputs, not image-conditioned confidence."
    ),
    calibration_claim_allowed=True,
    evidence_directory=TASK2_EVIDENCE_DIR / "b0_majority",
    figure_directory=TASK2_FIGURE_DIR,
)
assert b0_manifest["coverage"]["row_count"] == len(season_development)
assert b0_manifest["coverage"]["protected_id_count"] == 0
assert len(b0_manifest["run_ids"]) == len(set(b0_manifest["run_ids"])) == 5

with (ROOT / b0_manifest["artifacts"]["pooled_metrics"]["path"]).open(
    encoding="utf-8"
) as handle:
    b0_pooled_metrics = json.load(handle)
b0_fold_summary = pd.read_csv(
    ROOT / b0_manifest["artifacts"]["fold_summary"]["path"]
)
b0_scorecard = pd.DataFrame(
    [{
        "experiment_id": b0_manifest["experiment_id"],
        "OOF products": b0_pooled_metrics["n_samples"],
        "accuracy": b0_pooled_metrics["accuracy"],
        "balanced_accuracy": b0_pooled_metrics["balanced_accuracy"],
        "macro_f1": b0_pooled_metrics["macro_f1"],
        "Spring_f1": b0_pooled_metrics["per_class"]["Spring"]["f1"],
        "run_ids": " | ".join(b0_manifest["run_ids"]),
    }]
)
display(b0_scorecard)


# Additional single-output views rendered below:
# display(b0_fold_summary.loc[b0_fold_summary["metric"].eq("macro_f1")])
# display(Image(filename=str(ROOT / b0_manifest["artifacts"]["figure"]["path"])))


> **Interpretation:** **Why this baseline:** B0 was selected first because a learning-free, training-fold-only majority predictor is the cleanest pipeline check and the honest lower bound implied by the EDA. **Result:** it covered all 32,753 valid development products once, with no protected IDs. It reached 49.568% accuracy but only `0.165704` pooled macro-F1 and `0.250000` balanced accuracy. It predicted Summer for every product: Summer F1 was `0.662815`, while Fall, Spring, and Winter F1 were all zero. Fold macro-F1 SD was only `0.000029`. **Strength:** B0 is cheap, deterministic, leakage-safe, and makes the accuracy illusion visible. Its near-zero ECE only shows that training-fold priors match pooled class frequency. **Limitation:** it ignores every image and cannot discriminate four seasons or recover Spring. **Incremental handoff:** B1 must test the next EDA claim - whether image shape and colour contain useful signal - and every learned model must beat `0.165704` macro-F1. **Trace:** `results/evidence/task2/b0_majority/manifest.json`; runs `b0-majority-f0-s2753-8200c51534e3`, `b0-majority-f1-s2753-7c378e67d9e6`, `b0-majority-f2-s2753-e50abeab2421`, `b0-majority-f3-s2753-b2fb197c26c1`, and `b0-majority-f4-s2753-b068532dab0a`.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 5.1a B0 fold-level macro-F1 table

In [ ]:
from IPython.display import display

display(b0_fold_summary.loc[b0_fold_summary["metric"].eq("macro_f1")])

> **Interpretation — 5.1a B0 fold-level macro-F1 table**
>
> **Why this output exists.** This output isolates 5.1a b0 fold-level macro-f1 table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.
>
> **What it shows.** B0 predicts Summer for every row: pooled macro-F1 is 0.165704 and Spring, Fall, and Winter F1 are zero.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 5.1b B0 evidence figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(ROOT / b0_manifest["artifacts"]["figure"]["path"])))

> **Interpretation — 5.1b B0 evidence figure**
>
> **Why this output exists.** This output isolates 5.1b b0 evidence figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** In the confusion matrix, rows are true Season labels and columns are predictions; darker cells contain more rows. The companion bars are per-class F1 and the dashed line is pooled macro-F1.
>
> **What it shows.** B0 predicts Summer for every row: pooled macro-F1 is 0.165704 and Spring, Fall, and Winter F1 are zero.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 5.2 B1 HOG plus HSV linear-SVM baseline

B1 is selected next because the EDA suggests that product shape and colour may carry Season signal. It is a serious, interpretable classical baseline: a CNN must improve on it, not merely beat the image-blind majority rule. Its fixed features and uncalibrated scores motivate C1.

In [ ]:
b1_outputs = run_or_load_experiment(
    TASK2_CONFIG_DIR / "b1_hog_hsv_svm.json",
    mode="run_or_load",
)
b1_manifest = build_experiment_evidence(
    b1_outputs,
    registry_path=RUNS_CSV,
    expected_ids=season_development["id"],
    protected_ids=protected_ids,
    probability_note=(
        "LinearSVC decision scores are transformed with row-wise softmax only "
        "to satisfy a common OOF storage schema. They are not calibrated "
        "probabilities, so B1 NLL, Brier, and ECE are descriptive diagnostics "
        "only and cannot support calibration claims."
    ),
    calibration_claim_allowed=False,
    evidence_directory=TASK2_EVIDENCE_DIR / "b1_hog_hsv_svm",
    figure_directory=TASK2_FIGURE_DIR,
)
assert b1_manifest["coverage"]["row_count"] == len(season_development)
assert b1_manifest["coverage"]["protected_id_count"] == 0
assert not b1_manifest["calibration_claim_allowed"]
assert len(b1_manifest["run_ids"]) == len(set(b1_manifest["run_ids"])) == 5

with (ROOT / b1_manifest["artifacts"]["pooled_metrics"]["path"]).open(
    encoding="utf-8"
) as handle:
    b1_pooled_metrics = json.load(handle)
b1_fold_summary = pd.read_csv(
    ROOT / b1_manifest["artifacts"]["fold_summary"]["path"]
)
b1_registry = pd.read_csv(
    ROOT / b1_manifest["artifacts"]["registry_snapshot"]["path"]
)
b1_scorecard = pd.DataFrame(
    [{
        "experiment_id": b1_manifest["experiment_id"],
        "OOF products": b1_pooled_metrics["n_samples"],
        "accuracy": b1_pooled_metrics["accuracy"],
        "balanced_accuracy": b1_pooled_metrics["balanced_accuracy"],
        "macro_f1": b1_pooled_metrics["macro_f1"],
        "macro_f1_gain_over_B0": (
            b1_pooled_metrics["macro_f1"] - b0_pooled_metrics["macro_f1"]
        ),
        "Spring_precision": b1_pooled_metrics["per_class"]["Spring"]["precision"],
        "Spring_recall": b1_pooled_metrics["per_class"]["Spring"]["recall"],
        "Spring_f1": b1_pooled_metrics["per_class"]["Spring"]["f1"],
        "five_fold_runtime_minutes": b1_registry["runtime_seconds"].sum() / 60,
        "linear_parameters": int(b1_registry["parameter_count"].iloc[0]),
    }]
)
display(b1_scorecard)


# Additional single-output views rendered below:
# display(b1_fold_summary.loc[b1_fold_summary["metric"].eq("macro_f1")])
# display(Image(filename=str(ROOT / b1_manifest["artifacts"]["figure"]["path"])))


> **Interpretation:** **Why this baseline:** B1 was selected next because HOG tests shape and HSV histograms test colour, directly converting the EDA into a classical image-only experiment. **Result:** it covered all 32,753 valid development products once with no protected IDs. It reached `0.609561` pooled macro-F1, `0.657405` accuracy, and `0.620671` balanced accuracy: a `0.443857` absolute macro-F1 gain over B0. Spring precision, recall, and F1 were `0.423098`, `0.573363`, and `0.486901`; pooled F1 was `0.553375` for Fall, `0.714584` for Summer, and `0.683385` for Winter. Fold macro-F1 SD was `0.006962`. **Strength:** B1 proves that deterministic shape and colour features contain substantial Season signal and creates a serious threshold for deep models. **Limitation:** HOG/HSV cannot learn task-specific features, may exploit catalogue style, and its softmax-transformed LinearSVC scores are not calibrated probabilities. It still sends 3,664 Fall and 1,589 Winter products to Summer, and Spring remains weakest. **Incremental handoff:** C1 must learn features end to end and beat `0.609561`, or offer a clear robustness, efficiency, or minority-class benefit. **Trace:** `results/evidence/task2/b1_hog_hsv_svm/manifest.json`; runs `b1-hog-hsv-svm-f0-s2753-b23d1eafe35f`, `b1-hog-hsv-svm-f1-s2753-5735c51cc237`, `b1-hog-hsv-svm-f2-s2753-3f214f6a2bd4`, `b1-hog-hsv-svm-f3-s2753-83fd8ce0b942`, and `b1-hog-hsv-svm-f4-s2753-45b2e5123926`.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 5.2a B1 fold-level macro-F1 table

In [ ]:
from IPython.display import display

display(b1_fold_summary.loc[b1_fold_summary["metric"].eq("macro_f1")])

> **Interpretation — 5.2a B1 fold-level macro-F1 table**
>
> **Why this output exists.** This output isolates 5.2a b1 fold-level macro-f1 table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.
>
> **What it shows.** B1 raises pooled macro-F1 to 0.609561, proving that simple shape and colour carry Season signal, but it remains a fixed-feature ceiling.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 5.2b B1 evidence figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(ROOT / b1_manifest["artifacts"]["figure"]["path"])))

> **Interpretation — 5.2b B1 evidence figure**
>
> **Why this output exists.** This output isolates 5.2b b1 evidence figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** In the confusion matrix, rows are true Season labels and columns are predictions; darker cells contain more rows. The companion bars are per-class F1 and the dashed line is pooled macro-F1.
>
> **What it shows.** B1 raises pooled macro-F1 to 0.609561, proving that simple shape and colour carry Season signal, but it remains a fixed-feature ceiling.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 6. Scratch model families

Define three different deep-learning capacities and prove that the submitted candidates start from random weights.

### 6.1 Define the scratch model factories

Each architecture is a separate implementation unit and therefore receives its own subsubsection and code cell.

#### 6.1.1 C1 four-block SmallCNN

C1 is the simple deep baseline and makes capacity and failure analysis easy to explain.

In [ ]:
from fashion.models.season import (
    SeasonModelSpec,
    assert_final_model,
    build_season_model,
)

smallcnn = build_season_model(
    SeasonModelSpec(family="smallcnn", num_classes=len(SEASON_LABELS))
)
smallcnn_boundary = assert_final_model(smallcnn)
with torch.inference_mode():
    smallcnn_logits = smallcnn(torch.zeros(2, 3, 80, 60))
assert tuple(smallcnn_logits.shape) == (2, len(SEASON_LABELS))
display(pd.DataFrame([{**smallcnn_boundary, "output_shape": tuple(smallcnn_logits.shape)}]))

> **Interpretation:** C1 returns four logits for each image, is final-eligible, and is created from scratch with `weights=None`. It follows B1 because its convolutions learn task-specific features instead of freezing HOG and HSV by hand. Its strength is a compact, easy-to-audit capacity baseline. Its limitation is that four simple blocks may miss longer-range structure; C2 therefore tests whether residual depth adds useful discrimination, while C3 provides an efficiency alternative.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 6.1.2 C2 ResNet18 with a small-image stem

C2 tests residual learning without discarding small-image detail in the first layers.

In [ ]:
from torch import nn

resnet18_small_stem = build_season_model(
    SeasonModelSpec(family="resnet18_small_stem", num_classes=len(SEASON_LABELS))
)
resnet_boundary = assert_final_model(resnet18_small_stem)
stem = resnet18_small_stem.backbone.conv1
assert stem.kernel_size == (3, 3) and stem.stride == (1, 1)
assert isinstance(resnet18_small_stem.backbone.maxpool, nn.Identity)
with torch.inference_mode():
    resnet_logits = resnet18_small_stem(torch.zeros(2, 3, 80, 60))
assert tuple(resnet_logits.shape) == (2, len(SEASON_LABELS))
resnet_summary = {
    **resnet_boundary,
    "stem": "3x3, stride 1, no max-pool",
    "output_shape": tuple(resnet_logits.shape),
}
display(pd.DataFrame([resnet_summary]))

> **Interpretation:** C2 also returns four logits and is final-eligible, scratch-trained with `weights=None`. The `3x3`, stride-1 stem and removed max-pool preserve more of each 60 x 80 catalogue image than the standard ImageNet stem. C2 follows C1 to test whether deeper residual features solve C1's capacity limit. Its strength is higher representational capacity; its limitation is much greater cost, so G1 must measure whether the gain is large enough rather than assume that a larger model is better.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 6.1.3 C3 MobileNetV3-Small

C3 tests whether a mobile architecture offers a better quality-to-latency trade-off.

In [ ]:
mobilenet_v3_small = build_season_model(
    SeasonModelSpec(family="mobilenet_v3_small", num_classes=len(SEASON_LABELS))
)
mobilenet_boundary = assert_final_model(mobilenet_v3_small)
with torch.inference_mode():
    mobilenet_logits = mobilenet_v3_small(torch.zeros(2, 3, 80, 60))
assert tuple(mobilenet_logits.shape) == (2, len(SEASON_LABELS))
display(pd.DataFrame([{**mobilenet_boundary, "output_shape": tuple(mobilenet_logits.shape)}]))

> **Interpretation:** C3 returns four logits, is final-eligible, and is built from scratch with `weights=None`. It is not assumed to be an upgrade over C2; it is an alternative that tests whether a mobile architecture can preserve enough quality with lower training memory and deployment cost. Its strength should be efficiency, while its limitation may be underfitting this weak-visual-signal target. G1 therefore measures macro-F1, runtime, and VRAM instead of choosing it from architecture reputation.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 6.2 Audit shapes, parameters, and random initialisation

A forward-pass and weight-provenance audit catches architecture mistakes before GPU runs.

In [ ]:
def state_dict_sha256(model):
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        values = tensor.detach().cpu().contiguous().numpy()
        digest.update(name.encode("utf-8"))
        digest.update(str(values.dtype).encode("ascii"))
        digest.update(str(values.shape).encode("ascii"))
        digest.update(values.tobytes())
    return digest.hexdigest()

model_families = {
    "C1": smallcnn,
    "C2": resnet18_small_stem,
    "C3": mobilenet_v3_small,
}
real_images = validation_batch["image"][:2]
synthetic_images = torch.zeros(2, 3, 128, 96)
model_audit_rows = []
for model_id, model in model_families.items():
    model.eval()
    with torch.inference_mode():
        real_logits = model(real_images)
        synthetic_logits = model(synthetic_images)
    assert real_logits.shape == synthetic_logits.shape == (2, len(SEASON_LABELS))
    assert torch.isfinite(real_logits).all() and torch.isfinite(synthetic_logits).all()
    boundary = assert_final_model(model)
    model_audit_rows.append(
        {
            "model_id": model_id,
            "class": type(model).__name__,
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
            "trainable_parameters": sum(
                parameter.numel()
                for parameter in model.parameters()
                if parameter.requires_grad
            ),
            "initial_state_sha256": state_dict_sha256(model),
            **boundary,
            "p0_forward_pass": True,
            "p1_forward_pass": True,
        }
    )
scratch_model_audit = pd.DataFrame(model_audit_rows)
atomic_write_csv(TASK2_EVIDENCE_DIR / "scratch_model_audit.csv", scratch_model_audit)
display(scratch_model_audit)

> **Interpretation to write after running:** Compare capacity and expected deployment cost. Confirm that each submitted candidate is scratch-trained.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 6.3 Declare the pretrained benchmark boundary

A pretrained comparison is useful only when it is clearly excluded from final eligibility.

In [ ]:
from fashion.models.season import (
    BenchmarkModelSpec,
    ModelBoundaryError,
    build_benchmark_model,
    model_boundary_audit,
)

standard_stem_control = build_benchmark_model(
    BenchmarkModelSpec(family="resnet18_standard_scratch", num_classes=len(SEASON_LABELS))
)
standard_control_boundary = model_boundary_audit(standard_stem_control)
try:
    assert_final_model(standard_stem_control)
except ModelBoundaryError as boundary_error:
    benchmark_rejection = str(boundary_error)
else:
    raise AssertionError("benchmark model entered the final-model boundary")

pretrained_spec = BenchmarkModelSpec(
    family="resnet18_standard_pretrained",
    num_classes=len(SEASON_LABELS),
)
pretrained_spec.validate()
benchmark_boundary = pd.DataFrame(
    [
        {
            "experiment": "P0S",
            "family": "resnet18_standard_scratch",
            "training_origin": "scratch",
            "weights": None,
            "benchmark_only": True,
            "final_eligible": False,
        },
        {
            "experiment": "P*",
            "family": pretrained_spec.family,
            "training_origin": "imagenet_pretrained",
            "weights": "ResNet18_Weights.DEFAULT",
            "benchmark_only": True,
            "final_eligible": False,
        },
    ]
)
assert benchmark_boundary["benchmark_only"].all() and not benchmark_boundary["final_eligible"].any()
benchmark_boundary["final_boundary_rejection"] = benchmark_rejection
atomic_write_csv(TASK2_EVIDENCE_DIR / "benchmark_boundary.csv", benchmark_boundary)
display(benchmark_boundary)

> **Interpretation to write after running:** Explain what the pretrained gap says about data and representation learning, without selecting P* as the final model.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

## 7. Training and run registry

Prove the shared engine, checkpointing, failure handling, and registry before long runs.

### 7.1 G0 tiny-batch smoke test

A tiny overfit test catches label order, image mapping, loss, and backpropagation errors cheaply.

In [ ]:
from fashion.config import RUNS_CSV, TASK2_CONFIG_DIR
from fashion.task2.evidence import build_g0_evidence
from fashion.task2.smoke import run_or_load_g0_smoke

g0_result = run_or_load_g0_smoke(
    TASK2_CONFIG_DIR / "g0_pipeline_smoke.json",
    mode="run_or_load",
)
g0_manifest = build_g0_evidence(g0_result, registry_path=RUNS_CSV)
g0_summary = {
    "run_id": g0_result.run_id,
    "source": g0_result.source,
    "passed": g0_result.passed,
    "tiny_final_accuracy": g0_result.tiny_overfit["final_accuracy"],
    "tiny_loss_ratio": g0_result.tiny_overfit["loss_ratio"],
    "integration_epochs": g0_result.integration["epochs_completed"],
    "integration_macro_f1_non_comparison": (
        g0_result.integration["best_metrics"]["macro_f1"]
    ),
    "checkpoint_sha256": g0_result.artifacts["checkpoint"],
}
display(pd.Series(g0_summary, name="G0 pipeline gate").to_frame())


# Additional single-output views rendered below:
# display(Image(filename=str(ROOT / g0_manifest["figure_path"])))


> **Interpretation:** **Result —** clean run `g0-pipeline-smoke-f0-s2753-5ad5ee9d433c` passed: the balanced 64-image batch reached 100% accuracy after 100 steps, its final loss was `0.0000145` of the initial loss, gradients stayed finite, and the 512-image integration run completed two epochs with hash-verified artifacts. **Meaning —** the loader-cache implementation change correctly invalidated the stale G0 cache; the external rerun closed the current functional gate, and this Run All reused its verified bytes. **Provenance —** the selected registry snapshot records commit `7d70e3f`, `git_dirty=false`, the current implementation hash, and `status=completed`. The earlier dirty row remains in the append-only registry but supports no claim. **Decision —** G0 unlocks the measured experiments. Its integration macro-F1 is deliberately excluded from every leaderboard. **Limitation —** G0 uses small balanced subsets and fold 0 only, so it says nothing about comparative model quality or five-fold stability. **Trace —** `results/evidence/task2/g0/manifest.json`, its one-row clean registry snapshot, and the run ID above.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 7.1a G0 smoke evidence figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(ROOT / g0_manifest["figure_path"])))

> **Interpretation — 7.1a G0 smoke evidence figure**
>
> **Why this output exists.** This output isolates 7.1a g0 smoke evidence figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** In the confusion matrix, rows are true Season labels and columns are predictions; darker cells contain more rows. The companion bars are per-class F1 and the dashed line is pooled macro-F1.
>
> **What it shows.** G0 passes the overfit and integration gates; its tiny sample is deliberately excluded from model comparison.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 7.2 Audit registry and checkpoint traceability

Every reported number must be traceable to configuration, split, code, and artifact hashes.

In [ ]:
from fashion.config import RUNS_CSV
from fashion.train.registry import RUN_COLUMNS, RUN_STATUSES, RunRegistry

run_registry = RunRegistry(RUNS_CSV)
registry_rows = run_registry.read()
schema_matches = tuple(registry_rows.columns) == RUN_COLUMNS
unique_run_ids = not registry_rows["run_id"].duplicated().any()
known_statuses = set(registry_rows["status"]) <= RUN_STATUSES
declared_checkpoints = registry_rows.loc[registry_rows["checkpoint_path"].ne(""), "checkpoint_path"]
checkpoint_paths_exist = all((ROOT / path).is_file() for path in declared_checkpoints)
hash_columns = ["config_sha256", "split_sha256", "label_map_sha256", "implementation_sha256"]
identity_hashes_valid = registry_rows.empty or registry_rows[hash_columns].apply(
    lambda column: column.str.fullmatch(r"[0-9a-f]{64}").all()
).all()
registry_health = {
    "schema_version": "1.0.0",
    "registry_path": RUNS_CSV.relative_to(ROOT).as_posix(),
    "run_count": len(registry_rows),
    "schema_matches": bool(schema_matches),
    "unique_run_ids": bool(unique_run_ids),
    "known_statuses": bool(known_statuses),
    "identity_hashes_valid": bool(identity_hashes_valid),
    "declared_checkpoints_exist": bool(checkpoint_paths_exist),
    "status_counts": registry_rows["status"].value_counts().sort_index().to_dict(),
}
assert all(
    registry_health[key]
    for key in (
        "schema_matches",
        "unique_run_ids",
        "known_statuses",
        "identity_hashes_valid",
        "declared_checkpoints_exist",
    )
)
atomic_write_json(TASK2_EVIDENCE_DIR / "registry_health.json", registry_health)
display(pd.Series(registry_health, name="Registry health").to_frame())


# Additional single-output views rendered below:
# display(registry_rows.tail(10))


> **Interpretation to write after running:** Confirm that each future table can be regenerated from run IDs. Explain any failed run honestly.

> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.

#### 7.2a Registry lifecycle trace

In [ ]:
from IPython.display import display

display(registry_rows.tail(10).loc[:, [
        "experiment_id", "fold", "run_id", "status",
        "primary_metric_value", "git_dirty",
    ]])

> **Interpretation — 7.2a Registry lifecycle trace**
>
> **Why this output exists.** This output isolates 7.2a registry lifecycle trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** The ledger shows terminal, unique run identities and keeps failed or interrupted attempts instead of silently deleting them.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 8. Controlled experiment matrix

Execute broad comparisons under equal budgets, then spend extra compute only on justified finalists.

### 8.1 G1 and G3 model-family comparison

Screening and full finalist training answer different questions, so they are separated below.

#### 8.1.1 G1 equal-budget family screening

The screen decides which families deserve more compute under one fixed budget.

In [ ]:
from fashion.task2.evidence import build_g1_family_screen_evidence

g1_specs = {
    "g1-c1-smallcnn": ("g1_c1_smallcnn.json", "g1_c1_smallcnn"),
    "g1-c2-resnet18": ("g1_c2_resnet18.json", "g1_c2_resnet18"),
    "g1-c3-mobilenetv3": ("g1_c3_mobilenetv3.json", "g1_c3_mobilenetv3"),
}
g1_probability_note = (
    "Softmax probabilities come from each fold's best validation macro-F1 "
    "checkpoint. They are uncalibrated until the dedicated cross-fitted "
    "calibration gate, so current NLL, Brier, and ECE are diagnostics only."
)
g1_manifests = {}
g1_trace_rows = []
for experiment_id, (config_name, evidence_slug) in g1_specs.items():
    outputs = run_or_load_experiment(
        TASK2_CONFIG_DIR / config_name,
        mode="run_or_load",
    )
    experiment_manifest = build_experiment_evidence(
        outputs,
        registry_path=RUNS_CSV,
        expected_ids=season_development["id"],
        protected_ids=protected_ids,
        probability_note=g1_probability_note,
        calibration_claim_allowed=False,
        evidence_directory=TASK2_EVIDENCE_DIR / evidence_slug,
        figure_directory=TASK2_FIGURE_DIR,
    )
    assert experiment_manifest["experiment_id"] == experiment_id
    assert experiment_manifest["coverage"]["row_count"] == len(season_development)
    assert experiment_manifest["coverage"]["protected_id_count"] == 0
    g1_manifests[experiment_id] = experiment_manifest
    g1_trace_rows.extend(
        {
            "experiment_id": experiment_id,
            "fold": output.fold,
            "run_id": output.run_id,
            "source": output.source,
            "fold_macro_f1": output.metrics["macro_f1"],
        }
        for output in outputs
    )

g1_screen_manifest = build_g1_family_screen_evidence(
    [TASK2_EVIDENCE_DIR / slug / "manifest.json" for _, slug in g1_specs.values()],
    reference_manifest_path=TASK2_EVIDENCE_DIR / "b1_hog_hsv_svm/manifest.json",
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "g1_family_screen",
    figure_directory=TASK2_FIGURE_DIR,
)
g1_leaderboard = pd.read_csv(
    ROOT / g1_screen_manifest["artifacts"]["leaderboard"]["path"]
)
with (ROOT / g1_screen_manifest["artifacts"]["shortlist"]["path"]).open(
    encoding="utf-8"
) as handle:
    g1_shortlist = json.load(handle)
assert g1_shortlist["selected_experiment_ids"] == [
    "g1-c2-resnet18",
    "g1-c1-smallcnn",
]
assert g1_shortlist["rejected_experiment_ids"] == ["g1-c3-mobilenetv3"]
assert len(g1_trace_rows) == 15
display(g1_leaderboard.loc[:, [
        "rank", "experiment_id", "model_family", "pooled_macro_f1",
        "fold_sd_macro_f1", "spring_f1", "parameter_count",
        "five_fold_runtime_minutes", "shortlisted",
    ]].style.format({
        "pooled_macro_f1": "{:.4f}",
        "fold_sd_macro_f1": "{:.4f}",
        "spring_f1": "{:.4f}",
        "five_fold_runtime_minutes": "{:.2f}",
    }))


# Additional single-output views rendered below:
# display(pd.DataFrame(g1_trace_rows).sort_values(["experiment_id", "fold"]))
# display(
#     Image(filename=str(ROOT / g1_screen_manifest["artifacts"]["figure"]["path"]))
# )


> **Interpretation:** **Result:** all three scratch families produced exactly 32,753 pooled OOF predictions with zero protected IDs under the same P0/A0, seed 2753, five-fold, eight-epoch protocol. C2 ResNet18 ranked first (`0.707099` pooled macro-F1; fold SD `0.003872`; Spring F1 `0.738433`), followed by C1 SmallCNN (`0.699902`; SD `0.010936`; Spring F1 `0.726168`) and C3 MobileNetV3-Small (`0.638495`; SD `0.008339`; Spring F1 `0.668816`). All beat B1 (`0.609561`), by `0.097538`, `0.090341`, and `0.028934`, respectively. **Meaning:** residual capacity gave C2 the best and most stable screen score, but its `11,170,884` parameters and `29.21` training minutes bought only `0.007197` macro-F1 over C1, which used `1,174,244` parameters and `19.11` minutes. C3 was smallest in measured VRAM (`74.3` MB) and fastest (`16.66` minutes), but trailed C2 by `0.068604`; efficiency did not offset that quality loss. **Decision:** shortlist C2 as the quality leader and C1 as the compact efficiency challenger. Reject C3 from further tuning. Run P0/P1 and A0/A1 on C2, then compare C2 and C1 under the same full G3 budget. **Limitation:** this is one seed, one transform, and an eight-epoch screen; probabilities are not yet calibrated, so the screen cannot freeze the final winner. **Trace:** `results/evidence/task2/g1_family_screen/manifest.json` links the 15 immutable run IDs shown above. C1 runs: `g1-c1-smallcnn-f0-s2753-59e9743d3899`, `g1-c1-smallcnn-f1-s2753-f73cfae4a798`, `g1-c1-smallcnn-f2-s2753-8e178abd5435`, `g1-c1-smallcnn-f3-s2753-bf3525f7f30f`, `g1-c1-smallcnn-f4-s2753-476a7394ed3e`; C2 runs: `g1-c2-resnet18-f0-s2753-b91662d47026`, `g1-c2-resnet18-f1-s2753-a4ff07c863e4`, `g1-c2-resnet18-f2-s2753-2d6a8dffa3fe`, `g1-c2-resnet18-f3-s2753-f2eb5f7a2dac`, `g1-c2-resnet18-f4-s2753-2449eed2ce12`; C3 runs: `g1-c3-mobilenetv3-f0-s2753-9e07fe2a3158`, `g1-c3-mobilenetv3-f1-s2753-7f3f65f2ebbb`, `g1-c3-mobilenetv3-f2-s2753-35dc3f614c96`, `g1-c3-mobilenetv3-f3-s2753-a90ee2f892fb`, and `g1-c3-mobilenetv3-f4-s2753-bf6c9229b22b`.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.1.1a G1 fold trace

In [ ]:
import pandas as pd

from IPython.display import display

display(pd.DataFrame(g1_trace_rows).sort_values(["experiment_id", "fold"]))

> **Interpretation — 8.1.1a G1 fold trace**
>
> **Why this output exists.** This output isolates 8.1.1a g1 fold trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.
>
> **What it shows.** C2 leads the eight-epoch screen at 0.707099, C1 is close and much smaller, and C3 is rejected at 0.638495.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.1b G1 family-screen figure

In [ ]:
from IPython.display import Image, display

display(
    Image(filename=str(ROOT / g1_screen_manifest["artifacts"]["figure"]["path"]))
)

> **Interpretation — 8.1.1b G1 family-screen figure**
>
> **Why this output exists.** This output isolates 8.1.1b g1 family-screen figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is trainable parameters on a log scale and the y-axis is pooled OOF macro-F1. Each labelled point is one scratch family; the dashed red line is B1.
>
> **What it shows.** C2 leads the eight-epoch screen at 0.707099, C1 is close and much smaller, and C3 is rejected at 0.638495.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2 G3 full-budget finalist comparison

The finalist run tests the strongest families with equal training opportunity.

In [ ]:
from fashion.task2.evidence import build_g3_full_budget_evidence
from fashion.train.registry import RunRegistry

g3_specs = {
    "g3-c1-t1-smallcnn": ("g3_c1_t1_smallcnn.json", "g3_c1_t1_smallcnn"),
    "g3-c2-t0-resnet18": ("g3_c2_t0_resnet18.json", "g3_c2_t0_resnet18"),
}
g3_probability_note = (
    "Uncalibrated softmax probabilities from the best validation macro-F1 "
    "checkpoint in each fold; calibration metrics are diagnostic only."
)
g3_registry = RunRegistry(RUNS_CSV)
g3_attempt_frames = []
g3_current_attempt_frames = []
for experiment_id, (config_name, evidence_slug) in g3_specs.items():
    outputs = run_or_load_experiment(
        TASK2_CONFIG_DIR / config_name,
        mode="run_or_load",
    )
    experiment_manifest = build_experiment_evidence(
        outputs,
        registry_path=RUNS_CSV,
        expected_ids=season_development["id"],
        protected_ids=protected_ids,
        probability_note=g3_probability_note,
        calibration_claim_allowed=False,
        evidence_directory=TASK2_EVIDENCE_DIR / evidence_slug,
        figure_directory=TASK2_FIGURE_DIR,
    )
    assert experiment_manifest["experiment_id"] == experiment_id
    assert experiment_manifest["coverage"]["row_count"] == len(season_development)
    assert experiment_manifest["coverage"]["protected_id_count"] == 0
    assert len(experiment_manifest["run_ids"]) == 5

    current_run_ids = {output.run_id for output in outputs}
    assert len(current_run_ids) == 5
    assert set(experiment_manifest["run_ids"]) == current_run_ids
    attempts = g3_registry.find(experiment_id=experiment_id)
    current_attempts = attempts.loc[
        attempts["run_id"].isin(current_run_ids)
    ].copy()
    assert len(current_attempts) == 5
    assert current_attempts["status"].eq("completed").all()
    assert set(current_attempts["fold"].astype(int)) == set(range(5))
    assert not attempts["status"].eq("running").any()
    g3_current_attempt_frames.append(current_attempts)
    g3_attempt_frames.append(attempts)

g3_manifest = build_g3_full_budget_evidence(
    experiment_manifest_paths=[
        TASK2_EVIDENCE_DIR / evidence_slug / "manifest.json"
        for _, evidence_slug in g3_specs.values()
    ],
    experiment_config_paths=[
        TASK2_CONFIG_DIR / config_name
        for config_name, _ in g3_specs.values()
    ],
    tuning_manifest_path=TASK2_EVIDENCE_DIR / "g2_compact_tuning/manifest.json",
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "g3_full_budget",
    figure_directory=TASK2_FIGURE_DIR,
)
g3_leaderboard = pd.read_csv(
    ROOT / g3_manifest["artifacts"]["leaderboard"]["path"]
)
g3_paired_folds = pd.read_csv(
    ROOT / g3_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
g3_per_class = pd.read_csv(
    ROOT / g3_manifest["artifacts"]["per_class_comparison"]["path"]
)
g3_screen_to_full = pd.read_csv(
    ROOT / g3_manifest["artifacts"]["screen_to_full_budget"]["path"]
)
with (ROOT / g3_manifest["artifacts"]["decision"]["path"]).open(
    encoding="utf-8"
) as handle:
    g3_decision = json.load(handle)

assert g3_manifest["gate"] == "G3-F"
assert g3_manifest["near_tie"] is True
assert g3_manifest["ultimate_winner_frozen"] is False
assert g3_manifest["provisional_reference_experiment_id"] == "g3-c1-t1-smallcnn"
assert g3_decision["common_five_fold_horizons"] == {"C1": 18, "C2": 19}
assert len(g3_paired_folds) == 5
assert len(g3_per_class) == len(SEASON_LABELS)

g3_current_attempts = pd.concat(g3_current_attempt_frames, ignore_index=True)
assert len(g3_current_attempts) == 10
assert g3_current_attempts["run_id"].nunique() == 10
assert g3_current_attempts["status"].eq("completed").all()
g3_attempts = pd.concat(g3_attempt_frames, ignore_index=True)
g3_interrupted_attempts = g3_attempts.loc[
    g3_attempts["status"].eq("interrupted")
].copy()
display(g3_leaderboard.loc[:, [
        "family", "tuning_id", "experiment_id", "pooled_macro_f1",
        "fold_sd_macro_f1", "spring_f1", "full_minus_screen_macro_f1",
        "median_best_epoch", "parameter_count", "five_fold_runtime_minutes",
        "provisional_reference",
    ]].style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "full_minus_screen_macro_f1": "{:+.6f}",
        "five_fold_runtime_minutes": "{:.2f}",
    }))


# Additional single-output views rendered below:
# display(g3_screen_to_full.style.format({"pooled_macro_f1": "{:.6f}"}))
# display(g3_paired_folds)
# display(g3_per_class.style.format({"delta_c1_minus_c2_f1": "{:+.6f}"}))
# display(pd.json_normalize(g3_decision, sep=".").transpose())
# display(
#     g3_attempts.loc[:, [
#         "experiment_id",
#         "fold",
#         "run_id",
#         "status",
#         "error_type",
#         "git_commit",
#         "git_dirty",
#         "primary_metric_value",
#     ]].sort_values(["experiment_id", "fold", "status"])
# )
# display(
#     Image(filename=str(ROOT / g3_manifest["artifacts"]["c1_learning_curves"]["path"]))
# )
# display(
#     Image(filename=str(ROOT / g3_manifest["artifacts"]["c2_learning_curves"]["path"]))
# )


> **Interpretation:** **Incremental question:** after G1 shortlisted C2 for quality and C1 for efficiency, and G2 selected C1-T1 and C2-T0, does equal full-budget training change that ordering when seed `2753` also owns model initialisation? **Result:** C1 rose from `0.708075` to `0.737661` pooled OOF macro-F1 (`+0.029586`), while C2 rose from `0.707099` to `0.735036` (`+0.027936`). C1 leads by `0.002626`, below the frozen `0.005` near-tie threshold. **Class trade-off:** C1 is better on Fall (`+0.011802` F1) and Winter (`+0.004266`); C2 is better on Spring (`+0.002571`) and Summer (`+0.002993`). One pooled score therefore does not erase minority-class risk.

**Learning curves and EDA reflection:** the teacher-style left panels show five-fold mean plus SD for train and validation cross-entropy loss; the right panels show validation accuracy and the primary validation macro-F1. Both families still improve after the eight-epoch screen. Accuracy remains above macro-F1, so the EDA warning that the large Summer class can hide minority errors was accurate. The EDA expectation that a larger network should clearly help was not supported: mature C2 is still slightly behind compact C1. Training accuracy is absent because it was not logged; no value was invented. C1 is shown through epoch 18 and C2 through epoch 19, the last epoch shared by all five folds.

**Efficiency and decision:** C1 uses `1,174,244` parameters and `60.21` five-fold minutes; C2 uses `11,170,884` parameters and `91.58` minutes. C2 is `9.51×` larger and `1.52×` slower. Keep C1-T1 as the provisional reference, not the ultimate winner. C3 remains rejected because its equal-budget screen trailed C2 by `0.068604` and no later controlled result justifies full-budget compute. Retain C2 because it is slightly stronger on Spring and Summer. Next run I1 class balancing and I2 masked multi-task learning, then close second-seed, robustness, cost, and grouped-bootstrap checks before freezing. **Real process:** the first G3 attempt is preserved but invalid because its seed did not own model initialisation; it includes run `g3-c1-t1-smallcnn-f2-s2753-2283b6495a44`, which was interrupted outside Python, and clean replacement `g3-c1-t1-smallcnn-f2-s2753-e46d771b6d56`. The corrected ten-run evidence uses implementation commit `47442a2`. Its build exposed a separate minimum-delta history-audit bug, traced by commits `c46a0ff`, `a9c4800`, and `62e57e2`; no retraining was needed because the checkpoints were valid. **Trace:** `results/evidence/task2/g3_full_budget/manifest.json` hashes the two input manifests, two selected configs, ten corrected completed run records, paired folds, per-class results, full histories, the decision, and both learning-curve figures. Representative corrected runs are `g3-c1-t1-smallcnn-f0-s2753-1ce4f9978b12` and `g3-c2-t0-resnet18-f0-s2753-66ee7a85d5c6`. Probabilities remain uncalibrated diagnostics.

> **Chart guide.** The axis labels name the compared metric and candidate; positive candidate-minus-reference values favour the candidate, while zero means no observed change.

#### 8.1.2a G3 screen-to-full score table

In [ ]:
from IPython.display import display

display(g3_screen_to_full.style.format({"pooled_macro_f1": "{:.6f}"}))

> **Interpretation — 8.1.2a G3 screen-to-full score table**
>
> **Why this output exists.** This output isolates 8.1.2a g3 screen-to-full score table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Candidate/variant names the row; `pooled_macro_f1` is computed across all OOF rows; `fold_sd_macro_f1` is five-fold variation; Spring fields expose minority performance; delta is candidate minus reference; selected/eligible fields apply the frozen rule.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2b G3 paired-fold table

In [ ]:
from IPython.display import display

display(g3_paired_folds)

> **Interpretation — 8.1.2b G3 paired-fold table**
>
> **Why this output exists.** This output isolates 8.1.2b g3 paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2c G3 per-class comparison

In [ ]:
from IPython.display import display

display(g3_per_class.style.format({"delta_c1_minus_c2_f1": "{:+.6f}"}))

> **Interpretation — 8.1.2c G3 per-class comparison**
>
> **Why this output exists.** This output isolates 8.1.2c g3 per-class comparison so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2d G3 decision audit

In [ ]:
import pandas as pd

from IPython.display import display

g3_decision_view = pd.DataFrame([{
    "gate": g3_decision["gate"],
    "provisional_reference": g3_decision["provisional_reference_family"],
    "c1_minus_c2_macro_f1": g3_decision["observed_c1_minus_c2_macro_f1"],
    "near_tie_threshold": g3_decision["near_tie_threshold"],
    "near_tie": g3_decision["near_tie"],
    "ultimate_winner_frozen": g3_decision["ultimate_winner_frozen"],
}])
display(g3_decision_view.style.format({
    "c1_minus_c2_macro_f1": "{:+.6f}",
    "near_tie_threshold": "{:.6f}",
}))

> **Interpretation — 8.1.2d G3 decision audit**
>
> **Why this output exists.** This output isolates 8.1.2d g3 decision audit so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `gate` names the frozen rule; delta columns are candidate minus reference; boolean fields show whether thresholds passed, selection changed, or a winner was frozen.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2e G3 run trace

In [ ]:
from IPython.display import display

display(
    g3_attempts.loc[:, [
        "experiment_id",
        "fold",
        "run_id",
        "status",
        "error_type",
        "git_commit",
        "git_dirty",
        "primary_metric_value",
    ]].sort_values(["experiment_id", "fold", "status"])
)

> **Interpretation — 8.1.2e G3 run trace**
>
> **Why this output exists.** This output isolates 8.1.2e g3 run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2f C1 full-budget learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(filename=str(ROOT / g3_manifest["artifacts"]["c1_learning_curves"]["path"]))
)

> **Interpretation — 8.1.2f C1 full-budget learning curve**
>
> **Why this output exists.** This output isolates 8.1.2f c1 full-budget learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.1.2g C2 full-budget learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(filename=str(ROOT / g3_manifest["artifacts"]["c2_learning_curves"]["path"]))
)

> **Interpretation — 8.1.2g C2 full-budget learning curve**
>
> **Why this output exists.** This output isolates 8.1.2g c2 full-budget learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** Full-budget C1 scores 0.737661 versus C2 at 0.735036; the +0.002626 gap is below the 0.005 near-tie threshold.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 8.2 G2 transform and compact tuning ablations

Input size, augmentation, and tuning are separate factors. Each receives its own controlled subsubsection.

#### 8.2.1 P0 versus P1 input size

This ablation tests whether moderate upscaling helps the small source images.

In [ ]:
from fashion.task2.evidence import build_g2_input_size_evidence

g2_probability_note = (
    "Softmax probabilities come from each fold's best validation macro-F1 "
    "checkpoint. They are uncalibrated until the dedicated cross-fitted "
    "calibration gate, so current NLL, Brier, and ECE are diagnostics only."
)
g2_p1_outputs = run_or_load_experiment(
    TASK2_CONFIG_DIR / "g2_p1_c2_resnet18.json",
    mode="run_or_load",
)
g2_p1_manifest = build_experiment_evidence(
    g2_p1_outputs,
    registry_path=RUNS_CSV,
    expected_ids=season_development["id"],
    protected_ids=protected_ids,
    probability_note=g2_probability_note,
    calibration_claim_allowed=False,
    evidence_directory=TASK2_EVIDENCE_DIR / "g2_p1_c2_resnet18",
    figure_directory=TASK2_FIGURE_DIR,
)
assert g2_p1_manifest["coverage"]["row_count"] == len(season_development)
assert g2_p1_manifest["coverage"]["protected_id_count"] == 0
assert len(g2_p1_manifest["run_ids"]) == 5

g2_size_manifest = build_g2_input_size_evidence(
    p0_manifest_path=TASK2_EVIDENCE_DIR / "g1_c2_resnet18/manifest.json",
    p1_manifest_path=TASK2_EVIDENCE_DIR / "g2_p1_c2_resnet18/manifest.json",
    p0_config_path=TASK2_CONFIG_DIR / "g1_c2_resnet18.json",
    p1_config_path=TASK2_CONFIG_DIR / "g2_p1_c2_resnet18.json",
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "g2_input_size_ablation",
    figure_directory=TASK2_FIGURE_DIR,
)
g2_size_comparison = pd.read_csv(
    ROOT / g2_size_manifest["artifacts"]["comparison"]["path"]
)
g2_size_paired_folds = pd.read_csv(
    ROOT / g2_size_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
with (ROOT / g2_size_manifest["artifacts"]["decision"]["path"]).open(
    encoding="utf-8"
) as handle:
    g2_size_decision = json.load(handle)
assert g2_size_decision["selected_variant"] == "P0"
assert g2_size_decision["minimum_gain"] == 0.005

g2_p1_attempts = RunRegistry(RUNS_CSV).find(
    experiment_id="g2-p1-c2-resnet18"
)
assert len(g2_p1_attempts.loc[g2_p1_attempts["status"].eq("completed")]) == 5
assert not g2_p1_attempts["status"].eq("running").any()
display(g2_size_comparison.loc[:, [
        "variant", "image_height", "image_width", "pooled_macro_f1",
        "fold_sd_macro_f1", "spring_f1", "delta_vs_p0_macro_f1",
        "five_fold_runtime_minutes", "peak_vram_mb", "runtime_ratio_vs_p0",
    ]].style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "delta_vs_p0_macro_f1": "{:+.6f}",
        "five_fold_runtime_minutes": "{:.2f}",
        "peak_vram_mb": "{:.1f}",
        "runtime_ratio_vs_p0": "{:.3f}",
    }))


# Additional single-output views rendered below:
# display(
#     g2_size_paired_folds.loc[:, [
#         "fold",
#         "p0_run_id",
#         "p1_run_id",
#         "p0_macro_f1",
#         "p1_macro_f1",
#         "delta_p1_minus_p0_macro_f1",
#         "runtime_ratio_p1_vs_p0",
#     ]]
# )
# display(
#     g2_p1_attempts.loc[:, [
#         "run_id",
#         "fold",
#         "status",
#         "git_commit",
#         "git_dirty",
#         "error_type",
#         "started_at_utc",
#     ]].sort_values(["fold", "started_at_utc"])
# )
# display(
#     Image(filename=str(ROOT / g2_size_manifest["artifacts"]["figure"]["path"]))
# )


> **Interpretation:** **Result:** P0 `(80, 60)` achieved `0.707099` pooled five-fold OOF macro-F1 (fold SD `0.003872`; Spring F1 `0.738433`). P1 `(128, 96)` achieved `0.705312` (SD `0.006467`; Spring F1 `0.738673`). The P1-minus-P0 pooled delta was `-0.001787` (-0.179 percentage points), and four of five paired fold deltas favoured P0; only fold 3 favoured P1 (`+0.004827`). P1's Spring gain was only `+0.000239` (+0.024 points). P1 took `58.18` training minutes versus `29.21` for P0 (`1.992` times) and peaked at `1,321.9` MB versus `606.4` MB VRAM (`2.180` times), with the same `11,170,884` parameters. **Meaning:** moderate upscaling added interpolation and almost doubled measured training cost without adding pooled discrimination. **Decision:** retain P0 because P1 did not meet the frozen `+0.005` macro-F1 rule; run A0 versus A1 on C2 at `(80, 60)`. **Limitation:** this is an eight-epoch, one-seed transform screen; its softmax outputs are uncalibrated and it does not replace later robustness analysis. **Trace:** `results/evidence/task2/g2_input_size_ablation/manifest.json` hashes both configs, both experiment manifests, the paired-fold table, decision, and figure. P1 evidence runs were `g2-p1-c2-resnet18-f0-s2753-67217738d381`, `g2-p1-c2-resnet18-f1-s2753-66a1b27539dc`, `g2-p1-c2-resnet18-f2-s2753-b5a856f9e3b1`, `g2-p1-c2-resnet18-f3-s2753-8e6b55d11e02`, and `g2-p1-c2-resnet18-f4-s2753-9294db7bbaf4`. Three non-evidence attempts remain visible in the local registry: `g2-p1-c2-resnet18-f1-s2753-7327d09ce2cf` and `g2-p1-c2-resnet18-f1-s2753-dce57b0c7883` were interrupted; `g2-p1-c2-resnet18-f1-s2753-48ece6ab6852` failed because the first Windows background runner lacked a safe `__main__` guard. None enters the five completed OOF rows.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.2.1a P0/P1 paired-fold table

In [ ]:
from IPython.display import display

display(
    g2_size_paired_folds.loc[:, [
        "fold",
        "p0_run_id",
        "p1_run_id",
        "p0_macro_f1",
        "p1_macro_f1",
        "delta_p1_minus_p0_macro_f1",
        "runtime_ratio_p1_vs_p0",
    ]]
)

> **Interpretation — 8.2.1a P0/P1 paired-fold table**
>
> **Why this output exists.** This output isolates 8.2.1a p0/p1 paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** P1 changes pooled macro-F1 by -0.001787 while almost doubling runtime, so P0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.1b P1 run trace

In [ ]:
from IPython.display import display

display(
    g2_p1_attempts.loc[:, [
        "run_id",
        "fold",
        "status",
        "git_commit",
        "git_dirty",
        "error_type",
        "started_at_utc",
    ]].sort_values(["fold", "started_at_utc"])
)

> **Interpretation — 8.2.1b P1 run trace**
>
> **Why this output exists.** This output isolates 8.2.1b p1 run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** P1 changes pooled macro-F1 by -0.001787 while almost doubling runtime, so P0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.1c P0/P1 size-ablation figure

In [ ]:
from IPython.display import Image, display

display(
    Image(filename=str(ROOT / g2_size_manifest["artifacts"]["figure"]["path"]))
)

> **Interpretation — 8.2.1c P0/P1 size-ablation figure**
>
> **Why this output exists.** This output isolates 8.2.1c p0/p1 size-ablation figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The left panel compares P0/P1 pooled and Spring F1 against the +0.005 rule. The right panel is P1-minus-P0 macro-F1 by fold; bars below zero favour P0.
>
> **What it shows.** P1 changes pooled macro-F1 by -0.001787 while almost doubling runtime, so P0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.2 A0 versus A1 augmentation

This ablation tests whether mild colour jitter helps generalisation without erasing genuine seasonal colour cues.

In [ ]:
from fashion.task2.evidence import build_g2_augmentation_evidence

g2_a1_outputs = run_or_load_experiment(
    TASK2_CONFIG_DIR / "g2_a1_c2_resnet18.json",
    mode="run_or_load",
)
g2_a1_manifest = build_experiment_evidence(
    g2_a1_outputs,
    registry_path=RUNS_CSV,
    expected_ids=season_development["id"],
    protected_ids=protected_ids,
    probability_note=g2_probability_note,
    calibration_claim_allowed=False,
    evidence_directory=TASK2_EVIDENCE_DIR / "g2_a1_c2_resnet18",
    figure_directory=TASK2_FIGURE_DIR,
)
assert g2_a1_manifest["coverage"]["row_count"] == len(season_development)
assert g2_a1_manifest["coverage"]["protected_id_count"] == 0
assert len(g2_a1_manifest["run_ids"]) == 5

g2_augmentation_manifest = build_g2_augmentation_evidence(
    a0_manifest_path=TASK2_EVIDENCE_DIR / "g1_c2_resnet18/manifest.json",
    a1_manifest_path=TASK2_EVIDENCE_DIR / "g2_a1_c2_resnet18/manifest.json",
    a0_config_path=TASK2_CONFIG_DIR / "g1_c2_resnet18.json",
    a1_config_path=TASK2_CONFIG_DIR / "g2_a1_c2_resnet18.json",
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "g2_augmentation_ablation",
    figure_directory=TASK2_FIGURE_DIR,
)
g2_augmentation_comparison = pd.read_csv(
    ROOT / g2_augmentation_manifest["artifacts"]["comparison"]["path"]
)
g2_augmentation_paired = pd.read_csv(
    ROOT / g2_augmentation_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
g2_augmentation_per_class = pd.read_csv(
    ROOT / g2_augmentation_manifest["artifacts"]["per_class_comparison"]["path"]
)
with (ROOT / g2_augmentation_manifest["artifacts"]["decision"]["path"]).open(
    encoding="utf-8"
) as handle:
    g2_augmentation_decision = json.load(handle)
assert g2_augmentation_decision["decision_status"] == "closed"
assert not g2_augmentation_decision["quality_gate_passed"]
assert g2_augmentation_decision["robustness_evidence_status"] == "not_required"
assert g2_augmentation_decision["selected_variant"] == "A0"
assert (g2_augmentation_paired["delta_a1_minus_a0_macro_f1"] < 0).all()

g2_a1_attempts = RunRegistry(RUNS_CSV).find(
    experiment_id="g2-a1-c2-resnet18"
)
assert len(g2_a1_attempts.loc[g2_a1_attempts["status"].eq("completed")]) == 5
assert not g2_a1_attempts["status"].eq("running").any()
display(g2_augmentation_comparison.loc[:, [
        "variant", "augmentation", "pooled_macro_f1", "fold_sd_macro_f1",
        "spring_f1", "delta_vs_a0_macro_f1", "five_fold_runtime_minutes",
        "peak_vram_mb",
    ]].style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "delta_vs_a0_macro_f1": "{:+.6f}",
        "five_fold_runtime_minutes": "{:.2f}",
        "peak_vram_mb": "{:.1f}",
    }))


# Additional single-output views rendered below:
# display(
#     g2_augmentation_paired.loc[:, [
#         "fold",
#         "a0_run_id",
#         "a1_run_id",
#         "a0_macro_f1",
#         "a1_macro_f1",
#         "delta_a1_minus_a0_macro_f1",
#     ]]
# )
# display(
#     g2_augmentation_per_class.loc[:, [
#         "label",
#         "support",
#         "a0_precision",
#         "a1_precision",
#         "delta_a1_minus_a0_precision",
#         "a0_recall",
#         "a1_recall",
#         "delta_a1_minus_a0_recall",
#         "a0_f1",
#         "a1_f1",
#         "delta_a1_minus_a0_f1",
#     ]].style.format(precision=6)
# )
# display(
#     g2_a1_attempts.loc[:, [
#         "run_id",
#         "fold",
#         "status",
#         "git_commit",
#         "git_dirty",
#         "primary_metric_value",
#     ]].sort_values("fold")
# )
# display(
#     Image(
#         filename=str(
#             ROOT / g2_augmentation_manifest["artifacts"]["figure"]["path"]
#         )
#     )
# )


> **Interpretation:** **Result:** A0 achieved `0.707099` pooled five-fold OOF macro-F1 (fold SD `0.003872`; Spring F1 `0.738433`). A1 achieved `0.696662` (SD `0.006778`; Spring F1 `0.727354`). The A1-minus-A0 pooled delta was `-0.010438` (-1.044 percentage points), far below the frozen `+0.003` requirement, and every paired fold favoured A0. A1 changed class F1 by Fall `-0.035008`, Spring `-0.011079`, Summer `+0.000352`, and Winter `+0.003985`. Spring recall rose slightly (`+0.006772`), but precision fell much more (`-0.050154`); Fall recall fell by `-0.068996`. **Meaning:** mild colour jitter did not act as useful regularisation here. It removed colour information that helped distinguish Fall and Spring, while its tiny Summer and Winter gains did not offset those losses. **Decision:** retain A0 at P0 `(80, 60)`. Robustness evidence is explicitly `not_required` because A1 already failed the first condition of the frozen AND rule; running extra robustness probes cannot make A1 eligible. **Limitation:** this is an eight-epoch, one-seed transform screen. The later second-seed finalist check still tests stability, but all five negative paired-fold deltas make the present rejection consistent rather than fold-specific. **Trace:** `results/evidence/task2/g2_augmentation_ablation/manifest.json` hashes both configs, both experiment manifests, the paired-fold table, per-class comparison, decision, and figure. The five A1 run IDs are displayed above and remain linked to clean registry rows.

> **Chart guide.** The quality panel compares A0/A1 with the frozen gain threshold; paired-fold bars show A1-minus-A0 macro-F1, so negative bars favour A0.

#### 8.2.2a A0/A1 paired-fold table

In [ ]:
from IPython.display import display

display(
    g2_augmentation_paired.loc[:, [
        "fold",
        "a0_run_id",
        "a1_run_id",
        "a0_macro_f1",
        "a1_macro_f1",
        "delta_a1_minus_a0_macro_f1",
    ]]
)

> **Interpretation — 8.2.2a A0/A1 paired-fold table**
>
> **Why this output exists.** This output isolates 8.2.2a a0/a1 paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** A1 changes pooled macro-F1 by -0.010438 and loses on all five folds, so A0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.2b A0/A1 per-class comparison

In [ ]:
from IPython.display import display

display(
    g2_augmentation_per_class.loc[:, [
        "label",
        "support",
        "a0_precision",
        "a1_precision",
        "delta_a1_minus_a0_precision",
        "a0_recall",
        "a1_recall",
        "delta_a1_minus_a0_recall",
        "a0_f1",
        "a1_f1",
        "delta_a1_minus_a0_f1",
    ]].style.format(precision=6)
)

> **Interpretation — 8.2.2b A0/A1 per-class comparison**
>
> **Why this output exists.** This output isolates 8.2.2b a0/a1 per-class comparison so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.
>
> **What it shows.** A1 changes pooled macro-F1 by -0.010438 and loses on all five folds, so A0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.2c A1 run trace

In [ ]:
from IPython.display import display

display(
    g2_a1_attempts.loc[:, [
        "run_id",
        "fold",
        "status",
        "git_commit",
        "git_dirty",
        "primary_metric_value",
    ]].sort_values("fold")
)

> **Interpretation — 8.2.2c A1 run trace**
>
> **Why this output exists.** This output isolates 8.2.2c a1 run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** A1 changes pooled macro-F1 by -0.010438 and loses on all five folds, so A0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.2d A0/A1 augmentation figure

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / g2_augmentation_manifest["artifacts"]["figure"]["path"]
        )
    )
)

> **Interpretation — 8.2.2d A0/A1 augmentation figure**
>
> **Why this output exists.** This output isolates 8.2.2d a0/a1 augmentation figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The quality panel compares A0/A1 with the frozen gain threshold; paired-fold bars show A1-minus-A0 macro-F1, so negative bars favour A0.
>
> **What it shows.** A1 changes pooled macro-F1 by -0.010438 and loses on all five folds, so A0 is retained.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3 Compact finalist tuning

A small predeclared search avoids an open-ended hunt for a lucky configuration. The learning-curve layout follows the familiar two-panel diagnostic: loss on the left and validation scores on the right. Macro-F1 is shown beside accuracy because it, not accuracy, is the frozen selection metric.

In [ ]:
from fashion.task2.evidence import (
    build_g2_tuning_evidence,
    build_task2_selection_story_evidence,
)
from fashion.train.registry import RunRegistry

g2_new_tuning_specs = {
    "g2-t1-c1-smallcnn": ("g2_t1_c1_smallcnn.json", "g2_t1_c1_smallcnn"),
    "g2-t2-c1-smallcnn": ("g2_t2_c1_smallcnn.json", "g2_t2_c1_smallcnn"),
    "g2-t1-c2-resnet18": ("g2_t1_c2_resnet18.json", "g2_t1_c2_resnet18"),
    "g2-t2-c2-resnet18": ("g2_t2_c2_resnet18.json", "g2_t2_c2_resnet18"),
}
g2_tuning_attempt_frames = []
g2_registry = RunRegistry(RUNS_CSV)
for experiment_id, (config_name, evidence_slug) in g2_new_tuning_specs.items():
    outputs = run_or_load_experiment(
        TASK2_CONFIG_DIR / config_name,
        mode="run_or_load",
    )
    experiment_manifest = build_experiment_evidence(
        outputs,
        registry_path=RUNS_CSV,
        expected_ids=season_development["id"],
        protected_ids=protected_ids,
        probability_note=g2_probability_note,
        calibration_claim_allowed=False,
        evidence_directory=TASK2_EVIDENCE_DIR / evidence_slug,
        figure_directory=TASK2_FIGURE_DIR,
    )
    assert experiment_manifest["experiment_id"] == experiment_id
    assert experiment_manifest["coverage"]["row_count"] == len(season_development)
    assert experiment_manifest["coverage"]["protected_id_count"] == 0
    assert len(experiment_manifest["run_ids"]) == 5

    attempts = g2_registry.find(experiment_id=experiment_id)
    completed = attempts.loc[attempts["status"].eq("completed")]
    assert len(completed) == 5
    assert set(completed["fold"].astype(int)) == set(range(5))
    assert not attempts["status"].eq("running").any()
    g2_tuning_attempt_frames.append(attempts)

g2_all_tuning_specs = {
    "g1-c1-smallcnn": ("g1_c1_smallcnn.json", "g1_c1_smallcnn"),
    "g2-t1-c1-smallcnn": ("g2_t1_c1_smallcnn.json", "g2_t1_c1_smallcnn"),
    "g2-t2-c1-smallcnn": ("g2_t2_c1_smallcnn.json", "g2_t2_c1_smallcnn"),
    "g1-c2-resnet18": ("g1_c2_resnet18.json", "g1_c2_resnet18"),
    "g2-t1-c2-resnet18": ("g2_t1_c2_resnet18.json", "g2_t1_c2_resnet18"),
    "g2-t2-c2-resnet18": ("g2_t2_c2_resnet18.json", "g2_t2_c2_resnet18"),
}
g2_tuning_manifest = build_g2_tuning_evidence(
    experiment_manifest_paths=[
        TASK2_EVIDENCE_DIR / evidence_slug / "manifest.json"
        for _, evidence_slug in g2_all_tuning_specs.values()
    ],
    experiment_config_paths=[
        TASK2_CONFIG_DIR / config_name
        for config_name, _ in g2_all_tuning_specs.values()
    ],
    augmentation_decision_path=(
        TASK2_EVIDENCE_DIR / "g2_augmentation_ablation/decision.json"
    ),
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "g2_compact_tuning",
    figure_directory=TASK2_FIGURE_DIR,
)
selection_story_manifest = build_task2_selection_story_evidence(
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "selection_story",
)

g2_tuning_leaderboard = pd.read_csv(
    ROOT / g2_tuning_manifest["artifacts"]["leaderboard"]["path"]
)
g2_tuning_paired_folds = pd.read_csv(
    ROOT / g2_tuning_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
with (ROOT / g2_tuning_manifest["artifacts"]["decision"]["path"]).open(
    encoding="utf-8"
) as handle:
    g2_tuning_decision = json.load(handle)
incremental_model_selection = pd.read_csv(
    ROOT
    / selection_story_manifest["artifacts"]["incremental_model_selection"]["path"]
)
eda_reflection = pd.read_csv(
    ROOT / selection_story_manifest["artifacts"]["eda_reflection"]["path"]
)

assert g2_tuning_decision["decision_status"] == "closed"
assert g2_tuning_decision["retained_transform"] == "P0/A0"
assert g2_tuning_decision["families"]["C1"]["selected_tuning_id"] == "T1"
assert g2_tuning_decision["families"]["C2"]["selected_tuning_id"] == "T0"
assert len(g2_tuning_leaderboard) == 6
assert len(g2_tuning_paired_folds) == 20
assert selection_story_manifest["selected_finalist_experiment_ids"] == [
    "g2-t1-c1-smallcnn",
    "g1-c2-resnet18",
]
assert selection_story_manifest["artifacts"]["incremental_model_selection"][
    "path"
].endswith("incremental_model_selection.csv")
assert selection_story_manifest["artifacts"]["eda_reflection"]["path"].endswith(
    "eda_reflection.csv"
)

g2_tuning_attempts = pd.concat(g2_tuning_attempt_frames, ignore_index=True)
assert not g2_tuning_attempts["status"].eq("running").any()
display(g2_tuning_leaderboard.loc[:, [
        "family", "tuning_id", "learning_rate", "weight_decay",
        "pooled_macro_f1", "fold_sd_macro_f1", "spring_f1",
        "delta_vs_t0_macro_f1", "selected",
    ]].style.format({
        "learning_rate": "{:.4g}",
        "weight_decay": "{:.4g}",
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "delta_vs_t0_macro_f1": "{:+.6f}",
    }))


# Additional single-output views rendered below:
# display(g2_tuning_paired_folds)
# display(incremental_model_selection.style.format({"pooled_macro_f1": "{:.6f}"}))
# display(eda_reflection)
# display(
#     pd.DataFrame(
#         [
#             {"artifact": name, **declaration}
#             for name, declaration in selection_story_manifest["artifacts"].items()
#         ]
#     )
# )
# display(
#     g2_tuning_attempts.loc[:, [
#         "experiment_id",
#         "fold",
#         "run_id",
#         "status",
#         "git_commit",
#         "git_dirty",
#         "primary_metric_value",
#     ]].sort_values(["experiment_id", "fold", "status"])
# )
# display(
#     Image(
#         filename=str(
#             ROOT / g2_tuning_manifest["artifacts"]["c1_learning_curves"]["path"]
#         )
#     )
# )
# display(
#     Image(
#         filename=str(
#             ROOT / g2_tuning_manifest["artifacts"]["c2_learning_curves"]["path"]
#         )
#     )
# )


> **Interpretation:** **Incremental question:** after B0 exposed imbalance, B1 proved fixed shape/colour signal, and G1 showed learned features were better, G2-T asks only whether two frozen optimiser changes materially improve the shortlisted C1 and C2 families at retained P0/A0. **Result:** C1 scored T0 `0.699902`, T1 `0.708075` (`+0.008173`), and T2 `0.700275` (`+0.000373`); T1 passed the frozen `+0.003` gain rule. C2 scored T0 `0.707099`, T1 `0.700537` (`-0.006562`), and T2 `0.708246` (`+0.001146`); T2 was the best observed C2 score but did not pass the gain rule. **Learning curves:** the left panels show five-fold mean plus SD for train and validation loss; both decline, so optimisation is working. The right panels show validation accuracy and macro-F1; their persistent gap confirms that accuracy still hides minority-class errors. Training accuracy is not shown because it was not logged, and no value was invented. **EDA reflection:** the imbalance warning and the presence of shape/colour signal were supported. The ideas that a larger P1 image or extra A1 colour jitter would improve validation were contradicted. ArticleType, file size, and acquisition year shortcut risks are still untested and remain required OOF slices. **Decision:** select C1-T1 (`lr=1e-3`, `weight_decay=1e-4`) and retain C2-T0 (`lr=3e-4`, `weight_decay=1e-4`); do not expand the grid. Fully train those two at the same 30-epoch/patience-5 budget next. **Limitation:** this gate uses one seed and eight epochs, so it selects settings rather than the final model. **Trace:** `results/evidence/task2/g2_compact_tuning/manifest.json` hashes all six configs, all six experiment manifests, paired folds, per-class results, histories, decisions, and both charts. Selected C1-T1 includes `g2-t1-c1-smallcnn-f0-s2753-b5a391a13f44`; the rejected C2-T2 evidence includes `g2-t2-c2-resnet18-f2-s2753-c0de62f1594e`; each source manifest records all five completed run IDs.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.2.3a Tuning paired-fold table

In [ ]:
from IPython.display import display

display(g2_tuning_paired_folds)

> **Interpretation — 8.2.3a Tuning paired-fold table**
>
> **Why this output exists.** This output isolates 8.2.3a tuning paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3b Incremental selection story

In [ ]:
from IPython.display import display

display(incremental_model_selection.style.format({"pooled_macro_f1": "{:.6f}"}))

> **Interpretation — 8.2.3b Incremental selection story**
>
> **Why this output exists.** This output isolates 8.2.3b incremental selection story so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3c EDA reflection table

In [ ]:
from IPython.display import display

display(eda_reflection)

> **Interpretation — 8.2.3c EDA reflection table**
>
> **Why this output exists.** This output isolates 8.2.3c eda reflection table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `hypothesis` is the earlier EDA claim; status marks supported, contradicted, or untested; measured evidence and next action connect EDA to model selection.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3d Selection-story artifact trace

In [ ]:
import pandas as pd

from IPython.display import display

display(
    pd.DataFrame(
        [
            {"artifact": name, **declaration}
            for name, declaration in selection_story_manifest["artifacts"].items()
        ]
    )
)

> **Interpretation — 8.2.3d Selection-story artifact trace**
>
> **Why this output exists.** This output isolates 8.2.3d selection-story artifact trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `artifact` names the evidence object; `path` locates it inside the project; `sha256` detects any byte change.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3e Tuning run trace

In [ ]:
from IPython.display import display

display(
    g2_tuning_attempts.loc[:, [
        "experiment_id",
        "fold",
        "run_id",
        "status",
        "git_commit",
        "git_dirty",
        "primary_metric_value",
    ]].sort_values(["experiment_id", "fold", "status"])
)

> **Interpretation — 8.2.3e Tuning run trace**
>
> **Why this output exists.** This output isolates 8.2.3e tuning run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3f C1 tuning learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / g2_tuning_manifest["artifacts"]["c1_learning_curves"]["path"]
        )
    )
)

> **Interpretation — 8.2.3f C1 tuning learning curve**
>
> **Why this output exists.** This output isolates 8.2.3f c1 tuning learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.2.3g C2 tuning learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / g2_tuning_manifest["artifacts"]["c2_learning_curves"]["path"]
        )
    )
)

> **Interpretation — 8.2.3g C2 tuning learning curve**
>
> **Why this output exists.** This output isolates 8.2.3g c2 tuning learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** T1 improves C1 by +0.008173 and is selected; no C2 tuning option clears the +0.003 replacement rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 8.3 G4 problem-driven improvements and benchmark

Class balance, multi-task learning, and pretrained comparison test different hypotheses and are separated below.

#### 8.3.1 I1 class-balanced loss

I1 directly tests whether fold-fitted reweighting improves the Spring minority.

In [ ]:
from fashion.config import RUNS_CSV, TASK2_CONFIG_DIR
from fashion.task2.class_balance import run_or_load_i1_experiment
from fashion.task2.evidence import (
    build_experiment_evidence,
    build_i1_class_balance_evidence,
)
from fashion.train.registry import RunRegistry

i1_config_path = TASK2_CONFIG_DIR / "g4_i1_effective_number_c1.json"
i1_outputs = run_or_load_i1_experiment(
    i1_config_path,
    mode="run_or_load",
)
i1_probability_note = (
    "Uncalibrated softmax probabilities from the best validation macro-F1 "
    "checkpoint in each fold; calibration metrics are diagnostic only."
)
i1_experiment_manifest = build_experiment_evidence(
    i1_outputs,
    registry_path=RUNS_CSV,
    expected_ids=season_development["id"],
    protected_ids=protected_ids,
    probability_note=i1_probability_note,
    calibration_claim_allowed=False,
    evidence_directory=(
        TASK2_EVIDENCE_DIR / "g4_i1_effective_number_c1"
    ),
    figure_directory=TASK2_FIGURE_DIR,
)
i1_manifest = build_i1_class_balance_evidence(
    i1_manifest_path=(
        TASK2_EVIDENCE_DIR / "g4_i1_effective_number_c1/manifest.json"
    ),
    i1_config_path=i1_config_path,
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "i1_class_balance",
    figure_directory=TASK2_FIGURE_DIR,
)

current_run_ids = {output.run_id for output in i1_outputs}
assert len(current_run_ids) == 5
assert set(i1_experiment_manifest["run_ids"]) == current_run_ids
assert i1_experiment_manifest["coverage"]["row_count"] == len(
    season_development
)
assert i1_experiment_manifest["coverage"]["protected_id_count"] == 0

i1_registry = RunRegistry(RUNS_CSV)
i1_attempts = i1_registry.find(
    experiment_id="g4-i1-effective-number-c1"
)
i1_current_attempts = i1_attempts.loc[
    i1_attempts["run_id"].isin(current_run_ids)
].copy()
assert len(i1_current_attempts) == 5
assert i1_current_attempts["status"].eq("completed").all()
assert set(i1_current_attempts["fold"].astype(int)) == set(range(5))
assert not i1_attempts["status"].eq("running").any()

i1_comparison = pd.read_csv(
    ROOT / i1_manifest["artifacts"]["comparison"]["path"]
)
i1_paired_folds = pd.read_csv(
    ROOT / i1_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
i1_per_class = pd.read_csv(
    ROOT / i1_manifest["artifacts"]["per_class_comparison"]["path"]
)
i1_class_weights = pd.read_csv(
    ROOT / i1_manifest["artifacts"]["class_weights_by_fold"]["path"]
)
with (ROOT / i1_manifest["artifacts"]["decision"]["path"]).open(
    encoding="utf-8"
) as handle:
    i1_decision = json.load(handle)

assert i1_manifest["gate"] == "G4-I1"
assert i1_manifest["keep_i1"] is False
assert i1_manifest["selected_experiment_id"] == "g3-c1-t1-smallcnn"
assert i1_decision["loss_values_comparable_to_reference"] is False
assert len(i1_paired_folds) == 5
assert len(i1_per_class) == len(SEASON_LABELS)
assert len(i1_class_weights) == 5 * len(SEASON_LABELS)

i1_weight_summary = (
    i1_class_weights.groupby("label", as_index=False, sort=False)
    .agg(
        training_count_min=("class_count", "min"),
        training_count_max=("class_count", "max"),
        class_weight_min=("class_weight", "min"),
        class_weight_max=("class_weight", "max"),
    )
)
display(i1_comparison.loc[:, [
        "variant", "loss_id", "pooled_macro_f1", "fold_sd_macro_f1",
        "spring_f1", "delta_macro_f1_vs_reference",
        "delta_spring_f1_vs_reference", "selected_by_i1_gate",
    ]].style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "delta_macro_f1_vs_reference": "{:+.6f}",
        "delta_spring_f1_vs_reference": "{:+.6f}",
    }))


# Additional single-output views rendered below:
# display(i1_paired_folds)
# display(
#     i1_per_class.style.format(
#         {
#             "reference_precision": "{:.6f}",
#             "reference_recall": "{:.6f}",
#             "reference_f1": "{:.6f}",
#             "i1_precision": "{:.6f}",
#             "i1_recall": "{:.6f}",
#             "i1_f1": "{:.6f}",
#             "delta_i1_minus_reference_precision": "{:+.6f}",
#             "delta_i1_minus_reference_recall": "{:+.6f}",
#             "delta_i1_minus_reference_f1": "{:+.6f}",
#         }
#     )
# )
# display(
#     i1_weight_summary.style.format(
#         {
#             "class_weight_min": "{:.6f}",
#             "class_weight_max": "{:.6f}",
#         }
#     )
# )
# display(pd.json_normalize(i1_decision, sep=".").transpose())
# display(
#     i1_attempts.loc[:, [
#         "fold",
#         "run_id",
#         "status",
#         "error_type",
#         "git_commit",
#         "git_dirty",
#         "primary_metric_value",
#     ]].sort_values(["fold", "status", "run_id"])
# )
# display(
#     Image(
#         filename=str(
#             ROOT / i1_manifest["artifacts"]["learning_curves"]["path"]
#         )
#     )
# )
# display(
#     Image(
#         filename=str(
#             ROOT / i1_manifest["artifacts"]["per_class_f1_delta"]["path"]
#         )
#     )
# )


> **Interpretation:** **Incremental question:** B0 proved that class imbalance makes accuracy unsafe, B1 established usable shape-and-colour signal, and corrected G3 made C1-T1 the compact reference. I1 therefore changes only the loss: each training fold gives rare Spring about `2.511-2.513` weight and common Summer about `0.348` weight; the split, images, backbone, optimiser, seed, and data-preparation files stay fixed. **Result:** I1 reached `0.701471` pooled OOF macro-F1 versus G3-C1 `0.737661` (`-0.036191`). Spring F1 fell from `0.744975` to `0.702439` (`-0.042536`), and Fall was the worst other class at `-0.060738`. All three frozen criteria failed. **Why:** Spring recall did rise by `+0.022573`, but Spring precision fell by `-0.152558`; the model predicted Spring more often but created too many false positives, so minority F1 became worse. Fold SD also rose from `0.010013` to `0.034919`, with fold 4 only `0.644306`, so the intervention was less stable. **Learning curves:** fold-weighted training and validation loss generally decline, while validation accuracy and macro-F1 rise with wide fold bands. The five-fold mean stops at the common 11-epoch horizon so every point contains all folds. I1 and G3 loss values are not comparable because their loss functions use different scales. **EDA reflection:** the earlier imbalance finding was accurate, but the narrower hypothesis that this effective-number weighting would solve it was contradicted. Imbalance justifies macro-F1 and class-level analysis; it does not guarantee that reweighting helps. **Decision:** reject I1 and retain `g3-c1-t1-smallcnn`; test the separate I2 auxiliary-supervision question next. **Limitation:** this is one seed and one frozen `beta=0.9999` rule, so it does not prove that every class-balancing method fails; calibration is still untested. **Trace:** the five selected runs span `g4-i1-effective-number-c1-f0-s2753-9288633e212b` to `g4-i1-effective-number-c1-f4-s2753-a70502c769db`. The externally stopped fold-1 attempt remains `interrupted` in the local ledger but is excluded from evidence. `results/evidence/task2/i1_class_balance/manifest.json` hashes the OOF comparison, fold pairs, class weights, run IDs, histories, decision, learning curves, and per-class chart.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.3.1a I1 paired-fold table

In [ ]:
from IPython.display import display

display(i1_paired_folds)

> **Interpretation — 8.3.1a I1 paired-fold table**
>
> **Why this output exists.** This output isolates 8.3.1a i1 paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.1b I1 per-class comparison

In [ ]:
from IPython.display import display

display(
    i1_per_class.style.format(
        {
            "reference_precision": "{:.6f}",
            "reference_recall": "{:.6f}",
            "reference_f1": "{:.6f}",
            "i1_precision": "{:.6f}",
            "i1_recall": "{:.6f}",
            "i1_f1": "{:.6f}",
            "delta_i1_minus_reference_precision": "{:+.6f}",
            "delta_i1_minus_reference_recall": "{:+.6f}",
            "delta_i1_minus_reference_f1": "{:+.6f}",
        }
    )
)

> **Interpretation — 8.3.1b I1 per-class comparison**
>
> **Why this output exists.** This output isolates 8.3.1b i1 per-class comparison so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.1c I1 weight summary

In [ ]:
from IPython.display import display

display(
    i1_weight_summary.style.format(
        {
            "class_weight_min": "{:.6f}",
            "class_weight_max": "{:.6f}",
        }
    )
)

> **Interpretation — 8.3.1c I1 weight summary**
>
> **Why this output exists.** This output isolates 8.3.1c i1 weight summary so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Class/fold identifies the training-only estimate; min/max show the fitted weight range. Larger values give that class more loss contribution.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.1d I1 decision audit

In [ ]:
import pandas as pd

from IPython.display import display

i1_decision_view = pd.DataFrame([{
    "gate": i1_decision["gate"],
    "overall_delta": i1_decision["observed_i1_minus_reference_macro_f1"],
    "spring_f1_delta": i1_decision["observed_i1_minus_reference_spring_f1"],
    "worst_other_class": i1_decision["observed_worst_other_class"],
    "worst_other_f1_delta": i1_decision["observed_worst_other_class_f1_delta"],
    "keep_i1": i1_decision["keep_i1"],
    "selected_experiment_id": i1_decision["selected_experiment_id"],
}])
display(i1_decision_view.style.format({
    "overall_delta": "{:+.6f}",
    "spring_f1_delta": "{:+.6f}",
    "worst_other_f1_delta": "{:+.6f}",
}))

> **Interpretation — 8.3.1d I1 decision audit**
>
> **Why this output exists.** This output isolates 8.3.1d i1 decision audit so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `gate` names the frozen rule; delta columns are candidate minus reference; boolean fields show whether thresholds passed, selection changed, or a winner was frozen.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.1e I1 run trace

In [ ]:
from IPython.display import display

display(
    i1_attempts.loc[:, [
        "fold",
        "run_id",
        "status",
        "error_type",
        "git_commit",
        "git_dirty",
        "primary_metric_value",
    ]].sort_values(["fold", "status", "run_id"])
)

> **Interpretation — 8.3.1e I1 run trace**
>
> **Why this output exists.** This output isolates 8.3.1e i1 run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.1f I1 learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / i1_manifest["artifacts"]["learning_curves"]["path"]
        )
    )
)

> **Interpretation — 8.3.1f I1 learning curve**
>
> **Why this output exists.** This output isolates 8.3.1f i1 learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.1g I1 per-class delta figure

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / i1_manifest["artifacts"]["per_class_f1_delta"]["path"]
        )
    )
)

> **Interpretation — 8.3.1g I1 per-class delta figure**
>
> **Why this output exists.** This output isolates 8.3.1g i1 per-class delta figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The axis labels name the compared metric and candidate; positive candidate-minus-reference values favour the candidate, while zero means no observed change.
>
> **What it shows.** I1 lowers overall macro-F1 by 0.036191 and Spring F1 by 0.042536, so class weighting is rejected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2 I2 image-only multi-task training

I2 tests whether auxiliary ArticleType supervision improves shared visual features or causes negative transfer.

In [ ]:
from fashion.config import RUNS_CSV, TASK2_CONFIG_DIR
from fashion.task2.evidence import build_experiment_evidence
from fashion.task2.multitask import load_i2_config, run_i2_matrix
from fashion.task2.multitask_evidence import build_i2_transfer_evidence
from fashion.train.registry import RunRegistry

i2_config_paths = (
    TASK2_CONFIG_DIR / "g4_i2_article_type_lambda_0_1_c1.json",
    TASK2_CONFIG_DIR / "g4_i2_article_type_lambda_0_3_c1.json",
)
i2_configs = [load_i2_config(path) for path in i2_config_paths]
i2_outputs = run_i2_matrix(i2_configs, mode="run_or_load")
i2_probability_note = (
    "Uncalibrated softmax probabilities from the best validation Season "
    "macro-F1 checkpoint; ArticleType is training-only."
)
i2_experiment_manifests = {}
for config in i2_configs:
    config_outputs = [
        output
        for output in i2_outputs
        if output.experiment_id == config.experiment_id
    ]
    evidence_slug = config.experiment_id.replace("-", "_")
    i2_experiment_manifests[config.experiment_id] = (
        build_experiment_evidence(
            config_outputs,
            registry_path=RUNS_CSV,
            expected_ids=season_development["id"],
            protected_ids=protected_ids,
            probability_note=i2_probability_note,
            calibration_claim_allowed=False,
            evidence_directory=TASK2_EVIDENCE_DIR / evidence_slug,
            figure_directory=TASK2_FIGURE_DIR,
        )
    )

i2_manifest_paths = tuple(
    TASK2_EVIDENCE_DIR
    / config.experiment_id.replace("-", "_")
    / "manifest.json"
    for config in i2_configs
)
i2_manifest = build_i2_transfer_evidence(
    i2_manifest_paths=i2_manifest_paths,
    i2_config_paths=i2_config_paths,
    project_root=ROOT,
    evidence_directory=TASK2_EVIDENCE_DIR / "i2_multitask",
    figure_directory=TASK2_FIGURE_DIR,
)

current_run_ids = {output.run_id for output in i2_outputs}
assert len(current_run_ids) == 10
assert all(
    manifest["coverage"]["row_count"] == len(season_development)
    for manifest in i2_experiment_manifests.values()
)
assert all(
    manifest["coverage"]["protected_id_count"] == 0
    for manifest in i2_experiment_manifests.values()
)

i2_registry = RunRegistry(RUNS_CSV)
i2_attempts = pd.concat(
    [
        i2_registry.find(experiment_id=config.experiment_id)
        for config in i2_configs
    ],
    ignore_index=True,
)
i2_current_attempts = i2_attempts.loc[
    i2_attempts["run_id"].isin(current_run_ids)
].copy()
assert len(i2_current_attempts) == 10
assert i2_current_attempts["status"].eq("completed").all()
assert not i2_attempts["status"].eq("running").any()
assert set(i2_current_attempts["fold"].astype(int)) == set(range(5))
assert set(i2_current_attempts["seed"].astype(int)) == {2753}
assert (
    i2_current_attempts["git_dirty"].astype(str).str.lower().eq("false").all()
)

i2_comparison = pd.read_csv(
    ROOT / i2_manifest["artifacts"]["comparison"]["path"]
)
i2_paired_folds = pd.read_csv(
    ROOT / i2_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
i2_per_class = pd.read_csv(
    ROOT / i2_manifest["artifacts"]["per_class_comparison"]["path"]
)
i2_slices = pd.read_csv(
    ROOT / i2_manifest["artifacts"]["slice_metrics"]["path"]
)
with (ROOT / i2_manifest["artifacts"]["decision"]["path"]).open(
    encoding="utf-8"
) as handle:
    i2_decision = json.load(handle)

assert i2_manifest["gate"] == "G4-I2"
assert i2_manifest["keep_i2"] is True
assert (
    i2_manifest["selected_experiment_id"]
    == "g4-i2-article-type-lambda-0-3-c1"
)
assert i2_decision["loss_values_comparable_to_reference"] is False
assert len(i2_paired_folds) == 5
assert set(i2_per_class["label"]) == set(SEASON_LABELS)
assert {"aligned", "conflict"} <= set(i2_slices["shortcut_slice"])

display(i2_comparison.loc[:, [
        "variant", "auxiliary_weight", "pooled_macro_f1", "fold_sd_macro_f1",
        "spring_f1", "delta_macro_f1_vs_reference",
        "delta_spring_f1_vs_reference", "delta_conflict_macro_f1_vs_reference",
        "passes_i2_gate", "selected_by_i2_gate",
    ]].style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "delta_macro_f1_vs_reference": "{:+.6f}",
        "delta_spring_f1_vs_reference": "{:+.6f}",
        "delta_conflict_macro_f1_vs_reference": "{:+.6f}",
    }))


# Additional single-output views rendered below:
# display(i2_paired_folds)
# display(i2_per_class.style.format({"f1": "{:.6f}", "delta_f1_vs_reference": "{:+.6f}"}))
# display(
#     i2_slices.loc[
#         i2_slices["shortcut_slice"].isin(["aligned", "conflict"])
#     ].style.format({"accuracy": "{:.6f}", "macro_f1": "{:.6f}"})
# )
# display(pd.json_normalize(i2_decision, sep=".").transpose())
# display(
#     i2_current_attempts.loc[:, [
#         "experiment_id",
#         "fold",
#         "run_id",
#         "status",
#         "git_commit",
#         "git_dirty",
#         "primary_metric_value",
#     ]].sort_values(["experiment_id", "fold"])
# )
# display(
#     Image(
#         filename=str(
#             ROOT / i2_manifest["artifacts"]["learning_curves"]["path"]
#         )
#     )
# )
# display(
#     Image(
#         filename=str(
#             ROOT / i2_manifest["artifacts"]["transfer_deltas"]["path"]
#         )
#     )
# )


> **Interpretation:** **Incremental question:** B0 showed why accuracy alone is unsafe, B1 proved that simple shape-and-colour features carry some Season signal, C1/G3 supplied the strongest compact scratch reference, and I1 showed that changing only class weights made the result worse. I2 therefore keeps the G3 data, split, SmallCNN, optimiser, transform, and seed fixed, then adds one training-only ArticleType head; inference still receives only the image, and Notebook 01/data preparation is unchanged. **Result:** lambda `0.1` reached `0.750758` pooled five-fold OOF macro-F1 (`+0.013097` versus G3-C1), while lambda `0.3` reached `0.752687` (`+0.015026`) versus `0.737661`. For lambda `0.3`, Spring F1 rose from `0.744975` to `0.764784` (`+0.019809`), aligned-slice macro-F1 rose by `+0.013792`, and conflict-slice macro-F1 rose by `+0.027598`. Lambda `0.1` produced the larger conflict gain (`+0.031917`) but the lower overall score. Both candidates pass the frozen I2 gate; its stated priority selects lambda `0.3` because it has the highest overall pooled macro-F1. **Learning curves:** the teacher-style chart places train/validation total loss on the left and validation accuracy/macro-F1 on the right, using five-fold mean plus/minus one SD. Validation scores rise then flatten while training loss keeps falling, so the best validation macro-F1 checkpoint remains necessary. The common five-fold horizons are 27 epochs for lambda `0.1` and 19 for lambda `0.3`; total-loss values must be interpreted within a lambda because changing the auxiliary weight changes the loss scale. **EDA reflection:** the earlier EDA insight that ArticleType and Season are related was useful: auxiliary labels improved Spring and overall performance. The fear that I2 would only copy the ArticleType shortcut was too simple, because conflict rows also improved. However, this is still association, not proof that ArticleType causes Season; aligned/conflict is only one declared slice. **Decision:** keep I2 lambda `0.3` as the current candidate, but do not freeze the ultimate winner yet. Run the matched pretrained benchmark boundary and second-seed stability test next. **Limitations:** this conclusion uses one seed, only 19 unseen-ArticleType rows, no missing-ArticleType rows, and uncalibrated probabilities; robustness and uncertainty are still pending. **Trace:** the selected runs span `g4-i2-article-type-lambda-0-3-c1-f0-s2753-902fcc852d5f` to `g4-i2-article-type-lambda-0-3-c1-f4-s2753-8d210d54f01e`. `results/evidence/task2/i2_multitask/manifest.json` links all ten I2 run IDs, both configs, pooled OOF tables, shortcut slices, decision, learning curves, and transfer chart by SHA-256.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.3.2a I2 paired-fold table

In [ ]:
from IPython.display import display

display(i2_paired_folds)

> **Interpretation — 8.3.2a I2 paired-fold table**
>
> **Why this output exists.** This output isolates 8.3.2a i2 paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2b I2 per-class comparison

In [ ]:
from IPython.display import display

display(i2_per_class.style.format({"f1": "{:.6f}", "delta_f1_vs_reference": "{:+.6f}"}))

> **Interpretation — 8.3.2b I2 per-class comparison**
>
> **Why this output exists.** This output isolates 8.3.2b i2 per-class comparison so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2c I2 shortcut-slice table

In [ ]:
from IPython.display import display

display(
    i2_slices.loc[
        i2_slices["shortcut_slice"].isin(["aligned", "conflict"])
    ].style.format({"accuracy": "{:.6f}", "macro_f1": "{:.6f}"})
)

> **Interpretation — 8.3.2c I2 shortcut-slice table**
>
> **Why this output exists.** This output isolates 8.3.2c i2 shortcut-slice table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2d I2 decision audit

In [ ]:
import pandas as pd

from IPython.display import display

i2_selected_rule = i2_decision["candidate_criteria"][
    i2_decision["selected_experiment_id"]
]
i2_decision_view = pd.DataFrame([{
    "gate": i2_decision["gate"],
    "selected_experiment_id": i2_decision["selected_experiment_id"],
    "auxiliary_weight": i2_selected_rule["auxiliary_weight"],
    "overall_delta": i2_selected_rule["observed_overall_delta"],
    "conflict_delta": i2_selected_rule["observed_conflict_delta"],
    "keep_i2": i2_decision["keep_i2"],
    "ultimate_winner_frozen": i2_decision["ultimate_winner_frozen"],
}])
display(i2_decision_view.style.format({
    "overall_delta": "{:+.6f}",
    "conflict_delta": "{:+.6f}",
}))

> **Interpretation — 8.3.2d I2 decision audit**
>
> **Why this output exists.** This output isolates 8.3.2d i2 decision audit so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `gate` names the frozen rule; delta columns are candidate minus reference; boolean fields show whether thresholds passed, selection changed, or a winner was frozen.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2e I2 run trace

In [ ]:
from IPython.display import display

display(
    i2_current_attempts.loc[:, [
        "experiment_id",
        "fold",
        "run_id",
        "status",
        "git_commit",
        "git_dirty",
        "primary_metric_value",
    ]].sort_values(["experiment_id", "fold"])
)

> **Interpretation — 8.3.2e I2 run trace**
>
> **Why this output exists.** This output isolates 8.3.2e i2 run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2f I2 learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / i2_manifest["artifacts"]["learning_curves"]["path"]
        )
    )
)

> **Interpretation — 8.3.2f I2 learning curve**
>
> **Why this output exists.** This output isolates 8.3.2f i2 learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.2g I2 transfer-delta figure

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT / i2_manifest["artifacts"]["transfer_deltas"]["path"]
        )
    )
)

> **Interpretation — 8.3.2g I2 transfer-delta figure**
>
> **Why this output exists.** This output isolates 8.3.2g i2 transfer-delta figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The axis labels name the compared metric and candidate; positive candidate-minus-reference values favour the candidate, while zero means no observed change.
>
> **What it shows.** I2 lambda 0.3 improves overall macro-F1 by +0.015026 and conflict-slice macro-F1 by +0.027598, so it passes the gate.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.3 P* pretrained benchmark execution

P* estimates the value of transferred representation while remaining ineligible for submission.

In [ ]:
from fashion.config import RUNS_CSV, TASK2_CONFIG_DIR
from fashion.task2.evidence import build_experiment_evidence
from fashion.task2.experiments import load_experiment_config
from fashion.task2.pretraining import run_pretraining_matrix
from fashion.task2.pretraining_evidence import (
    build_pretraining_benchmark_evidence,
)
from fashion.train.registry import RunRegistry

pretraining_config_paths = (
    TASK2_CONFIG_DIR / "g4_p0s_resnet18_standard_scratch.json",
    TASK2_CONFIG_DIR / "g4_pstar_resnet18_standard_pretrained.json",
)
pretraining_configs = [
    load_experiment_config(path) for path in pretraining_config_paths
]
pretraining_outputs = run_pretraining_matrix(
    pretraining_configs,
    mode="run_or_load",
)
pretraining_probability_note = (
    "Uncalibrated softmax probabilities from the checkpoint selected by "
    "validation Season macro-F1."
)
pretraining_experiment_manifests = {}
for config in pretraining_configs:
    config_outputs = [
        output
        for output in pretraining_outputs
        if output.experiment_id == config.experiment_id
    ]
    evidence_slug = config.experiment_id.replace("-", "_")
    pretraining_experiment_manifests[config.experiment_id] = (
        build_experiment_evidence(
            config_outputs,
            registry_path=RUNS_CSV,
            expected_ids=season_development["id"],
            protected_ids=protected_ids,
            probability_note=pretraining_probability_note,
            calibration_claim_allowed=False,
            evidence_directory=TASK2_EVIDENCE_DIR / evidence_slug,
            figure_directory=TASK2_FIGURE_DIR,
        )
    )

pretraining_manifest_paths = tuple(
    TASK2_EVIDENCE_DIR
    / config.experiment_id.replace("-", "_")
    / "manifest.json"
    for config in pretraining_configs
)
pretraining_manifest = build_pretraining_benchmark_evidence(
    experiment_manifest_paths=pretraining_manifest_paths,
    experiment_config_paths=pretraining_config_paths,
    project_root=ROOT,
    expected_row_count=len(season_development),
    evidence_directory=TASK2_EVIDENCE_DIR / "pretraining_benchmark",
    figure_directory=TASK2_FIGURE_DIR,
)

current_run_ids = {output.run_id for output in pretraining_outputs}
assert len(current_run_ids) == 10
assert all(
    manifest["coverage"]["row_count"] == len(season_development)
    for manifest in pretraining_experiment_manifests.values()
)
assert all(
    manifest["coverage"]["protected_id_count"] == 0
    for manifest in pretraining_experiment_manifests.values()
)

pretraining_registry = RunRegistry(RUNS_CSV)
pretraining_attempts = pd.concat(
    [
        pretraining_registry.find(experiment_id=config.experiment_id)
        for config in pretraining_configs
    ],
    ignore_index=True,
)
pretraining_current_attempts = pretraining_attempts.loc[
    pretraining_attempts["run_id"].isin(current_run_ids)
].copy()
assert len(pretraining_current_attempts) == 10
assert pretraining_current_attempts["status"].eq("completed").all()
assert not pretraining_attempts["status"].eq("running").any()
assert set(pretraining_current_attempts["fold"].astype(int)) == set(range(5))
assert set(pretraining_current_attempts["seed"].astype(int)) == {2753}
assert (
    pretraining_current_attempts["git_dirty"]
    .astype(str)
    .str.lower()
    .eq("false")
    .all()
)

pretraining_comparison = pd.read_csv(
    ROOT / pretraining_manifest["artifacts"]["comparison"]["path"]
)
pretraining_paired_folds = pd.read_csv(
    ROOT
    / pretraining_manifest["artifacts"]["paired_fold_metrics"]["path"]
)
pretraining_per_class = pd.read_csv(
    ROOT
    / pretraining_manifest["artifacts"]["per_class_comparison"]["path"]
)
with (
    ROOT / pretraining_manifest["artifacts"]["decision"]["path"]
).open(encoding="utf-8") as handle:
    pretraining_decision = json.load(handle)

assert pretraining_manifest["gate"] == "G4-PSTAR"
assert pretraining_manifest["candidate_selection_affected"] is False
assert pretraining_manifest["ultimate_winner_frozen"] is False
assert pretraining_decision["p0s_benchmark_only"] is True
assert pretraining_decision["pstar_benchmark_only"] is True
assert pretraining_decision["pstar_final_eligible"] is False
assert len(pretraining_paired_folds) == 5
assert set(pretraining_per_class["label"]) == set(SEASON_LABELS)

display(pretraining_comparison.loc[:, [
        "variant", "training_origin", "weights", "scratch", "benchmark_only",
        "final_eligible", "pooled_macro_f1", "fold_sd_macro_f1", "spring_f1",
        "median_best_epoch", "five_fold_runtime_minutes",
    ]].style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_f1": "{:.6f}",
        "five_fold_runtime_minutes": "{:.2f}",
    }))


# Additional single-output views rendered below:
# display(pretraining_paired_folds)
# display(
#     pretraining_per_class.style.format(
#         {
#             "f1_p0s": "{:.6f}",
#             "f1_pstar": "{:.6f}",
#             "pstar_minus_p0s_f1": "{:+.6f}",
#         }
#     )
# )
# display(pd.json_normalize(pretraining_decision, sep=".").transpose())
# display(
#     pretraining_current_attempts.loc[:, [
#         "experiment_id",
#         "fold",
#         "run_id",
#         "status",
#         "git_commit",
#         "git_dirty",
#         "primary_metric_value",
#     ]].sort_values(["experiment_id", "fold"])
# )
# display(
#     Image(
#         filename=str(
#             ROOT
#             / pretraining_manifest["artifacts"]["learning_curves"]["path"]
#         )
#     )
# )
# display(
#     Image(
#         filename=str(
#             ROOT
#             / pretraining_manifest["artifacts"]["effect_figure"]["path"]
#         )
#     )
# )


> **Interpretation:** **Incremental question:** B0 exposed class imbalance, B1 proved that hand-built shape and colour features contain real Season signal, and the scratch C1/C2 path then learned stronger image features. The matched P0S/P* pair now isolates one further question: with the same standard-stem ResNet18, folds, P0/A0 transform, optimiser, loss, budget, and seed, what changes when only the initial weights come from ImageNet? Both rows are benchmark-only and final-ineligible; Notebook 01/data preparation is unchanged. **Result:** P0S reached `0.731172` pooled five-fold OOF macro-F1 and P* reached `0.754196`, an observed `+0.023024` effect. Spring F1 improved by `+0.038829`, the largest class gain, and all five paired folds favoured P*. Median best epoch fell from `22` to `11`, while five-fold runtime fell from `32.47` to `22.69` minutes because early stopping ended the pretrained runs sooner; parameter count and peak VRAM stayed matched. **Learning curves:** the teacher-style chart shows train/validation loss on the left and validation accuracy/macro-F1 on the right, with five-fold mean plus/minus one SD. It uses common horizons of 20 epochs for P0S and 12 for P*, so late curves are not biased toward folds that simply ran longer. In both rows, training loss keeps falling after validation scores flatten, and validation loss eventually rises; this supports best-macro-F1 checkpointing and early stopping. P* reaches its plateau sooner and higher. **EDA reflection:** the earlier EDA view that colour and shape provide useful but incomplete Season evidence was accurate: learned representations beat B1, and P* improves every class. The expectation that the development set alone would let the standard-stem scratch model close the representation gap was too optimistic. This matched pair estimates an initialisation effect only inside the fixed `80x60` pipeline; it does not prove that P* is better than I2 because their architecture and loss differ. **Decision:** use P* only as a benchmark ceiling. It cannot enter the eligible winner table, so candidate selection is unchanged and I2 lambda `0.3` remains the current scratch candidate. The ultimate winner is still open until seed `2026` stability and later analysis finish. **Limitations:** this gate uses one seed, the project transform is not the standard 224x224 ImageNet recipe, and probabilities remain uncalibrated. `ResNet18_Weights.DEFAULT` resolved to `ResNet18_Weights.IMAGENET1K_V1`. **Trace:** runs span `g4-p0s-resnet18-standard-scratch-f0-s2753-7db32dfd4d00` through `g4-p0s-resnet18-standard-scratch-f4-s2753-3a6604ca32a6` and `g4-pstar-resnet18-standard-pretrained-f0-s2753-ebc35155ebef` through `g4-pstar-resnet18-standard-pretrained-f4-s2753-a15e79285ae2`. `results/evidence/task2/pretraining_benchmark/manifest.json` links both configs, all ten runs, OOF metrics, boundaries, learning curves, and effect chart by SHA-256.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.3.3a P0S/P* paired-fold table

In [ ]:
from IPython.display import display

display(pretraining_paired_folds)

> **Interpretation — 8.3.3a P0S/P* paired-fold table**
>
> **Why this output exists.** This output isolates 8.3.3a p0s/p* paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** ImageNet initialisation adds +0.023024 macro-F1 in the matched benchmark, but P* remains benchmark-only and final-ineligible.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.3b P0S/P* per-class comparison

In [ ]:
from IPython.display import display

display(
    pretraining_per_class.style.format(
        {
            "f1_p0s": "{:.6f}",
            "f1_pstar": "{:.6f}",
            "pstar_minus_p0s_f1": "{:+.6f}",
        }
    )
)

> **Interpretation — 8.3.3b P0S/P* per-class comparison**
>
> **Why this output exists.** This output isolates 8.3.3b p0s/p* per-class comparison so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.
>
> **What it shows.** ImageNet initialisation adds +0.023024 macro-F1 in the matched benchmark, but P* remains benchmark-only and final-ineligible.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.3c P0S/P* decision audit

In [ ]:
import pandas as pd

from IPython.display import display

pretraining_decision_view = pd.DataFrame([{
    "gate": pretraining_decision["gate"],
    "control": pretraining_decision["control_experiment_id"],
    "benchmark": pretraining_decision["benchmark_experiment_id"],
    "macro_f1_delta": pretraining_decision["observed_pstar_minus_p0s_macro_f1"],
    "spring_f1_delta": pretraining_decision["observed_pstar_minus_p0s_spring_f1"],
    "pstar_final_eligible": pretraining_decision["pstar_final_eligible"],
    "candidate_selection_affected": pretraining_decision[
        "candidate_selection_affected"
    ],
}])
display(pretraining_decision_view.style.format({
    "macro_f1_delta": "{:+.6f}",
    "spring_f1_delta": "{:+.6f}",
}))

> **Interpretation — 8.3.3c P0S/P* decision audit**
>
> **Why this output exists.** This output isolates 8.3.3c p0s/p* decision audit so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `gate` names the frozen rule; delta columns are candidate minus reference; boolean fields show whether thresholds passed, selection changed, or a winner was frozen.
>
> **What it shows.** ImageNet initialisation adds +0.023024 macro-F1 in the matched benchmark, but P* remains benchmark-only and final-ineligible.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.3d P0S/P* run trace

In [ ]:
from IPython.display import display

display(
    pretraining_current_attempts.loc[:, [
        "experiment_id",
        "fold",
        "run_id",
        "status",
        "git_commit",
        "git_dirty",
        "primary_metric_value",
    ]].sort_values(["experiment_id", "fold"])
)

> **Interpretation — 8.3.3d P0S/P* run trace**
>
> **Why this output exists.** This output isolates 8.3.3d p0s/p* run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** ImageNet initialisation adds +0.023024 macro-F1 in the matched benchmark, but P* remains benchmark-only and final-ineligible.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.3e P0S/P* learning curve

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT
            / pretraining_manifest["artifacts"]["learning_curves"]["path"]
        )
    )
)

> **Interpretation — 8.3.3e P0S/P* learning curve**
>
> **Why this output exists.** This output isolates 8.3.3e p0s/p* learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** ImageNet initialisation adds +0.023024 macro-F1 in the matched benchmark, but P* remains benchmark-only and final-ineligible.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.3.3f P0S/P* effect figure

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename=str(
            ROOT
            / pretraining_manifest["artifacts"]["effect_figure"]["path"]
        )
    )
)

> **Interpretation — 8.3.3f P0S/P* effect figure**
>
> **Why this output exists.** This output isolates 8.3.3f p0s/p* effect figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The axis labels name the compared metric and candidate; positive candidate-minus-reference values favour the candidate, while zero means no observed change.
>
> **What it shows.** ImageNet initialisation adds +0.023024 macro-F1 in the matched benchmark, but P* remains benchmark-only and final-ineligible.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 8.4 G5 second-seed stability

A second seed checks whether the final ordering depends on random initialisation.

In [ ]:
from IPython.display import Image

from fashion.train.artifacts import verify_artifact

stability_root = TASK2_EVIDENCE_DIR / "seed_stability"
stability_manifest_path = stability_root / "manifest.json"
with stability_manifest_path.open(encoding="utf-8") as handle:
    stability_manifest = json.load(handle)

for artifact_group in ("artifacts", "input_configs", "input_manifests"):
    for artifact in stability_manifest[artifact_group].values():
        verify_artifact(ROOT / artifact["path"], artifact["sha256"])

input_manifests = {}
for experiment_id, artifact in stability_manifest["input_manifests"].items():
    with (ROOT / artifact["path"]).open(encoding="utf-8") as handle:
        input_manifests[experiment_id] = json.load(handle)
assert all(
    manifest["coverage"]["row_count"] == len(season_development)
    for manifest in input_manifests.values()
)
assert all(
    manifest["coverage"]["protected_id_count"] == 0
    for manifest in input_manifests.values()
)

artifact_paths = {
    name: ROOT / artifact["path"]
    for name, artifact in stability_manifest["artifacts"].items()
}
seed_stability = pd.read_csv(artifact_paths["seed_stability"])
seed_drift = pd.read_csv(artifact_paths["seed_drift"])
paired_folds = pd.read_csv(artifact_paths["paired_fold_metrics"])
stability_registry = pd.read_csv(artifact_paths["registry_snapshot"])
with artifact_paths["decision"].open(encoding="utf-8") as handle:
    stability_decision = json.load(handle)

assert stability_manifest["gate"] == "G5-SEED"
assert stability_manifest["ordering_stable"] is True
assert stability_manifest["ultimate_winner_frozen"] is False
assert stability_decision["current_candidate"] == "I2"
assert stability_decision["candidate_selection_affected"] is False
assert len(seed_stability) == 4
assert set(seed_stability["candidate"]) == {"C2", "I2"}
assert set(seed_stability["seed"].astype(int)) == {2753, 2026}
assert len(paired_folds) == 5
assert len(stability_registry) == 20
assert stability_registry["status"].eq("completed").all()
assert stability_registry["git_dirty"].astype(str).str.lower().eq("false").all()
assert set(stability_registry["fold"].astype(int)) == set(range(5))
for seed in (2753, 2026):
    scores = (
        seed_stability.loc[seed_stability["seed"].eq(seed)]
        .set_index("candidate")["pooled_macro_f1"]
    )
    assert scores["I2"] > scores["C2"]

display(seed_stability.loc[:, [
        "candidate", "seed", "pooled_macro_f1", "fold_sd_macro_f1",
        "spring_recall", "spring_f1", "five_fold_runtime_minutes",
        "parameter_count",
    ]].sort_values(["seed", "candidate"]).style.format({
        "pooled_macro_f1": "{:.6f}",
        "fold_sd_macro_f1": "{:.6f}",
        "spring_recall": "{:.6f}",
        "spring_f1": "{:.6f}",
        "five_fold_runtime_minutes": "{:.2f}",
    }))
drift_formats = {
    column: "{:+.6f}"
    for column in seed_drift.columns
    if "minus" in column
}


# Additional single-output views rendered below:
# display(seed_drift.style.format(drift_formats))
# display(paired_folds)
# display(pd.json_normalize(stability_decision, sep=".").transpose())
# display(
#     stability_registry.loc[:, [
#         "candidate",
#         "seed",
#         "fold",
#         "run_id",
#         "git_commit",
#         "implementation_sha256",
#     ]].sort_values(["candidate", "seed", "fold"])
# )
# display(Image(filename=str(artifact_paths["learning_curves"])))
# display(Image(filename=str(artifact_paths["comparison_figure"])))


> **Interpretation — result:** The pre-registered G5 rule is satisfied: I2 remains above the retained C2 comparator at both complete five-fold seeds. At seed `2753`, pooled OOF macro-F1 is `0.752687` for I2 and `0.735036` for C2 (`+0.017651`). At seed `2026`, it is `0.744743` versus `0.733137` (`+0.011607`). All four OOF packs contain exactly `32,753` development IDs, no protected ID, and five clean folds. The extra ten G5 runs are stored under commit `04ef69d`; the table above exposes every run ID and implementation hash.
>
> **Seed drift and honest limitation:** C2 changes by `-0.001899` macro-F1 across seeds, while I2 changes by `-0.007944`; therefore I2 keeps the lead but is more seed-sensitive. I2 beats C2 on three of five paired folds at seed `2026`, not all five. Its Spring F1 remains higher (`0.754291` versus `0.747865`), but the Spring-recall advantage narrows to only `+0.002257`. Two seeds support the direction of the decision; they do not prove that every random start will preserve it. Grouped bootstrap uncertainty is still pending.
>
> **Learning curves:** The teacher-style figure places train/validation loss on the left and validation accuracy/macro-F1 on the right. Solid lines are seed `2753`; dashed lines are seed `2026`; shaded bands show fold spread. Both models learn quickly and then plateau while training loss continues to fall, supporting best-macro-F1 checkpointing and early stopping. I2 is much smaller (`1,206,112` parameters versus `11,170,884`) and the seed-2026 five-fold run is faster (`26.44` versus `60.92` minutes), but cost is not yet the selection metric.
>
> **EDA reflection and incremental decision:** The EDA hypothesis that ArticleType carries useful Season-related visual structure was directionally accurate: the image-only auxiliary task improves pooled macro-F1 at both seeds. The stronger assumption that this gain would be uniform was inaccurate: two seed-2026 folds reverse, and the minority-class recall gain nearly disappears. This remains an association, not proof that ArticleType causes better Season representations, and aligned/conflict shortcut slices must still be tested. Keep I2 lambda `0.3` as the current eligible scratch candidate, add no new architecture after G5, and do **not** freeze the ultimate winner yet. Next run shortcut/error, robustness/cost, calibration, grouped-bootstrap, and Grad-CAM analysis. Trace: `results/evidence/task2/seed_stability/manifest.json` links the four configs, twenty runs, tables, and both figures by SHA-256.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 8.4a Seed-drift table

In [ ]:
from IPython.display import display

display(seed_drift.style.format(drift_formats))

> **Interpretation — 8.4a Seed-drift table**
>
> **Why this output exists.** This output isolates 8.4a seed-drift table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `candidate` and `seed` identify the fitted pair; macro-F1 and Spring columns are OOF metrics; drift is seed 2026 minus seed 2753.
>
> **What it shows.** I2 stays above C2 at both seeds (+0.017651 and +0.011607), although its larger seed drift prevents an all-seeds claim.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.4b G5 paired-fold table

In [ ]:
from IPython.display import display

display(paired_folds)

> **Interpretation — 8.4b G5 paired-fold table**
>
> **Why this output exists.** This output isolates 8.4b g5 paired-fold table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` pairs models on identical validation rows; reference and candidate columns are fold macro-F1; `delta` is candidate minus reference, so its sign shows the winner.
>
> **What it shows.** I2 stays above C2 at both seeds (+0.017651 and +0.011607), although its larger seed drift prevents an all-seeds claim.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.4c G5 decision audit

In [ ]:
import pandas as pd

from IPython.display import display

stability_decision_view = pd.DataFrame([{
    "gate": stability_decision["gate"],
    "current_candidate": stability_decision["current_candidate"],
    "i2_minus_c2_seed_2753": stability_decision[
        "i2_minus_c2_macro_f1_seed_2753"
    ],
    "i2_minus_c2_seed_2026": stability_decision[
        "i2_minus_c2_macro_f1_seed_2026"
    ],
    "ordering_stable": stability_decision["ordering_stable"],
    "candidate_selection_affected": stability_decision[
        "candidate_selection_affected"
    ],
}])
display(stability_decision_view.style.format({
    "i2_minus_c2_seed_2753": "{:+.6f}",
    "i2_minus_c2_seed_2026": "{:+.6f}",
}))

> **Interpretation — 8.4c G5 decision audit**
>
> **Why this output exists.** This output isolates 8.4c g5 decision audit so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `gate` names the frozen rule; delta columns are candidate minus reference; boolean fields show whether thresholds passed, selection changed, or a winner was frozen.
>
> **What it shows.** I2 stays above C2 at both seeds (+0.017651 and +0.011607), although its larger seed drift prevents an all-seeds claim.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.4d G5 run trace

In [ ]:
from IPython.display import display

display(stability_registry.loc[:, [
        "candidate", "seed", "fold", "run_id", "git_commit",
    ]].assign(git_commit=lambda frame: frame["git_commit"].str[:7]).sort_values(
        ["candidate", "seed", "fold"]
    ))

> **Interpretation — 8.4d G5 run trace**
>
> **Why this output exists.** This output isolates 8.4d g5 run trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `run_id` identifies one physical run; `fold` is its canonical validation fold; `status` is the immutable lifecycle result; `primary_metric_value` is that fold's macro-F1; commit/dirty fields bind the code state.
>
> **What it shows.** I2 stays above C2 at both seeds (+0.017651 and +0.011607), although its larger seed drift prevents an all-seeds claim.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.4e G5 learning-curves figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(artifact_paths["learning_curves"])))

> **Interpretation — 8.4e G5 learning-curves figure**
>
> **Why this output exists.** This output isolates 8.4e g5 learning-curves figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** I2 stays above C2 at both seeds (+0.017651 and +0.011607), although its larger seed drift prevents an all-seeds claim.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 8.4f G5 candidate-comparison figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(artifact_paths["comparison_figure"])))

> **Interpretation — 8.4f G5 candidate-comparison figure**
>
> **Why this output exists.** This output isolates 8.4f g5 candidate-comparison figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The axis labels name the compared metric and candidate; positive candidate-minus-reference values favour the candidate, while zero means no observed change.
>
> **What it shows.** I2 stays above C2 at both seeds (+0.017651 and +0.011607), although its larger seed drift prevents an all-seeds claim.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 9. Cross-validated results

Turn registry-backed OOF artifacts into the main comparison evidence.

### 9.1 OOF leaderboard and fold stability

The leaderboard must be generated from artifacts, not typed by hand.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

from fashion.config import ROOT
from fashion.train.artifacts import verify_artifact


def load_verified_notebook_manifest(relative_path):
    manifest_path = ROOT / relative_path
    with manifest_path.open(encoding="utf-8") as handle:
        manifest = json.load(handle)
    verified_declarations = {}

    def walk_declarations(value, prefix=""):
        if isinstance(value, dict):
            if {"path", "sha256"} <= set(value):
                declared_path = ROOT / value["path"]
                verify_artifact(declared_path, value["sha256"])
                verified_declarations[prefix] = declared_path
            for name, child in value.items():
                child_prefix = f"{prefix}.{name}" if prefix else str(name)
                walk_declarations(child, child_prefix)
        elif isinstance(value, list):
            for index, child in enumerate(value):
                walk_declarations(child, f"{prefix}[{index}]")

    walk_declarations(manifest)
    artifact_paths = {
        name: verified_declarations[f"artifacts.{name}"]
        for name in manifest["artifacts"]
    }
    return manifest, artifact_paths


stability_manifest, stability_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/seed_stability/manifest.json"
)
assert stability_manifest["gate"] == "G5-SEED"
assert stability_manifest["ordering_stable"] is True
assert stability_manifest["ultimate_winner_frozen"] is False

finalist_leaderboard = pd.read_csv(stability_artifacts["seed_stability"])
stability_registry = pd.read_csv(
    stability_artifacts["registry_snapshot"], dtype=str, keep_default_na=False
)
assert len(finalist_leaderboard) == 4
assert set(finalist_leaderboard["candidate"]) == {"C2", "I2"}
assert set(finalist_leaderboard["seed"]) == {2753, 2026}
assert len(stability_registry) == stability_registry["run_id"].nunique() == 20
assert stability_registry["status"].eq("completed").all()
assert stability_registry["git_dirty"].str.lower().eq("false").all()
for _, rows in stability_registry.groupby(["candidate", "seed"]):
    assert set(rows["fold"].astype(int)) == set(range(5))

finalist_scorecard = finalist_leaderboard.loc[
    :,
    [
        "candidate",
        "seed",
        "pooled_macro_f1",
        "accuracy",
        "balanced_accuracy",
        "fold_sd_macro_f1",
        "spring_f1",
        "parameter_count",
        "five_fold_runtime_minutes",
        "median_best_epoch",
    ],
].sort_values(["seed", "pooled_macro_f1"], ascending=[True, False])
display(finalist_scorecard)


# Additional single-output views rendered below:
# display(Image(filename=str(stability_artifacts["comparison_figure"])))


> **Interpretation — measured evidence:** Every candidate/seed pack contains five completed, clean folds and exactly 32,753 OOF IDs; the 20 run IDs are in the verified registry snapshot. I2 leads C2 at both seeds: `0.752687` versus `0.735036` at seed `2753`, and `0.744743` versus `0.733137` at seed `2026`. The smaller second-seed margin and I2 drift of `-0.007944` stop this from becoming an all-seeds claim. I2 is also the practical model: `1,206,112` parameters versus C2's `11,170,884`, and primary-seed five-fold runtime `28.56` versus `91.58` minutes. I2 remains the current candidate; the ultimate winner is not frozen here. Trace: `results/evidence/task2/seed_stability/manifest.json`.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 9.1a Finalist comparison figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(stability_artifacts["comparison_figure"])))

> **Interpretation — 9.1a Finalist comparison figure**
>
> **Why this output exists.** This output isolates 9.1a finalist comparison figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The axis labels name the compared metric and candidate; positive candidate-minus-reference values favour the candidate, while zero means no observed change.
>
> **What it shows.** I2 leads both seeds while using about one ninth of C2's parameters and much less training time.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 9.2 Per-class performance and confusion matrices

Per-class evidence shows whether overall improvement is real or only comes from Summer.

In [ ]:
candidate_evidence = {
    "C2": "results/evidence/task2/g3_c2_t0_resnet18/manifest.json",
    "I2": (
        "results/evidence/task2/g4_i2_article_type_lambda_0_3_c1/manifest.json"
    ),
}
per_class_parts = []
confusion_tables = {}
candidate_artifact_paths = {}
for candidate, manifest_path in candidate_evidence.items():
    candidate_manifest, candidate_artifacts = load_verified_notebook_manifest(
        manifest_path
    )
    candidate_artifact_paths[candidate] = candidate_artifacts
    assert candidate_manifest["coverage"]["row_count"] == 32753
    assert candidate_manifest["coverage"]["protected_id_count"] == 0
    assert len(candidate_manifest["run_ids"]) == 5
    metrics = pd.read_csv(candidate_artifacts["per_class_metrics"])
    assert metrics["label"].tolist() == ["Fall", "Spring", "Summer", "Winter"]
    metrics.insert(0, "candidate", candidate)
    per_class_parts.append(metrics)
    confusion_tables[candidate] = pd.read_csv(
        candidate_artifacts["confusion_matrix"]
    ).set_index("true_label")

per_class_finalists = pd.concat(per_class_parts, ignore_index=True)
display(per_class_finalists)


> **Interpretation — measured evidence:** I2 improves every class F1 over C2 at seed `2753`: Fall `+0.023211`, Spring `+0.017238`, Summer `+0.010526`, and Winter `+0.019631`. The largest error remains Fall -> Summer, reduced from `3,041` C2 rows to `2,685` I2 rows. Winter -> Summer also falls from `1,583` to `1,453`. Spring is still the hardest recall problem: I2 recovers `860/1,329` Spring rows (`0.647103` recall), while `314` go to Summer. The gain is therefore broad, but it does not remove the weak-visual-signal and minority-class limits. Each figure and table is tied to five run IDs in its verified manifest.

> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.

#### 9.2.4 I2 confusion matrix

In [ ]:
from IPython.display import display

display(confusion_tables["I2"])

> **Interpretation — 9.2.4 I2 confusion matrix**
>
> **Why this output exists.** This output isolates 9.2.4 i2 confusion matrix so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Rows are true labels, columns are predicted labels, and each cell is an OOF row count; the diagonal is correct classification.
>
> **What it shows.** I2 improves all four class F1 scores; Fall and Spring errors into Summer remain the main weakness.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.2.3 C2 confusion matrix

In [ ]:
from IPython.display import display

display(confusion_tables["C2"])

> **Interpretation — 9.2.3 C2 confusion matrix**
>
> **Why this output exists.** This output isolates 9.2.3 c2 confusion matrix so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Rows are true labels, columns are predicted labels, and each cell is an OOF row count; the diagonal is correct classification.
>
> **What it shows.** I2 improves all four class F1 scores; Fall and Spring errors into Summer remain the main weakness.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.2.2 I2 finalist evidence figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(candidate_artifact_paths["I2"]["figure"])))

> **Interpretation — 9.2.2 I2 finalist evidence figure**
>
> **Why this output exists.** This output isolates 9.2.2 i2 finalist evidence figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** In the confusion matrix, rows are true Season labels and columns are predictions; darker cells contain more rows. The companion bars are per-class F1 and the dashed line is pooled macro-F1.
>
> **What it shows.** I2 improves all four class F1 scores; Fall and Spring errors into Summer remain the main weakness.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.2.1 C2 finalist evidence figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(candidate_artifact_paths["C2"]["figure"])))

> **Interpretation — 9.2.1 C2 finalist evidence figure**
>
> **Why this output exists.** This output isolates 9.2.1 c2 finalist evidence figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** In the confusion matrix, rows are true Season labels and columns are predictions; darker cells contain more rows. The companion bars are per-class F1 and the dashed line is pooled macro-F1.
>
> **What it shows.** I2 improves all four class F1 scores; Fall and Spring errors into Summer remain the main weakness.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 9.3 Calibration and learning behaviour

Confidence quality and epoch behaviour require different figures, so they are split below.

#### 9.3.1 Calibration and risk-coverage

Reliable confidence supports the app's human-review decision.

In [ ]:
calibration_manifest, calibration_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/calibration/manifest.json"
)
assert calibration_manifest["gate"] == "G6-CALIBRATION"
assert calibration_manifest["holdout_opened"] is False
assert calibration_manifest["cross_fitted_evaluation_claim_allowed"] is True
assert calibration_manifest["deployment_temperature_evaluation_claim_allowed"] is False
assert calibration_manifest["app_threshold_frozen"] is False
assert calibration_manifest["ultimate_winner_frozen"] is False

calibration_summary = pd.read_csv(calibration_artifacts["calibration_summary"])
review_budgets = pd.read_csv(calibration_artifacts["review_budget_summary"])
assert set(calibration_summary["support"]) == {32753}
assert set(calibration_summary["candidate"]) == {"C2", "I2"}
assert set(calibration_summary["calibration_method"]) == {
    "uncalibrated",
    "cross_fitted_temperature",
}
diagnostic_budget_20 = review_budgets.loc[
    review_budgets["calibration_method"].eq("cross_fitted_temperature")
    & review_budgets["review_budget"].eq(0.2)
].copy()
display(calibration_summary)


# Additional single-output views rendered below:
# display(
#     diagnostic_budget_20.loc[
#         :,
#         [
#             "candidate",
#             "review_rate",
#             "coverage",
#             "confidence_threshold",
#             "selective_macro_f1",
#             "selective_risk",
#         ],
#     ]
# )
# display(Image(filename=str(calibration_artifacts["calibration_reliability_figure"])))
# display(Image(filename=str(calibration_artifacts["risk_coverage_figure"])))


> **Interpretation — measured evidence:** Both raw models are overconfident. Five-fold cross-fitted temperature scaling leaves labels and macro-F1 unchanged but improves C2 NLL `0.665736 -> 0.638867` and ECE `0.057456 -> 0.011701`; I2 improves from NLL `0.632467 -> 0.608241` and ECE `0.046210 -> 0.016767`. At a diagnostic 20% review budget, retained-row macro-F1 is `0.808008` for C2 and `0.825837` for I2. This is a risk-coverage example, not an app policy: the threshold is not frozen because no business error cost was supplied. Only cross-fitted temperatures support evaluation claims; the scalar fitted on all OOF rows is future bundle metadata, and holdout remains sealed.

> **Chart guide.** Coverage is the fraction retained without review; the other axis shows selective risk or macro-F1. Moving toward lower coverage reviews more uncertain rows.

#### 9.3.1a 20% review-budget table

In [ ]:
from IPython.display import display

display(
    diagnostic_budget_20.loc[
        :,
        [
            "candidate",
            "review_rate",
            "coverage",
            "confidence_threshold",
            "selective_macro_f1",
            "selective_risk",
        ],
    ]
)

> **Interpretation — 9.3.1a 20% review-budget table**
>
> **Why this output exists.** This output isolates 9.3.1a 20% review-budget table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `review_rate` is the fraction sent to a person, `coverage` is the retained fraction, `confidence_threshold` is the diagnostic cutoff, and selective metrics are computed only on retained rows.
>
> **What it shows.** Cross-fitted temperature scaling sharply reduces NLL and ECE without changing labels; the 20% review example is diagnostic, not a policy.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.3.1b Calibration reliability figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(calibration_artifacts["calibration_reliability_figure"])))

> **Interpretation — 9.3.1b Calibration reliability figure**
>
> **Why this output exists.** This output isolates 9.3.1b calibration reliability figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Mean confidence is on the x-axis and observed accuracy on the y-axis. The diagonal is ideal calibration; orange is raw softmax and blue is cross-fitted temperature scaling. Lower panels show how many rows populate each bin.
>
> **What it shows.** Cross-fitted temperature scaling sharply reduces NLL and ECE without changing labels; the 20% review example is diagnostic, not a policy.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.3.1c Risk-coverage figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(calibration_artifacts["risk_coverage_figure"])))

> **Interpretation — 9.3.1c Risk-coverage figure**
>
> **Why this output exists.** This output isolates 9.3.1c risk-coverage figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Coverage is the fraction retained without review; the other axis shows selective risk or macro-F1. Moving toward lower coverage reviews more uncertain rows.
>
> **What it shows.** Cross-fitted temperature scaling sharply reduces NLL and ECE without changing labels; the 20% review example is diagnostic, not a policy.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.3.2 Learning curves and epoch behaviour

Train and validation curves explain whether the fixed budget and refit epoch rule are reasonable.

In [ ]:
learning_curve_summary = pd.read_csv(
    stability_artifacts["learning_curve_summary"]
)
assert learning_curve_summary["fold_count"].eq(5).all()
assert set(learning_curve_summary["candidate"]) == {"C2", "I2"}
primary_seed_epochs = finalist_leaderboard.loc[
    finalist_leaderboard["seed"].eq(2753),
    ["candidate", "median_best_epoch", "fold_sd_macro_f1"],
].sort_values("candidate")
assert primary_seed_epochs["median_best_epoch"].tolist() == [24.0, 24.0]
common_horizons = (
    learning_curve_summary.groupby(["candidate", "seed"], observed=True)
    ["common_five_fold_horizon"]
    .first()
    .rename("common_five_fold_horizon")
    .reset_index()
)
display(primary_seed_epochs)


# Additional single-output views rendered below:
# display(common_horizons)
# display(Image(filename=str(stability_artifacts["learning_curves"])))


> **Interpretation — measured evidence:** The teacher-style chart keeps all five folds at every shown epoch: common horizons are 11 epochs for C2 and 14 for I2 across both seeds. Training loss keeps falling after validation loss and validation macro-F1 begin to flatten or vary, so the best-validation-macro-F1 checkpoint and patience rule are justified. The median best epoch is `24` for both primary-seed candidates. If the scorecard selects either one, refit therefore uses 24 development epochs; this value comes only from CV history, never holdout. Train and validation loss are not compared numerically across C2 and I2 because I2 includes an auxiliary loss.

> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.

#### 9.3.2a Common five-fold horizons

In [ ]:
from IPython.display import display

display(common_horizons)

> **Interpretation — 9.3.2a Common five-fold horizons**
>
> **Why this output exists.** This output isolates 9.3.2a common five-fold horizons so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `fold` is the canonical validation fold; fold metrics are computed on disjoint OOF rows; mean/SD summarize level and variation without selecting the best fold.
>
> **What it shows.** Validation macro-F1 flattens before training loss stops falling, supporting best-macro-F1 checkpointing and the frozen 24-epoch refit rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 9.3.2b Learning-behaviour figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(stability_artifacts["learning_curves"])))

> **Interpretation — 9.3.2b Learning-behaviour figure**
>
> **Why this output exists.** This output isolates 9.3.2b learning-behaviour figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** Validation macro-F1 flattens before training loss stops falling, supporting best-macro-F1 checkpointing and the frozen 24-epoch refit rule.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 10. Error and shortcut analysis

Test the exact weaknesses identified by EDA using OOF predictions only.

### 10.1 Spring and ArticleType shortcut slices

Minority-class failure and shortcut conflict are distinct analyses, so each has its own code cell.

#### 10.1.1 Spring minority analysis

Spring needs direct evidence because it represents only 4.06% of valid labels.

In [ ]:
slice_manifest, slice_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/shortcut_error_slices/manifest.json"
)
assert slice_manifest["gate"] == "G6-SLICE"
assert slice_manifest["decision_status"] == "closed"
assert slice_manifest["analysis_role"] == "development_oof_diagnosis_only"
assert slice_manifest["candidate_selection_affected"] is False
assert slice_manifest["ultimate_winner_frozen"] is False

spring_metrics = pd.read_csv(slice_artifacts["spring_metrics"])
spring_destinations = pd.read_csv(slice_artifacts["spring_destinations"])
assert set(spring_metrics["candidate"]) == {"C2", "I2"}
assert set(spring_metrics["seed"]) == {2753, 2026}
assert spring_metrics["support"].eq(1329).all()
assert spring_metrics["main_error_destination"].eq("Summer").all()

baseline_manifests = {
    "B0 majority": "results/evidence/task2/b0_majority/manifest.json",
    "B1 HOG+HSV SVM": "results/evidence/task2/b1_hog_hsv_svm/manifest.json",
}
spring_progression_rows = []
for stage, manifest_path in baseline_manifests.items():
    _, artifacts = load_verified_notebook_manifest(manifest_path)
    spring_row = pd.read_csv(artifacts["per_class_metrics"]).query(
        "label == 'Spring'"
    ).iloc[0]
    spring_progression_rows.append(
        {
            "stage": stage,
            "precision": spring_row["precision"],
            "recall": spring_row["recall"],
            "f1": spring_row["f1"],
            "support": spring_row["support"],
        }
    )
_, i1_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/i1_class_balance/manifest.json"
)
i1_spring = pd.read_csv(i1_artifacts["per_class_comparison"]).query(
    "label == 'Spring'"
).iloc[0]
for prefix, stage in (("reference", "C1 unweighted"), ("i1", "I1 balanced")):
    spring_progression_rows.append(
        {
            "stage": stage,
            "precision": i1_spring[f"{prefix}_precision"],
            "recall": i1_spring[f"{prefix}_recall"],
            "f1": i1_spring[f"{prefix}_f1"],
            "support": i1_spring[f"{prefix}_support"],
        }
    )
for candidate in ("C2", "I2"):
    row = spring_metrics.query("candidate == @candidate and seed == 2753").iloc[0]
    spring_progression_rows.append(
        {
            "stage": f"{candidate} seed 2753",
            "precision": row["precision"],
            "recall": row["recall"],
            "f1": row["f1"],
            "support": row["support"],
        }
    )
spring_progression = pd.DataFrame(spring_progression_rows)
display(spring_progression)


# Additional single-output views rendered below:
# display(
#     spring_metrics.loc[
#         :,
#         [
#             "candidate",
#             "seed",
#             "precision",
#             "recall",
#             "f1",
#             "mean_confidence",
#             "error_mean_confidence",
#             "main_error_destination",
#             "main_error_count",
#         ],
#     ].sort_values(["seed", "candidate"])
# )
# display(Image(filename=str(slice_artifacts["spring_destinations_figure"])))


> **Interpretation — measured evidence:** The class-imbalance EDA warning was accurate: B0 ignores all `1,329` Spring rows (`F1 = 0`), while B1 proves that visual features can recover Spring (`F1 = 0.486901`). C1 reaches `0.744975`; I1 raises recall by `0.022573` but loses precision and falls to `0.702439`, so imbalance was real but class weighting was the wrong remedy. I2 reaches Spring F1 `0.764784` at seed `2753` and `0.754291` at seed `2026`, beating C2 by `+0.017238` and `+0.006426`. It still sends `314` and `332` Spring rows to Summer. The model no longer ignores Spring, but Spring remains visually ambiguous and needs class-specific reporting. Trace: `results/evidence/task2/shortcut_error_slices/manifest.json`.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 10.1.1a Spring slice metrics

In [ ]:
from IPython.display import display

display(
    spring_metrics.loc[
        :,
        [
            "candidate",
            "seed",
            "precision",
            "recall",
            "f1",
            "mean_confidence",
            "error_mean_confidence",
            "main_error_destination",
            "main_error_count",
        ],
    ].sort_values(["seed", "candidate"])
)

> **Interpretation — 10.1.1a Spring slice metrics**
>
> **Why this output exists.** This output isolates 10.1.1a spring slice metrics so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `label` names the true class; `support` is its row count; precision measures false positives, recall measures missed true rows, F1 balances both, and `delta` is candidate minus reference.
>
> **What it shows.** I2 improves Spring F1 over C2 at both seeds, but hundreds of Spring rows are still predicted as Summer.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 10.1.1b Spring error-destination figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(slice_artifacts["spring_destinations_figure"])))

> **Interpretation — 10.1.1b Spring error-destination figure**
>
> **Why this output exists.** This output isolates 10.1.1b spring error-destination figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Bars count where true Spring rows are predicted. Smaller non-Spring bars are better; the Summer bar exposes the main minority-class error destination.
>
> **What it shows.** I2 improves Spring F1 over C2 at both seeds, but hundreds of Spring rows are still predicted as Summer.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 10.1.2 ArticleType aligned versus conflict

The conflict slice tests whether image performance survives when the ArticleType shortcut is wrong.

In [ ]:
slice_metrics = pd.read_csv(slice_artifacts["slice_metrics"])
slice_deltas = pd.read_csv(slice_artifacts["candidate_slice_deltas"])
article_type_audit = pd.read_csv(slice_artifacts["article_type_fold_audit"])
article_type_rows = slice_deltas.loc[
    slice_deltas["slice_family"].eq("article_type_shortcut"),
    [
        "seed",
        "slice_name",
        "c2_support",
        "c2_macro_f1",
        "i2_macro_f1",
        "i2_minus_c2_macro_f1",
        "c2_mean_confidence",
        "i2_mean_confidence",
    ],
].copy()
assert set(article_type_rows["slice_name"]) == {
    "aligned",
    "conflict",
    "missing_article_type",
    "unseen_article_type",
}
missing_article_type = article_type_rows.loc[
    article_type_rows["slice_name"].eq("missing_article_type")
]
assert missing_article_type["c2_support"].eq(0).all()
assert missing_article_type[["c2_macro_f1", "i2_macro_f1"]].isna().all().all()
assert len(article_type_audit) == 5
assert set(article_type_audit["fold"]) == set(range(5))
assert article_type_audit["training_id_sha256"].nunique() == 5
assert (
    article_type_audit["training_products"]
    == article_type_audit["auxiliary_training_products"]
).all()
display(article_type_audit)


# Additional single-output views rendered below:
# display(article_type_rows.sort_values(["seed", "slice_name"]))
# display(Image(filename=str(slice_artifacts["slice_macro_f1_figure"])))


> **Interpretation — measured evidence:** The EDA shortcut warning is accurate: aligned rows score about `0.85-0.87` macro-F1, but the same frozen models fall to `0.40-0.43` on `11,619` conflict rows. I2 nevertheless improves both groups at both seeds. Its conflict gain over C2 is `+0.024830` at seed `2753` and `+0.026993` at seed `2026`, larger than its aligned gains (`+0.008887` and `+0.009981`). This weakens the claim that I2 only memorises the ArticleType shortcut; it does not remove shortcut risk because conflict performance remains poor. The `19` unseen-ArticleType rows are too small to judge. Every mapping was fitted on the matching training fold, and true ArticleType is never an inference input.

> **Chart guide.** The chart separates fold-fitted ArticleType-aligned and conflict slices. Higher macro-F1 is better; the aligned-to-conflict gap measures shortcut sensitivity.

#### 10.1.2a ArticleType aligned/conflict table

In [ ]:
from IPython.display import display

display(article_type_rows.sort_values(["seed", "slice_name"]))

> **Interpretation — 10.1.2a ArticleType aligned/conflict table**
>
> **Why this output exists.** This output isolates 10.1.2a articletype aligned/conflict table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.
>
> **What it shows.** Conflict rows remain much harder than aligned rows, while I2 improves both slices and therefore is not explained by shortcut alignment alone.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 10.1.2b ArticleType slice figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(slice_artifacts["slice_macro_f1_figure"])))

> **Interpretation — 10.1.2b ArticleType slice figure**
>
> **Why this output exists.** This output isolates 10.1.2b articletype slice figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The chart separates fold-fitted ArticleType-aligned and conflict slices. Higher macro-F1 is better; the aligned-to-conflict gap measures shortcut sensitivity.
>
> **What it shows.** Conflict rows remain much harder than aligned rows, while I2 improves both slices and therefore is not explained by shortcut alignment alone.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 10.2 Acquisition year and compression slices

Acquisition era and file compression are separate shortcut warnings and are analysed independently.

#### 10.2.1 Acquisition-year slice

The year slice tests the strong collection-era association found in development EDA.

In [ ]:
year_rows = slice_deltas.loc[
    slice_deltas["slice_family"].eq("acquisition_year")
    & slice_deltas["slice_name"].ne("missing_year"),
    [
        "seed",
        "slice_name",
        "c2_support",
        "c2_accuracy",
        "i2_accuracy",
        "c2_macro_f1",
        "i2_macro_f1",
        "i2_minus_c2_macro_f1",
        "c2_mean_confidence",
        "i2_mean_confidence",
    ],
].copy()
assert set(year_rows["slice_name"]) == {"dominant_2011_2012", "other_years"}
assert set(year_rows["c2_support"]) == {22896, 9857}
year_contrasts = pd.read_csv(slice_artifacts["slice_contrasts"]).query(
    "slice_family == 'acquisition_year'"
)
assert len(year_contrasts) == 4
display(year_rows.sort_values(["seed", "slice_name"]))


# Additional single-output views rendered below:
# display(
#     year_contrasts.loc[
#         :,
#         [
#             "candidate",
#             "seed",
#             "reference_macro_f1",
#             "comparison_macro_f1",
#             "comparison_minus_reference_macro_f1",
#         ],
#     ]
# )


> **Interpretation — measured evidence:** The acquisition-era EDA warning is supported, but its simple story is not. The `22,896` rows from 2011-2012 have higher accuracy yet much lower macro-F1 than the `9,857` other-year rows: I2 seed `2753` is `0.774764` versus `0.733996` accuracy, but `0.591334` versus `0.682438` macro-F1. This metric reversal shows that era and class mix interact; it does not show that old images cause a prediction. I2 improves the dominant-era macro-F1 by `+0.042406` at seed `2753` and `+0.034396` at seed `2026`, while other-year gains are smaller. Year was joined only after OOF prediction and is never a model feature.

> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.

#### 10.2.1a Acquisition-year contrast table

In [ ]:
from IPython.display import display

display(
    year_contrasts.loc[
        :,
        [
            "candidate",
            "seed",
            "reference_macro_f1",
            "comparison_macro_f1",
            "comparison_minus_reference_macro_f1",
        ],
    ]
)

> **Interpretation — 10.2.1a Acquisition-year contrast table**
>
> **Why this output exists.** This output isolates 10.2.1a acquisition-year contrast table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.
>
> **What it shows.** The dominant acquisition years have higher accuracy but lower macro-F1, showing an era/class-mix interaction rather than a causal year effect.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 10.2.2 Training-fitted file-size slice

File-size quartiles test sensitivity to compression traces without using file bytes as a feature.

In [ ]:
file_size_boundaries = pd.read_csv(slice_artifacts["file_size_boundaries"])
file_size_rows = slice_deltas.loc[
    slice_deltas["slice_family"].eq("file_size_quartile"),
    [
        "seed",
        "slice_name",
        "c2_support",
        "c2_macro_f1",
        "i2_macro_f1",
        "i2_minus_c2_macro_f1",
        "c2_mean_confidence",
        "i2_mean_confidence",
    ],
].copy()
assert len(file_size_boundaries) == 5
assert set(file_size_boundaries["fold"]) == set(range(5))
assert file_size_boundaries["training_id_sha256"].nunique() == 5
assert file_size_boundaries["training_products"].between(26201, 26203).all()
assert set(file_size_rows["slice_name"]) == {
    "q1_smallest",
    "q2",
    "q3",
    "q4_largest",
}
display(file_size_boundaries)


# Additional single-output views rendered below:
# display(file_size_rows.sort_values(["seed", "slice_name"]))


> **Interpretation — measured evidence:** The file-size EDA warning is supported as a risk, not as causation. The smallest-file quartile is consistently harder than the largest: at seed `2753`, C2 rises from `0.651847` to `0.708408` macro-F1 and I2 from `0.682055` to `0.720897`; the I2 gap is `0.038842` (`0.043815` at seed `2026`). The middle quartiles are not monotonic, so file size is not a simple quality scale. Each fold uses only its `26,201-26,203` training rows to fit q25/q50/q75 boundaries; file bytes are never supplied to the CNN. The result motivates the controlled JPEG probe in Section 11, not a metadata feature.

> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.

#### 10.2.2a File-size quartile table

In [ ]:
from IPython.display import display

display(file_size_rows.sort_values(["seed", "slice_name"]))

> **Interpretation — 10.2.2a File-size quartile table**
>
> **Why this output exists.** This output isolates 10.2.2a file-size quartile table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.
>
> **What it shows.** The smallest-file quartile is consistently harder, but the middle quartiles are not monotonic; file size is a risk proxy, not a quality score.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 10.3 Family-size and image-mode slices

Related-product structure and rare image mode answer different generalisation questions.

#### 10.3.1 Product-family size slice

This slice checks whether results are stronger for products with related rows.

In [ ]:
family_rows = slice_deltas.loc[
    slice_deltas["slice_family"].eq("product_family_size"),
    [
        "seed",
        "slice_name",
        "c2_support",
        "c2_accuracy",
        "i2_accuracy",
        "c2_macro_f1",
        "i2_macro_f1",
        "i2_minus_c2_macro_f1",
    ],
].copy()
family_contrasts = pd.read_csv(slice_artifacts["slice_contrasts"]).query(
    "slice_family == 'product_family_size'"
)
assert set(family_rows["slice_name"]) == {"singleton", "multirow"}
assert set(family_rows["c2_support"]) == {19023, 13730}
assert len(family_contrasts) == 4
display(family_rows.sort_values(["seed", "slice_name"]))


# Additional single-output views rendered below:
# display(
#     family_contrasts.loc[
#         :,
#         [
#             "candidate",
#             "seed",
#             "reference_macro_f1",
#             "comparison_macro_f1",
#             "comparison_minus_reference_macro_f1",
#         ],
#     ]
# )


> **Interpretation — measured evidence:** Quality does not improve on the multi-row family slice. I2 scores `0.750922` on `19,023` singleton rows versus `0.741507` on `13,730` multi-row rows at seed `2753`; the multi-row-minus-singleton gap is `-0.009415` and repeats as `-0.011120` at seed `2026`. C2 shows the same small direction. This weakens the concern that aggregate quality depends on related products. It does not prove independence: `product_family_group` is a conservative split group, not a verified SKU identity, and the folds already prevent a group from crossing training and validation.

> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.

#### 10.3.1a Product-family contrast table

In [ ]:
from IPython.display import display

display(
    family_contrasts.loc[
        :,
        [
            "candidate",
            "seed",
            "reference_macro_f1",
            "comparison_macro_f1",
            "comparison_minus_reference_macro_f1",
        ],
    ]
)

> **Interpretation — 10.3.1a Product-family contrast table**
>
> **Why this output exists.** This output isolates 10.3.1a product-family contrast table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.
>
> **What it shows.** Multi-row families are slightly harder at both seeds, so aggregate quality is not inflated by the family slice.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 10.3.2 Greyscale versus RGB slice

Rare greyscale images may reveal colour dependence or transform problems.

In [ ]:
image_mode_rows = slice_deltas.loc[
    slice_deltas["slice_family"].eq("image_mode"),
    [
        "seed",
        "slice_name",
        "c2_support",
        "c2_accuracy",
        "i2_accuracy",
        "c2_macro_f1",
        "i2_macro_f1",
        "i2_minus_c2_macro_f1",
        "c2_mean_confidence",
        "i2_mean_confidence",
    ],
].copy()
assert set(image_mode_rows["slice_name"]) == {"greyscale", "rgb", "other_mode"}
assert set(image_mode_rows["c2_support"]) == {0, 294, 32459}
other_mode = image_mode_rows.loc[image_mode_rows["slice_name"].eq("other_mode")]
assert other_mode["c2_support"].eq(0).all()
assert other_mode[["c2_macro_f1", "i2_macro_f1"]].isna().all().all()
greyscale_deltas = image_mode_rows.loc[
    image_mode_rows["slice_name"].eq("greyscale"),
    "i2_minus_c2_macro_f1",
]
assert (greyscale_deltas.min() < 0) and (greyscale_deltas.max() > 0)
display(image_mode_rows.sort_values(["seed", "slice_name"]))

> **Interpretation — measured evidence:** Greyscale is harder than RGB for both candidates, but only `294` greyscale rows exist. More importantly, I2 versus C2 reverses across seeds: `-0.045315` macro-F1 at seed `2753`, then `+0.080164` at seed `2026`. The EDA colour/transform warning therefore remains plausible, while any winner claim from this rare slice is rejected. Together with A1 colour jitter hurting the main score, the safe conclusion is that colour may carry real Season signal and should not be aggressively distorted. All images are still converted to RGB by the shared transform; this slice uses the structural audit only after prediction.

> **Column guide.** Candidate and seed identify the model; slice name identifies the subgroup; `support` is its row count; accuracy and macro-F1 measure different class-balance views; delta is comparison minus reference.

## 11. Robustness and efficiency

Measure controlled image degradation and the practical cost of the final candidates.

### 11.1 Deterministic perturbation tests

JPEG, brightness, and blur tests check whether mild real-world changes break the same frozen model.

In [ ]:
robustness_manifest, robustness_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/robustness_cost/manifest.json"
)
assert robustness_manifest["gate"] == "G6-ROBUSTNESS-COST"
assert robustness_manifest["decision_status"] == "closed"
assert robustness_manifest["git_dirty"] is False
assert (
    robustness_manifest["analysis_role"]
    == "development_stress_and_machine_cost_diagnosis_only"
)
assert robustness_manifest["candidate_selection_affected"] is False
assert robustness_manifest["ultimate_winner_frozen"] is False

robustness_metrics = pd.read_csv(robustness_artifacts["pooled_metrics"])
robustness_comparison = pd.read_csv(
    robustness_artifacts["candidate_comparison"]
)
clean_reconciliation = pd.read_csv(
    robustness_artifacts["clean_reconciliation"]
)
probe_registry = pd.read_csv(
    robustness_artifacts["probe_registry"], dtype=str, keep_default_na=False
)
probe_registry["rows"] = pd.to_numeric(
    probe_registry["rows"], errors="raise"
).astype(int)
expected_conditions = {
    "clean",
    "jpeg_quality_85",
    "brightness_0_85",
    "brightness_1_15",
    "gaussian_blur_radius_1",
}
assert set(robustness_metrics["candidate"]) == {"C2", "I2"}
assert set(robustness_metrics["condition"]) == expected_conditions
assert robustness_metrics["support"].eq(32753).all()
assert len(probe_registry) == probe_registry["probe_id"].nunique() == 50
assert probe_registry["status"].eq("completed").all()
assert (
    probe_registry.groupby(
        ["candidate", "condition"], observed=True
    )["rows"].sum().eq(32753).all()
)
assert clean_reconciliation["support"].eq(32753).all()
assert clean_reconciliation["clean_reference"].eq("verified_frozen_oof").all()
assert clean_reconciliation["prediction_agreement"].eq(1.0).all()
assert clean_reconciliation["maximum_probability_delta"].eq(0.0).all()
display(
    robustness_metrics.loc[
        :,
        [
            "candidate",
            "condition",
            "macro_f1",
            "delta_macro_f1_vs_clean",
            "spring_recall",
            "prediction_agreement_with_clean",
            "material_macro_f1_degradation",
        ],
    ]
)


# Additional single-output views rendered below:
# display(robustness_comparison)
# display(Image(filename=str(robustness_artifacts["robustness_figure"])))


> **Interpretation — measured evidence:** Darkening to brightness `0.85` is the worst stress for both models: C2 falls from `0.735036` to `0.337542` macro-F1 and I2 from `0.752687` to `0.363961`; clean-label agreement is only `0.512625` and `0.614112`. JPEG quality 85 is mildest, but still costs `0.012671` C2 and `0.022496` I2 macro-F1. Blur and brightness `1.15` also cause material drops. I2 remains above C2 in every condition, yet its Spring recall almost vanishes under darkening (`0.003010`). This supports the EDA acquisition/colour warning and keeps I2 as the practical candidate, but it also requires a human-review path for shifted images. These are synthetic primary-seed stresses, not proof of all real-world robustness, and they do not reopen G5 selection or freeze a winner.

> **Chart guide.** Use the labelled axes, legend, and reference lines to identify the compared quantity; the numeric decision remains tied to the verified table in the same subsection.

#### 11.1a Robustness comparison table

In [ ]:
from IPython.display import display

display(robustness_comparison)

> **Interpretation — 11.1a Robustness comparison table**
>
> **Why this output exists.** This output isolates 11.1a robustness comparison table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Candidate/variant names the row; `pooled_macro_f1` is computed across all OOF rows; `fold_sd_macro_f1` is five-fold variation; Spring fields expose minority performance; delta is candidate minus reference; selected/eligible fields apply the frozen rule.
>
> **What it shows.** Brightness 0.85 is the worst perturbation and nearly removes Spring recall; I2 nevertheless remains above C2 under every declared condition.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 11.1b Robustness figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(robustness_artifacts["robustness_figure"])))

> **Interpretation — 11.1b Robustness figure**
>
> **Why this output exists.** This output isolates 11.1b robustness figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Conditions are compared with each model's clean OOF reference. Quality or delta is on the y-axis; larger negative movement means greater sensitivity to that shift.
>
> **What it shows.** Brightness 0.85 is the worst perturbation and nearly removes Spring recall; I2 nevertheless remains above C2 under every declared condition.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 11.2 Deployment cost

Inference latency and resource size use different measurement protocols, so they are separated below. Seed stability is already handled in Section 8.4.

#### 11.2.1 Batch-one inference latency

The app serves one uploaded image at a time, so batch-one timing is the relevant cost.

In [ ]:
deployment_cost = pd.read_csv(robustness_artifacts["deployment_cost"])
with robustness_artifacts["runtime"].open(encoding="utf-8") as handle:
    robustness_runtime = json.load(handle)
assert len(deployment_cost) == 4
assert deployment_cost["available"].all()
assert set(deployment_cost["candidate"]) == {"C2", "I2"}
assert set(deployment_cost["device"]) == {"cpu", "cuda"}
assert deployment_cost["batch_size"].eq(1).all()
assert deployment_cost["model_only_warmups"].eq(30).all()
assert deployment_cost["model_only_repeats"].eq(200).all()
assert deployment_cost["end_to_end_warmups"].eq(10).all()
assert deployment_cost["end_to_end_repeats"].eq(50).all()
latency_view = deployment_cost.loc[
    :,
    [
        "candidate",
        "device",
        "model_only_median_ms",
        "model_only_p95_ms",
        "end_to_end_median_ms",
        "end_to_end_p95_ms",
        "model_only_throughput_per_second_at_median",
    ],
].sort_values(["device", "candidate"])
hardware_view = pd.DataFrame(
    [
        {
            "platform": robustness_runtime["platform"],
            "python": robustness_runtime["python"],
            "cpu_threads": robustness_runtime["logical_cpu_count"],
            "gpu": robustness_runtime["gpu"]["name"],
            "cuda_runtime": robustness_runtime["cuda_runtime"],
        }
    ]
)
display(hardware_view)


# Additional single-output views rendered below:
# display(latency_view)


> **Interpretation — measured evidence:** Both models are fast enough for one-image interaction on the recorded machine, and I2 is clearly faster. Warm batch-one end-to-end median/p95 is `6.493/7.069 ms` on one CPU thread for I2 versus `47.357/51.721 ms` for C2. On the RTX 4070 Laptop GPU it is `1.882/2.198 ms` versus `3.052/3.301 ms`. Model-only medians show the same order: `6.547` versus `47.976 ms` on CPU and `1.312` versus `2.246 ms` on CUDA. Preprocessing is included only in the end-to-end columns; training and data loading are excluded. These warm-cache numbers are machine-specific, not universal service latency.

> **Column guide.** `device` identifies CPU/GPU; measurement scope distinguishes model-only from end-to-end; median is typical latency and p95 is the slower-tail value in milliseconds.

#### 11.2.1a Batch-one latency table

In [ ]:
from IPython.display import display

display(latency_view)

> **Interpretation — 11.2.1a Batch-one latency table**
>
> **Why this output exists.** This output isolates 11.2.1a batch-one latency table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `device` identifies CPU/GPU; measurement scope distinguishes model-only from end-to-end; median is typical latency and p95 is the slower-tail value in milliseconds.
>
> **What it shows.** I2 has much lower batch-one CPU and GPU latency than C2 on the recorded machine.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 11.2.2 Model and training resource cost

Model size, RAM/VRAM, and training time complete the practical comparison.

In [ ]:
primary_resources = finalist_leaderboard.loc[
    finalist_leaderboard["seed"].eq(2753),
    [
        "candidate",
        "pooled_macro_f1",
        "parameter_count",
        "peak_vram_mb",
        "five_fold_runtime_minutes",
    ],
].copy()
storage_cost = (
    deployment_cost.loc[
        deployment_cost["device"].eq("cpu"),
        [
            "candidate",
            "parameter_and_buffer_bytes",
            "training_checkpoint_bytes",
            "process_rss_before_bytes",
            "process_rss_after_inference_bytes",
            "process_rss_delta_bytes",
        ],
    ]
    .drop_duplicates("candidate")
    .copy()
)
cuda_memory = deployment_cost.loc[
    deployment_cost["device"].eq("cuda"),
    ["candidate", "peak_cuda_allocated_bytes"],
].copy()
resource_scorecard = (
    primary_resources.merge(storage_cost, on="candidate", validate="one_to_one")
    .merge(cuda_memory, on="candidate", validate="one_to_one")
    .sort_values("candidate")
)
for _, row in resource_scorecard.iterrows():
    expected = deployment_cost.loc[
        deployment_cost["candidate"].eq(row["candidate"]),
        "parameter_count",
    ]
    assert expected.eq(row["parameter_count"]).all()
assert resource_scorecard["candidate"].tolist() == ["C2", "I2"]
display(resource_scorecard)


# Additional single-output views rendered below:
# display(Image(filename=str(robustness_artifacts["deployment_cost_figure"])))


> **Interpretation — measured evidence:** I2 improves primary-seed macro-F1 by `0.017651` while using less of every stable resource measure. It has `1,206,112` parameters versus `11,170,884`; parameter/buffer storage is `4,832,192` versus `44,722,096` bytes (`0.108x`), and its training checkpoint is `14,521,135` versus `134,179,223` bytes. Five-fold training took `28.56` versus `91.58` minutes and peak training VRAM was `190.10` versus `606.39 MiB`. The isolated CUDA latency probe also peaks at about `15.9` versus `61.7 MiB`. Process RSS is shown for completeness but is not used to rank models because separate Python processes start with different allocator state. Here the quality gain does not need to justify extra cost: I2 is both better and cheaper.

> **Chart guide.** Panels compare quality with parameters, storage, training time, memory, and latency. Higher macro-F1 is better; lower cost measures are better.

#### 11.2.2a Resource-cost figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(robustness_artifacts["deployment_cost_figure"])))

> **Interpretation — 11.2.2a Resource-cost figure**
>
> **Why this output exists.** This output isolates 11.2.2a resource-cost figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Panels compare quality with parameters, storage, training time, memory, and latency. Higher macro-F1 is better; lower cost measures are better.
>
> **What it shows.** I2 is both more accurate and cheaper: fewer parameters, less storage, lower VRAM, and shorter training time.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 12. Explainability and failure cases

Use deterministic examples to understand where the model looks and why it fails.

### 12.1 Select representative examples deterministically

Fixed selection prevents cherry-picking attractive successes or dramatic failures.

In [ ]:
gradcam_manifest, gradcam_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/gradcam_failure_review/manifest.json"
)
assert gradcam_manifest["gate"] == "G6-GRADCAM-FAILURE-REVIEW"
assert gradcam_manifest["decision_status"] == "closed"
assert gradcam_manifest["git_dirty"] is False
assert gradcam_manifest["holdout_opened"] is False
assert gradcam_manifest["causal_failure_claim_allowed"] is False
assert gradcam_manifest["candidate_selection_affected"] is False
assert gradcam_manifest["ultimate_winner_frozen"] is False

selected_examples = pd.read_csv(gradcam_artifacts["selected_examples"])
heatmap_index = pd.read_csv(gradcam_artifacts["heatmap_index"])
gradcam_checkpoints = pd.read_csv(gradcam_artifacts["checkpoint_audit"])
assert len(selected_examples) == len(heatmap_index) == 48
assert selected_examples["id"].nunique() == 44
assert set(selected_examples["candidate"]) == {"C2", "I2"}
assert set(selected_examples["true_label"]) == {
    "Fall",
    "Spring",
    "Summer",
    "Winter",
}
assert set(selected_examples["selection_group"]) == {"correct", "incorrect"}
selection_counts = (
    selected_examples.groupby(
        ["candidate", "true_label", "selection_group"], observed=True
    )
    .size()
    .rename("selected_examples")
    .reset_index()
)
assert len(selection_counts) == 16
assert selection_counts["selected_examples"].eq(3).all()
assert heatmap_index["array_index"].tolist() == list(range(48))
assert gradcam_checkpoints["run_id"].nunique() == 8
display(selection_counts)


# Additional single-output views rendered below:
# display(
#     selected_examples.loc[
#         :,
#         [
#             "candidate",
#             "true_label",
#             "selection_group",
#             "selection_rank",
#             "id",
#             "fold",
#             "predicted_label",
#             "calibrated_confidence",
#             "run_id",
#         ],
#     ]
# )


> **Interpretation — measured evidence:** The predeclared rule selects exactly three high-confidence correct and three high-confidence incorrect OOF rows for every class and each candidate: `48` rows, `44` distinct images, and no class shortage. Confidence rank then ID fixes ties before any heatmap is viewed, preventing attractive maps from driving selection. Eight hash-verified checkpoint run IDs cover the folds containing these extremes; fold 2 simply contributes no selected extreme. This is balanced diagnostic coverage, not a random sample: the error counts below must never be read as population prevalence, and holdout remains sealed.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 12.1a Selected-example trace

In [ ]:
from IPython.display import display

display(
    selected_examples.loc[
        :,
        [
            "candidate",
            "true_label",
            "selection_group",
            "selection_rank",
            "id",
            "fold",
            "predicted_label",
            "calibrated_confidence",
            "run_id",
        ],
    ]
)

> **Interpretation — 12.1a Selected-example trace**
>
> **Why this output exists.** This output isolates 12.1a selected-example trace so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `id` is the development product; true/predicted labels define correctness; confidence ranks the fixed selection group; `run_id` identifies the fold checkpoint.
>
> **What it shows.** The deterministic rule selects balanced high-confidence correct and incorrect cases before any heatmap is inspected.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 12.2 Generate Grad-CAM diagnostics

Grad-CAM can reveal whether the model attends to the product or to borders and background.

In [ ]:
attention_metrics = pd.read_csv(gradcam_artifacts["attention_metrics"])
assert len(attention_metrics) == 48
assert attention_metrics["zero_heatmap"].eq(False).all()
assert attention_metrics["attention_review_flag"].eq(False).all()
assert attention_metrics["probability_max_absolute_delta"].max() <= 1e-4
assert (
    attention_metrics["gradcam_target_label"]
    == attention_metrics["predicted_label"]
).all()
attention_summary = (
    attention_metrics.groupby("candidate", observed=True)
    .agg(
        examples=("id", "size"),
        mean_foreground_attention_share=("foreground_attention_share", "mean"),
        mean_foreground_attention_lift=("foreground_attention_lift", "mean"),
        mean_border_attention_lift=("border_attention_lift", "mean"),
        max_probability_delta=("probability_max_absolute_delta", "max"),
        zero_heatmaps=("zero_heatmap", "sum"),
        review_flags=("attention_review_flag", "sum"),
    )
    .reset_index()
)
display(attention_summary)


# Additional single-output views rendered below:
# display(Image(filename=str(gradcam_artifacts["c2_contact_sheet"])))
# display(Image(filename=str(gradcam_artifacts["i2_contact_sheet"])))


> **Interpretation — measured evidence:** All `48` predicted-class Grad-CAM maps are non-zero and reconcile to the frozen OOF probabilities within `1e-4`. Mean foreground attention share/lift is `0.633160/1.490200` for C2 and `0.642716/1.719056` for I2; mean border lift stays below `1.0` for both (`0.674038` and `0.623785`), meaning the border receives less attention per pixel than a spatially uniform map. No map crosses the declared empty-map or border/background review rule. The sheets therefore show useful product-focused sensitivity in these fixed cases, but zero flags do not prove that shortcuts are absent. Grad-CAM is coarse, the non-white foreground mask misses some white products, and a heatmap cannot establish why a prediction occurred or whether a region is causal.

> **Chart guide.** Each tile overlays predicted-class sensitivity on the input image. Warmer colours mean greater local sensitivity; they do not prove that a region caused the decision.

#### 12.2a C2 Grad-CAM contact sheet

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(gradcam_artifacts["c2_contact_sheet"])))

> **Interpretation — 12.2a C2 Grad-CAM contact sheet**
>
> **Why this output exists.** This output isolates 12.2a c2 grad-cam contact sheet so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Each tile overlays predicted-class sensitivity on the input image. Warmer colours mean greater local sensitivity; they do not prove that a region caused the decision.
>
> **What it shows.** Grad-CAM sensitivity is mostly product-focused in the selected cases, but the maps cannot establish causality or shortcut absence.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 12.2b I2 Grad-CAM contact sheet

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(gradcam_artifacts["i2_contact_sheet"])))

> **Interpretation — 12.2b I2 Grad-CAM contact sheet**
>
> **Why this output exists.** This output isolates 12.2b i2 grad-cam contact sheet so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** Each tile overlays predicted-class sensitivity on the input image. Warmer colours mean greater local sensitivity; they do not prove that a region caused the decision.
>
> **What it shows.** Grad-CAM sensitivity is mostly product-focused in the selected cases, but the maps cannot establish causality or shortcut absence.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 12.3 Build the failure taxonomy

The rubric rewards honest analysis of limitations, not only successful examples.

In [ ]:
failure_taxonomy = pd.read_csv(gradcam_artifacts["failure_taxonomy"])
failure_summary = pd.read_csv(gradcam_artifacts["failure_taxonomy_summary"])
assert len(failure_taxonomy) == 24
assert failure_taxonomy["selection_group"].eq("incorrect").all()
assert failure_taxonomy["human_label_ambiguity_review_required"].eq(True).all()
assert set(failure_taxonomy["primary_failure_hypothesis"]) == {
    "article_type_shortcut_conflict",
    "weak_data_proxy",
}
assert failure_summary["selected_error_count"].sum() == 24
assert (
    failure_summary.groupby("candidate", observed=True)["selected_error_count"]
    .sum()
    .eq(12)
).all()
display(failure_summary)


# Additional single-output views rendered below:
# display(
#     failure_taxonomy.loc[
#         :,
#         [
#             "candidate",
#             "id",
#             "true_label",
#             "predicted_label",
#             "calibrated_confidence",
#             "articleType",
#             "article_type_shortcut",
#             "primary_failure_hypothesis",
#             "diagnostic_tags",
#             "run_id",
#         ],
#     ]
# )


> **Interpretation — measured evidence:** Among the `12` deliberately severe errors per model, C2 is split between `6` ArticleType-shortcut conflicts and `6` weak-data proxies; I2 has `9` conflicts and `3` weak-data proxies. Every selected error also requires human ambiguity review because a catalogue image often cannot make Season objectively clear. The real-use problem is therefore not only model capacity: label ambiguity and metadata associations can produce very confident wrong answers. The app should show probabilities and flag uncertain or shifted cases for a person. These tags are review hypotheses, not causal labels or frequency estimates; human causal adjudication is not complete, and the failure review does not change candidate selection.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 12.3a Failure-taxonomy table

In [ ]:
from IPython.display import display

display(
    failure_taxonomy.loc[
        :,
        [
            "candidate",
            "id",
            "true_label",
            "predicted_label",
            "calibrated_confidence",
            "articleType",
            "article_type_shortcut",
            "primary_failure_hypothesis",
            "diagnostic_tags",
            "run_id",
        ],
    ]
)

> **Interpretation — 12.3a Failure-taxonomy table**
>
> **Why this output exists.** This output isolates 12.3a failure-taxonomy table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** True/predicted labels and confidence describe the error; shortcut and hypothesis columns are diagnostic review tags; `run_id` preserves traceability.
>
> **What it shows.** The taxonomy records shortcut conflict, weak-data proxies, and human ambiguity as review hypotheses, not causal labels or prevalence estimates.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 13. Statistical and external comparison

Measure uncertainty between finalists and compare literature only when assumptions are made explicit.

### 13.1 Paired family bootstrap

Rows inside a product family may be related, so uncertainty should preserve those blocks.

In [ ]:
bootstrap_manifest, bootstrap_artifacts = load_verified_notebook_manifest(
    "results/evidence/task2/paired_bootstrap/manifest.json"
)
assert bootstrap_manifest["gate"] == "G6-PAIRED-BOOTSTRAP"
assert bootstrap_manifest["decision_status"] == "closed"
assert bootstrap_manifest["git_dirty"] is False
assert bootstrap_manifest["holdout_opened"] is False
assert bootstrap_manifest["new_candidates_allowed"] is False
assert bootstrap_manifest["random_seed_generalisability_claim_allowed"] is False
assert bootstrap_manifest["candidate_selection_affected"] is False
assert bootstrap_manifest["ultimate_winner_frozen"] is False

bootstrap_intervals = pd.read_csv(bootstrap_artifacts["interval_summary"])
bootstrap_observed = pd.read_csv(bootstrap_artifacts["observed_metrics"])
bootstrap_groups = pd.read_csv(bootstrap_artifacts["group_audit"])
bootstrap_registry = pd.read_csv(
    bootstrap_artifacts["registry_snapshot"], dtype=str, keep_default_na=False
)
macro_intervals = bootstrap_intervals.loc[
    bootstrap_intervals["metric_scope"].eq("overall")
    & bootstrap_intervals["metric"].eq("macro_f1")
].copy()
assert len(bootstrap_intervals) == 12
assert len(macro_intervals) == 2
assert macro_intervals["replicates"].eq(10000).all()
assert macro_intervals["confidence_level"].eq(0.95).all()
assert macro_intervals["interval_contains_zero"].eq(False).all()
assert macro_intervals["observed_within_practical_tie"].eq(False).all()
assert (
    macro_intervals["ci_lower"] > macro_intervals["practical_tie_threshold"]
).all()
assert bootstrap_groups.loc[0, "row_count"] == 32753
assert bootstrap_groups.loc[0, "unique_group_count"] == 22885
assert (
    bootstrap_groups.loc[0, "group_semantics"]
    == "conservative_dependency_block_not_verified_sku"
)
assert len(bootstrap_registry) == bootstrap_registry["run_id"].nunique() == 20
display(bootstrap_observed)


# Additional single-output views rendered below:
# display(
#     macro_intervals.loc[
#         :,
#         [
#             "comparison_id",
#             "seed",
#             "observed_delta",
#             "ci_lower",
#             "ci_upper",
#             "fraction_replicates_above_zero",
#             "replicates",
#             "practical_tie_threshold",
#         ],
#     ]
# )
# display(bootstrap_groups)
# display(Image(filename=str(bootstrap_artifacts["paired_bootstrap_figure"])))


> **Interpretation — measured evidence:** The paired family bootstrap supports I2 over C2 for both already-fitted seed pairs. Seed `2753` gives observed delta `+0.017651` with 95% percentile interval `[0.013050, 0.022453]`; seed `2026` gives `+0.011607` with `[0.005793, 0.017341]`. Neither interval includes zero or the predeclared `0.005` practical-tie region. Each interval uses `10,000` resamples of the same `22,885` conservative family blocks, not independent rows. This supports a positive fitted-pair difference; it does not prove I2 wins under every training seed. At seed `2026`, the separate Fall and Spring class intervals include zero, so class-level gains are less certain even though overall macro-F1 is separated.

> **Chart guide.** The horizontal scale is I2-minus-C2 macro-F1. Points mark observed deltas, intervals mark the 95% grouped-bootstrap range, and vertical references mark zero and the 0.005 practical-tie threshold.

#### 13.1a Bootstrap interval table

In [ ]:
from IPython.display import display

display(
    macro_intervals.loc[
        :,
        [
            "comparison_id",
            "seed",
            "observed_delta",
            "ci_lower",
            "ci_upper",
            "fraction_replicates_above_zero",
            "replicates",
            "practical_tie_threshold",
        ],
    ]
)

> **Interpretation — 13.1a Bootstrap interval table**
>
> **Why this output exists.** This output isolates 13.1a bootstrap interval table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `observed_delta` is I2 minus C2; `ci_lower`/`ci_upper` form the 95% interval; `fraction_replicates_above_zero` is the positive-draw share; `replicates` is 10,000.
>
> **What it shows.** Both 10,000-draw grouped-bootstrap intervals keep I2's fitted-pair macro-F1 gain above zero and the 0.005 practical-tie boundary.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 13.1b Bootstrap group audit

In [ ]:
from IPython.display import display

display(bootstrap_groups)

> **Interpretation — 13.1b Bootstrap group audit**
>
> **Why this output exists.** This output isolates 13.1b bootstrap group audit so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `observed_delta` is I2 minus C2; `ci_lower`/`ci_upper` form the 95% interval; `fraction_replicates_above_zero` is the positive-draw share; `replicates` is 10,000.
>
> **What it shows.** Both 10,000-draw grouped-bootstrap intervals keep I2's fitted-pair macro-F1 gain above zero and the 0.005 practical-tie boundary.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

#### 13.1c Paired-bootstrap figure

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(bootstrap_artifacts["paired_bootstrap_figure"])))

> **Interpretation — 13.1c Paired-bootstrap figure**
>
> **Why this output exists.** This output isolates 13.1c paired-bootstrap figure so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The horizontal scale is I2-minus-C2 macro-F1. Points mark observed deltas, intervals mark the 95% grouped-bootstrap range, and vertical references mark zero and the 0.005 practical-tie threshold.
>
> **What it shows.** Both 10,000-draw grouped-bootstrap intervals keep I2's fitted-pair macro-F1 gain above zero and the 0.005 practical-tie boundary.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 13.2 Qualified literature comparison

Published work uses different targets, splits, pretraining, resolution, or multimodal inputs, so scores are not direct benchmarks.

In [ ]:
literature_comparison = pd.DataFrame(
    [
        {
            "study": "This Task 2",
            "dataset": "Fashion Product Images; 32,753 labelled development rows",
            "target": "4-class Season",
            "split": "5 fixed family-safe OOF folds",
            "image_size": "80x60 aspect-preserved",
            "training": "scratch; ArticleType auxiliary label only for I2",
            "inference_input": "image only",
            "metric": "pooled OOF macro-F1",
            "direct_score_comparison": "reference row",
            "source": "verified manifests in this notebook",
        },
        {
            "study": "Seo et al. (2025), ResNet-BERT",
            "dataset": "same public source; under-sampled to 9,250 rows",
            "target": "37 product categories",
            "split": "6,475 train / 925 validation / 1,850 test",
            "image_size": "resize 256 then centre crop 224",
            "training": "pretrained ResNet50 and pretrained BERT",
            "inference_input": "image plus product-title text",
            "metric": "accuracy and training time",
            "direct_score_comparison": "no",
            "source": (
                "https://journals.plos.org/plosone/article?"
                "id=10.1371/journal.pone.0324621"
            ),
        },
        {
            "study": "Kolisnik et al. (2021), Condition-CNN",
            "dataset": "same public source; 41,027 images",
            "target": "masterCategory / subCategory / articleType hierarchy",
            "split": "65% train / 10% validation / 25% test",
            "image_size": "224x224",
            "training": "custom VGG16-like conditional hierarchy",
            "inference_input": "image only",
            "metric": "level-wise accuracy",
            "direct_score_comparison": "no",
            "source": (
                "https://www.sciencedirect.com/science/article/pii/"
                "S0957417421006291"
            ),
        },
    ]
)
assert literature_comparison["direct_score_comparison"].tolist() == [
    "reference row",
    "no",
    "no",
]
method_citations = pd.DataFrame(
    [
        {
            "method": "Grad-CAM",
            "role_here": "coarse predicted-class sensitivity map",
            "boundary": "post-hoc and non-causal",
            "source": (
                "https://openaccess.thecvf.com/content_iccv_2017/html/"
                "Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html"
            ),
        },
        {
            "method": "Saliency sanity checks",
            "role_here": "limits claims made from heatmaps",
            "boundary": "visual plausibility is not validation",
            "source": (
                "https://proceedings.neurips.cc/paper/2018/hash/"
                "294a8ed24b1ad22ec2e7efea049b8737-Abstract.html"
            ),
        },
        {
            "method": "Cluster bootstrap",
            "role_here": "resample related product-family blocks",
            "boundary": "blocks are conservative, not verified SKUs",
            "source": "https://doi.org/10.1111/j.1467-9868.2007.00593.x",
        },
    ]
)
display(literature_comparison)


# Additional single-output views rendered below:
# display(method_citations)


> **Interpretation — measured evidence:** The two fashion papers support architecture ideas, not a numeric benchmark. Condition-CNN shows that hierarchical fashion labels can structure visual learning, which makes the I2 auxiliary-head test reasonable. ResNet-BERT shows that pretrained image features plus title text can help product-category recognition, which explains why P* is useful as a representation ceiling. Their targets, row subsets, splits, resolution, pretraining, inputs, and accuracy metrics all differ from this four-class, image-only, scratch Season protocol; their scores therefore cannot validate or rank our macro-F1. The method papers also bound our claims: grouped bootstrap respects dependence, while Grad-CAM and saliency sanity checks require non-causal, human-reviewed interpretation.

> **Column guide.** Paper/method identifies the source; task, inputs, split, and metric expose protocol differences; direct-comparison fields state that published scores are not rankable here.

#### 13.2a Method-boundary citation table

In [ ]:
from IPython.display import display

display(method_citations)

> **Interpretation — 13.2a Method-boundary citation table**
>
> **Why this output exists.** This output isolates 13.2a method-boundary citation table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Paper/method identifies the source; task, inputs, split, and metric expose protocol differences; direct-comparison fields state that published scores are not rankable here.
>
> **What it shows.** The cited work motivates architecture and interpretation choices but cannot serve as a direct score benchmark for this protocol.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 14. Ultimate Judgement and freeze

Apply the predeclared rule, reject alternatives explicitly, and freeze all choices before holdout.

### 14.1 Apply the final decision scorecard

The selected model must balance class quality, robustness, calibration, and deployment cost.

In [ ]:
from fashion.task2.ultimate_judgement import (
    load_verified_selection_freeze,
    load_verified_ultimate_judgement_manifest,
)

ultimate_manifest, ultimate_manifest_path, ultimate_artifacts = (
    load_verified_ultimate_judgement_manifest(
        "results/evidence/task2/ultimate_judgement/manifest.json",
        project_root=ROOT,
    )
)
assert ultimate_manifest["gate"] == "G7-ULTIMATE-JUDGEMENT"
assert ultimate_manifest["decision_status"] == "closed"
assert ultimate_manifest["git_dirty"] is False
assert ultimate_manifest["ultimate_winner_frozen"] is True
assert ultimate_manifest["holdout_opened"] is False
assert ultimate_manifest["holdout_metrics_present"] is False
assert ultimate_manifest["selected_candidate"] == "I2"
assert (
    ultimate_manifest["selected_experiment_id"]
    == "g4-i2-article-type-lambda-0-3-c1"
)

scorecard = pd.read_csv(ultimate_artifacts["scorecard"])
rejected_alternatives = pd.read_csv(
    ultimate_artifacts["rejected_alternatives"]
)
with ultimate_artifacts["decision"].open(encoding="utf-8") as handle:
    decision = json.load(handle)
assert len(scorecard) == 2
assert scorecard["selected"].sum() == 1
assert scorecard.loc[scorecard["selected"], "candidate"].item() == "I2"
assert len(rejected_alternatives) == 12
assert decision["direct_selection_rule_passed"] is True
assert decision["cost_tie_break_used"] is False
assert all(decision["selection_checks"].values())
assert decision["holdout_opened"] is False
assert decision["holdout_metrics_present"] is False

display(
    scorecard.loc[
        :,
        [
            "candidate",
            "primary_macro_f1",
            "stability_macro_f1",
            "primary_spring_f1",
            "worst_stress_macro_f1",
            "cpu_end_to_end_median_ms",
            "parameter_count",
            "selected",
        ],
    ]
)


# Additional single-output views rendered below:
# display(
#     rejected_alternatives.loc[
#         :, ["alternative", "role", "final_eligible", "reason"]
#     ]
# )


> **Interpretation — measured evidence:** I2 is the frozen Task 2 winner. Its primary-seed pooled OOF macro-F1 is `0.752687`, versus `0.735036` for C2, a `+0.017651` lead; it also leads at both seeds, passes both grouped-bootstrap intervals and every declared shortcut, JPEG, and robustness guard. The practical-tie rule was not needed. I2 is also smaller and faster than C2, so quality and deployment cost point to the same choice. The strongest limitation is brightness `0.85`: it reduces I2 macro-F1 to `0.363961` and Spring recall to `0.003010`. This is development evidence from two fixed seeds, not proof for every seed or real-world shift.

> **Column guide.** Candidate/variant names the row; `pooled_macro_f1` is computed across all OOF rows; `fold_sd_macro_f1` is five-fold variation; Spring fields expose minority performance; delta is candidate minus reference; selected/eligible fields apply the frozen rule.

#### 14.1a Rejected-alternative table

In [ ]:
from IPython.display import display

display(
    rejected_alternatives.loc[
        :, ["alternative", "role", "final_eligible", "reason"]
    ]
)

> **Interpretation — 14.1a Rejected-alternative table**
>
> **Why this output exists.** This output isolates 14.1a rejected-alternative table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** `alternative` names the model, `role` states why it was tested, `final_eligible` enforces the assignment boundary, and `reason` records the measured rejection.
>
> **What it shows.** The fixed scorecard selects I2; quality, stability, robustness guards, and deployment cost point to the same choice.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

### 14.2 Write the freeze manifest

The manifest proves that model choices were locked before independent evaluation.

In [ ]:
selection_freeze, selection_freeze_path = load_verified_selection_freeze(
    TASK2_SELECTION_FREEZE_JSON,
    project_root=ROOT,
)
assert ultimate_artifacts["selection_freeze"] == selection_freeze_path
assert selection_freeze["status"] == "frozen"
assert selection_freeze["holdout_opened"] is False
assert selection_freeze["holdout_metrics_present"] is False
assert selection_freeze["selected_model"]["candidate"] == "I2"
assert selection_freeze["selected_model"]["scratch"] is True
assert selection_freeze["selected_model"]["weights"] is None
assert selection_freeze["selected_model"]["benchmark_only"] is False
assert selection_freeze["selected_model"]["final_eligible"] is True
assert selection_freeze["selected_model"]["inference_inputs"] == ["image"]
assert selection_freeze["refit_rule"]["dataset"] == (
    "all_development_rows_with_valid_season"
)
assert selection_freeze["refit_rule"]["epochs"] == 24
assert selection_freeze["refit_rule"]["seed"] == RANDOM_SEED
assert (
    selection_freeze["refit_rule"]["validation_or_holdout_early_stopping"]
    is False
)
assert selection_freeze["calibration"]["evaluation_claim_allowed"] is False
assert selection_freeze["calibration"]["app_review_threshold"] is None
assert len(selection_freeze["primary_development_evidence"]["run_ids"]) == 5
assert (
    selection_freeze["decision_provenance"]["implementation_sha256"]
    == ultimate_manifest["implementation_sha256"]
)

freeze_summary = pd.DataFrame(
    [
        {
            "freeze_id": selection_freeze["freeze_id"],
            "candidate": selection_freeze["selected_model"]["candidate"],
            "experiment_id": selection_freeze["selected_model"]["experiment_id"],
            "development_rows": selection_freeze["primary_development_evidence"]["valid_development_rows"],
            "refit_epochs": selection_freeze["refit_rule"]["epochs"],
            "refit_seed": selection_freeze["refit_rule"]["seed"],
            "temperature": selection_freeze["calibration"]["temperature"],
            "holdout_opened": selection_freeze["holdout_opened"],
            "decision_timestamp_utc": selection_freeze["decision_recorded_at_utc"],
            "freeze_sha256": ultimate_manifest["artifacts"]["selection_freeze"]["sha256"],
        }
    ]
)
display(freeze_summary.T)

> **Interpretation — frozen handoff:** The refit decision is now complete before holdout: train the scratch I2 configuration on all `32,753` valid development rows for exactly `24` epochs with seed `2753`, recompute normalisation from development content pixels, mask missing ArticleType labels, and use no validation or holdout early stopping. Inference remains image-only; ArticleType is never required at prediction time. The scalar temperature is frozen only for future bundle confidence, and no app review threshold is claimed without a business cost. Notebook 06 may evaluate only this hashed bundle after refit; the holdout remains sealed here and cannot change the model.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 14.3 Refit the frozen model on all development data

Final refit uses all labelled development rows only after CV selection is complete.

In [ ]:
from IPython.display import Image, display
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure

from fashion.task2.refit import load_verified_development_refit_manifest

refit_manifest, refit_manifest_path, refit_bundle = (
    load_verified_development_refit_manifest(project_root=ROOT)
)
assert refit_manifest["gate"] == "G8-DEVELOPMENT-REFIT"
assert refit_manifest["selected_candidate"] == "I2"
assert refit_manifest["scratch"] is True
assert refit_manifest["weights"] is None
assert refit_manifest["valid_development_rows"] == 32753
assert refit_manifest["final_epoch"] == 24
assert refit_manifest["validation_used"] is False
assert refit_manifest["early_stopping_used"] is False
assert refit_manifest["holdout_opened"] is False
assert refit_manifest["holdout_metrics_present"] is False
assert refit_manifest["primary_metric_name"] is None
assert refit_bundle["inference"]["inputs"] == ["image"]
assert refit_bundle["auxiliary"]["used_at_inference"] is False

refit_history = pd.read_csv(
    ROOT / refit_manifest["artifacts"]["history"]["path"]
)
assert refit_history["epoch"].tolist() == list(range(1, 25))
assert refit_history["train_samples"].eq(32753).all()
assert not any("validation" in column.lower() for column in refit_history.columns)

refit_summary = pd.Series(
    {
        "run_id": refit_manifest["run_id"],
        "candidate": refit_manifest["selected_candidate"],
        "development_rows": refit_manifest["valid_development_rows"],
        "seed": refit_manifest["seed"],
        "final_epoch": refit_manifest["final_epoch"],
        "first_train_loss": refit_history.iloc[0]["train_loss"],
        "final_train_loss": refit_history.iloc[-1]["train_loss"],
        "first_train_accuracy": refit_history.iloc[0]["train_accuracy"],
        "final_train_accuracy": refit_history.iloc[-1]["train_accuracy"],
        "temperature": refit_manifest["temperature"],
        "model_sha256": refit_manifest["bundle"]["sha256"],
        "validation_used": refit_manifest["validation_used"],
        "holdout_opened": refit_manifest["holdout_opened"],
    },
    name="value",
).rename_axis("field").to_frame()
display(refit_summary)

refit_figure_path = TASK2_FIGURE_DIR / "development_refit_training_curve.png"
figure = Figure(figsize=(11, 4.2), constrained_layout=True)
FigureCanvasAgg(figure)
loss_axis, accuracy_axis = figure.subplots(1, 2)
loss_axis.plot(refit_history["epoch"], refit_history["train_loss"], label="total train loss")
loss_axis.plot(refit_history["epoch"], refit_history["train_season_loss"], label="Season train loss")
loss_axis.plot(
    refit_history["epoch"],
    0.3 * refit_history["train_auxiliary_loss"],
    label="0.3 x ArticleType train loss",
)
loss_axis.set(xlabel="Epoch", ylabel="Loss", title="Fixed-epoch refit losses")
loss_axis.grid(alpha=0.25)
loss_axis.legend()
accuracy_axis.plot(
    refit_history["epoch"],
    refit_history["train_accuracy"],
    color="tab:blue",
    label="train accuracy",
)
accuracy_axis.set(
    xlabel="Epoch",
    ylabel="Accuracy",
    title="Optimisation diagnostic only - no validation curve",
)
accuracy_axis.set_ylim(0, 1)
accuracy_axis.grid(alpha=0.25)
accuracy_axis.legend()
figure.suptitle("Development refit after model selection was frozen")
figure.savefig(refit_figure_path, dpi=180, bbox_inches="tight")


# Additional single-output views rendered below:
# display(Image(filename=str(refit_figure_path)))


> **Interpretation - measured refit evidence:** The verified scratch I2 bundle was trained on all `32,753` valid development rows for the frozen `24` epochs with seed `2753`; the model has `1,206,112` parameters and SHA-256 `e2511acc2b4e383790d4ba844bb368b1e4b7a40974614a8687a34724aad2566d`. Normalisation was recomputed only from development content pixels (`mean=[0.850058, 0.833309, 0.827673]`, `std=[0.271147, 0.282704, 0.286244]`), Season class weights remained `None`, and all available ArticleType labels were used only through the masked auxiliary loss. Total training loss fell from `1.789050` to `0.551781`, while training accuracy rose from `0.559124` to `0.820505`; these are optimisation diagnostics, not unbiased performance estimates. The teacher-style train/validation learning curves used for model comparison remain the five-fold CV curves in Section 8.4. A validation curve is deliberately absent here because the refit used no validation selection, no early stopping, and no holdout labels. The OOF-fitted temperature `1.365002` is stored for future confidence only, ArticleType is not an inference input, and the holdout remains sealed. Trace: run `task2-season-i2-refit-fall-s2753-4ab5682a30e1`, manifest `models/task2_season.manifest.json`.

> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.

#### 14.3a Development-refit learning curve

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(refit_figure_path)))

> **Interpretation — 14.3a Development-refit learning curve**
>
> **Why this output exists.** This output isolates 14.3a development-refit learning curve so it can be read without a wide, scroll-heavy result block.
>
> **Chart guide.** The x-axis is epoch. Loss panels use lower-is-better train and validation curves; metric panels use higher-is-better validation accuracy and macro-F1. Solid lines are fold means and shaded bands are fold variability.
>
> **What it shows.** The frozen I2 bundle trains for exactly 24 epochs on all 32,753 development rows; its curve is optimisation evidence only.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.

## 15. Handoff to final evaluation

Audit every artifact and hand the frozen Task 2 component to Notebook 06 without opening holdout here.

### 15.1 Audit final artifacts and notebook evidence

Artifact traceability and image-only inference are separate final checks.

#### 15.1.1 Artifact and traceability audit

Every final claim must map back to a run, configuration, prediction file, table, figure, or checkpoint.

In [ ]:
from fashion.task2.handoff import audit_task2_artifacts

task2_artifact_audit = audit_task2_artifacts()
assert len(task2_artifact_audit) == 10
assert task2_artifact_audit["artifact"].is_unique
assert task2_artifact_audit["status"].eq("PASS").all()
assert {
    "selection_freeze",
    "development_refit_bundle",
    "inference_source",
    "registry_binding",
}.issubset(set(task2_artifact_audit["artifact"]))
display(task2_artifact_audit)

> **Interpretation — measured artifact audit:** All `10/10` Task 2 component checks pass. The audit reconnects the G7 selection freeze, G8 registry-bound refit manifest, scratch bundle, 24-epoch history, runtime, canonical split and label map, image-only inference source, and exact completed registry row. The final bundle remains `e2511acc2b4e383790d4ba844bb368b1e4b7a40974614a8687a34724aad2566d`, from run `task2-season-i2-refit-fall-s2753-4ab5682a30e1`. This proves Task 2 artifact consistency only; it does not claim holdout performance or complete the group report. Trace: `results/evidence/task2/final_handoff/artifact_audit.csv`.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 15.1.2 Verify image-only inference smoke

One labelled development image checks the frozen preprocessing, model, calibration, and output schema without touching protected evaluation data.

In [ ]:
from fashion.task2.inference import load_season_bundle, predict_season

smoke_rows = season_development.loc[season_development["id"].astype(str).eq("1163")]
assert len(smoke_rows) == 1
smoke_row = smoke_rows.iloc[0]
assert smoke_row["partition"] == "development"
task2_inference_bundle = load_season_bundle(device="cpu")
task2_smoke_prediction = predict_season(
    task2_inference_bundle, ROOT / str(smoke_row["path"])
)
assert tuple(task2_smoke_prediction.probabilities) == SEASON_LABELS
assert abs(sum(task2_smoke_prediction.probabilities.values()) - 1.0) < 1e-6
assert task2_smoke_prediction.review_required is None
display(pd.DataFrame([task2_smoke_prediction.to_dict()]))

> **Interpretation — measured inference smoke:** The verified CPU path loads the frozen scratch I2 package and predicts development product `1163` as Summer with confidence `0.928210`. The ordered probabilities are Fall `0.056100`, Spring `0.004713`, Summer `0.928210`, and Winter `0.010977`; they sum to one. `review_required` stays `None` because no business threshold was frozen. This one-image check proves that image decoding, frozen normalisation, logits, temperature scaling, and provenance work together; it is not an accuracy estimate. Trace: `results/evidence/task2/final_handoff/inference_smoke.json`.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

### 15.2 Verify the sealed-holdout handoff

Notebook 03 prepares the final model but must never open protected evaluation labels.

In [ ]:
from fashion.task2.handoff import (
    build_task2_handoff_evidence,
    load_verified_task2_handoff,
)

task2_handoff, task2_handoff_path = build_task2_handoff_evidence(
    task2_artifact_audit, task2_smoke_prediction
)
task2_handoff, _, verified_handoff_audit, inference_smoke = (
    load_verified_task2_handoff(task2_handoff_path)
)
assert task2_handoff["status"] == "ready_for_group_freeze"
assert task2_handoff["task2_component_ready"] is True
assert task2_handoff["group_freeze_verified"] is False
assert task2_handoff["notebook_06_unlocked"] is False
assert task2_handoff["holdout_opened"] is False
assert task2_handoff["holdout_metrics_present"] is False
assert task2_handoff["model_change_allowed"] is False
assert len(verified_handoff_audit) == 10
handoff_summary = pd.DataFrame(
    [{
        "status": task2_handoff["status"],
        "task2_component_ready": task2_handoff["task2_component_ready"],
        "group_freeze_verified": task2_handoff["group_freeze_verified"],
        "notebook_06_unlocked": task2_handoff["notebook_06_unlocked"],
        "holdout_opened": task2_handoff["holdout_opened"],
        "run_id": task2_handoff["run_id"],
        "smoke_product_id": inference_smoke["product_id"],
        "next_gate": task2_handoff["next_gate"],
    }]
)
display(handoff_summary)


# Additional single-output views rendered below:
# display(pd.DataFrame({"unresolved_risk": task2_handoff["unresolved_risks"]}))


> **Interpretation — locked group handoff:** The Task 2 component is `ready_for_group_freeze`: all ten artifact checks pass, the frozen image-only bundle loads, and the development smoke prediction carries the correct run, manifest, and bundle hashes. This does **not** unlock final evaluation: `group_freeze_verified=False`, `notebook_06_unlocked=False`, and `holdout_opened=False`. The next gate is a machine-readable whole-group freeze. Notebook 06 must later report the one-shot holdout result and retain the known limits: two seeds do not cover all randomness, brightness `0.85` is severe, ArticleType conflict remains difficult, Grad-CAM is non-causal, and no app review threshold is justified. No Task 2 model change is allowed after this handoff. Trace: `results/evidence/task2/final_handoff/manifest.json`.

> **Column guide.** The first column names the audited item or row; metric columns report its measured value; boolean columns are contract checks rather than model scores.

#### 15.2a Handoff unresolved-risk table

In [ ]:
import pandas as pd

from IPython.display import display

display(pd.DataFrame({"unresolved_risk": task2_handoff["unresolved_risks"]}))

> **Interpretation — 15.2a Handoff unresolved-risk table**
>
> **Why this output exists.** This output isolates 15.2a handoff unresolved-risk table so it can be read without a wide, scroll-heavy result block.
>
> **Column guide.** Each row is one explicit risk that must survive into the group handoff and final judgement.
>
> **What it shows.** Task 2 is ready for the whole-group freeze, while group_freeze_verified, notebook_06_unlocked, and holdout_opened all remain false.
>
> **Decision and limit.** It supports the frozen development decision in the parent subsection; complete hashes remain in the artifact, and holdout evidence is not introduced here.